# 🧠 Neural Network Surrogate for EBRT: 3D TransUNet with FiLM Conditioning

## Notebook 2/2: Surrogate Training + Treatment Optimization

---

### Project Context

This notebook is the **second part** of a complete pipeline to optimize **External Beam Radiation Therapy (EBRT)** treatments using artificial intelligence.

- **Notebook 1** (`ebrt_montecarlo_simulacion.ipynb`): Monte Carlo simulations with OpenGATE/GEANT4 → generates the training dataset (3D photon dose maps)
- **Notebook 2** (THIS): Trains a neural network that **replaces** Monte Carlo (×1000 faster) and uses it to **optimize** treatment parameters

### Clinical Equipment — Hospital (Varian TrueBeam SVC 3.0)

| Component | Specification |
|---|---|
| **Accelerator** | Varian TrueBeam SVC 3.0 |
| **Photons (FF)** | 6 MV, 10 MV, 15 MV |
| **Photons (FFF)** | 6 MV FFF, 10 MV FFF |
| **MLC** | HD 120 leaves (32 central pairs × 2.5 mm + 28 outer pairs × 5 mm) |
| **Maximum MLC field** | 40 × 22 cm² |
| **TPS** | Eclipse v17.0.0 with AcurosXB (Dose-to-medium) |
| **Calculation resolution** | 2 × 2 × 2 mm³ |
| **Anatomical sites** | Pelvis · Head-and-neck · Lung |

### Two-Stage Architecture

```
╔══════════════════════════════════════════════════════════════════════════════╗
║                    COMPLETE PIPELINE: SURROGATE + OPTIMIZER                 ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  ┌── STAGE 1: SURROGATE (Forward Model) ──────────────────────────────┐     ║
║  │                                                                     │     ║
║  │   INPUT:                          OUTPUT:                           │     ║
║  │   • 3D density map ────┐     ┌──── 3D dose map           │     ║
║  │     (patient CT)        │     │     (instant prediction)   │     ║
║  │   • Beam parameters ─────┤     │                                │     ║
║  │     (energy, FFF, field,    │     │     ≈ AcurosXB / Monte Carlo   │     ║
║  │      SSD, gantry, collimator, │     │     but ×1000 faster      │     ║
║  │      patient type, ...)     │     │                                │     ║
║  │                    ┌─────────────┐ │                                │     ║
║  │                    │ 3D TransUNet│─┘                                │     ║
║  │                    │ + FiLM      │                                  │     ║
║  │                    └─────────────┘                                  │     ║
║  └─────────────────────────────────────────────────────────────────────┘     ║
║                              │                                               ║
║                              ▼ (the surrogate is differentiable)               ║
║                                                                              ║
║  ┌── STAGE 2: OPTIMIZER (Inverse Problem) ────────────────────────────┐     ║
║  │                                                                     │     ║
║  │   OBJECTIVE: Find beam parameters that maximize         │     ║
║  │   dose in the tumor AND minimize dose in healthy organs       │     ║
║  │                                                                     │     ║
║  │   Emulates Eclipse inverse planning:                       │     ║
║  │     1. Define objectives → 2. Optimizer searches for optimal fluence    │     ║
║  │     3. AcurosXB calculates dose (3-4 min) → Our surrogate: <1s  │     ║
║  │                                                                     │     ║
║  │   Method: Gradient optimization (autograd through the network) │     ║
║  │           + clinical constraints as hard constraints           │     ║
║  │                                                                     │     ║
║  └─────────────────────────────────────────────────────────────────────┘     ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

### Input Data: Hospital DICOM

The real data we will receive from the hospital includes, per patient:
- **CT** (DICOM): 3D density map of the patient (Hounsfield → density)
- **RT-Structure**: PTV contours, OARs contoured by the oncologist
- **RT-Plan**: Beam parameters, positions of the 120 HD MLC leaves at each control point
- **RT-Dose**: 3D dose calculated with AcurosXB (Dose-to-medium, 2mm) — our ground truth

### 3 Anatomical Sites

| Site | Heterogeneity | Main Challenge | Gradients |
|---|---|---|---|
| **Pelvis** | Low (homogeneous soft tissue) | Large volumes, nearby OARs (rectum, bladder) | Moderate |
| **Head-and-neck** | Medium (dense bone, air cavities) | Very steep dose gradients | High |
| **Lung** | High (tissue-air interface, ρ≈0.26) | Loss of lateral electronic equilibrium, simulations fail with simple algorithms | Complex |

### Why Two Stages Instead of a Single End-to-End Network?

| Criterion | Two stages (our choice) | End-to-end |
|---|---|---|
| **Forward problem** ($\text{params} \to \text{dose}$) | Well-posed: unique solution | — |
| **Inverse problem** ($\text{objective} \to \text{params}$) | Ill-posed: solved with explicit optimizer | Mode collapse (multiple configs yield similar dose) |
| **Clinical constraints** | Hard constraints in the optimizer | Difficult to guarantee |
| **Interpretability** | The surrogate explains how each param affects dose | Black box |
| **Generalization** | The surrogate works for any objective function | Only for the loss it was trained with |

### Which Architecture?

**3D TransUNet with FiLM conditioning** — a CNN + Transformer hybrid:

- **3D CNN Encoder**: captures local features (density interfaces, anatomical boundaries, material gradients)
- **Transformer bottleneck**: captures long-range dependencies (a distant bone absorbs radiation that no longer reaches the tumor — this spatial correlation is global)
- **FiLM decoder** (Feature-wise Linear Modulation): injects the **10 beam+anatomy parameters** at each level of the decoder, modulating dose reconstruction according to energy, FFF, angle, patient type, etc.
- **Deep supervision**: auxiliary losses at multiple decoder scales to accelerate convergence

### Table of Contents

1. **Environment and dependencies** — Kaggle/local setup
2. **Data loading** — Pipeline from the HDF5 of Notebook 1 or hospital DICOM data
3. **Model architecture** — 3D TransUNet + FiLM detailed
4. **Loss function** — Physics-informed with 7 components
5. **Training** — With validation and early stopping
6. **Evaluation** — Gamma-index 3%/3mm, DVH, profiles
7. **Stage 2: Optimizer** — Search for optimal parameters (emulates inverse planning)
8. **Conclusions** — Results and next steps

---

> **Key reference**: Dosimetric deep learning (DDL) — Xing et al., *Med. Phys.* (2020); TransUNet — Chen et al., *MICCAI* (2021); FiLM — Perez et al., *AAAI* (2018)
> **TrueBeam data**: Saravanakumar et al., *IJAMSCR* 13(3) 2025 — Installation and Commissioning of TrueBeam SVC 3.0

---
# PART 1: ENVIRONMENT AND DEPENDENCIES
---

## 1.1 — What Do We Need and Why?

| Package | Minimum Version | Role in This Notebook |
|---|---|---|
| **PyTorch** | ≥ 2.0 | Deep learning framework — model, autograd, optimization |
| **einops** | ≥ 0.6 | Readable tensor operations (`rearrange`, `repeat`) — key for the 3D Transformer |
| **h5py** | ≥ 3.0 | Reading the HDF5 dataset generated in Notebook 1 |
| **SimpleITK** | ≥ 2.0 | Reading medical images (.mhd) as fallback |
| **matplotlib** | ≥ 3.5 | 2D visualization of dose, profiles, DVH |
| **numpy** | ≥ 1.20 | General numerical computation |
| **scipy** | ≥ 1.7 | Gamma-index, interpolation, filters |
| **tqdm** | ≥ 4.0 | Progress bars during training |
| **json** | stdlib | Reading simulation metadata |

### Why PyTorch and Not TensorFlow/JAX?

- **Dynamic autograd**: Stage 2 (optimization) requires computing $\partial D / \partial \theta_{beam}$ by propagating gradients *through* the entire trained network. PyTorch does this natively with `requires_grad=True` on the input parameters.
- **Medical ecosystem**: most dosimetric deep learning papers use PyTorch (MONAI, TorchIO).
- **Debugging**: eager mode facilitates interactive debugging in notebooks.

### Why einops?

In 3D networks with Transformers, reshape operations between formats `(B, C, D, H, W)` ↔ `(B, N, C)` are frequent and **very error-prone** with `torch.reshape`. `einops` makes them explicit and readable:

```python
# Without einops (error-prone):
x = x.reshape(B, C, -1).permute(0, 2, 1)

# With einops (clear and safe):
x = rearrange(x, 'b c d h w -> b (d h w) c')
```

In [ ]:
"""
CELL 1: Dependency installation and imports
================================================
We install everything needed and verify the environment.
"""
import subprocess, sys, os, platform, shutil

def install(pkg):
    """Instala un paquete Python silenciosamente."""
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                          stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

# --- Install packages that might be missing ---
for pkg in ['einops', 'h5py', 'SimpleITK', 'tqdm']:
    try:
        __import__(pkg.lower().replace('-', '_').replace('SimpleITK', 'SimpleITK'))
    except ImportError:
        print(f"  Instalando {pkg}...")
        install(pkg)

# --- Core imports ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import json
import time
import math
import gc
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass, field

# --- PyTorch ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split

# --- Utilidades ---
from einops import rearrange, repeat
from tqdm.auto import tqdm
import h5py

# ─── Global notebook timer ───
# Used for the Kaggle timeout: we measure TOTAL notebook time,
# not just the training loop, to avoid exceeding the 12h session.
T_NOTEBOOK_START = time.time()

# --- Environment verification ---
print("=" * 65)
print("  EXECUTION ENVIRONMENT — NOTEBOOK 2: NEURAL NETWORK")
print("=" * 65)

print(f"\n  Python:     {sys.version.split()[0]}")
print(f"  PyTorch:    {torch.__version__}")
print(f"  NumPy:      {np.__version__}")
print(f"  CUDA:       {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ Not available (CPU mode)'}")
if torch.cuda.is_available():
    print(f"  VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  RAM:        ", end="")
try:
    with open('/proc/meminfo') as f:
        for line in f:
            if 'MemTotal' in line:
                print(f"{int(line.split()[1]) / 1e6:.1f} GB")
                break
except:
    print("N/A")

# --- Configurar device ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n  🎯 Device seleccionado: {DEVICE}")

# --- Detect environment ---
IN_KAGGLE = os.path.exists('/kaggle')
BASE_INPUT = '/kaggle/input/notebooks/ibongarciagomez' if IN_KAGGLE else '.'
BASE_OUTPUT = '/kaggle/working' if IN_KAGGLE else './output'
os.makedirs(BASE_OUTPUT, exist_ok=True)

print(f"  📍 Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"  📂 Input:   {BASE_INPUT}")
print(f"  📂 Output:  {BASE_OUTPUT}")

# --- Reproducibilidad ---
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # benchmark=True: cuDNN auto-tune of conv3d kernels → ~15-25% faster
    # We sacrifice bit-for-bit reproducibility for speed (differences ~1e-6)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
print(f"\n  🎲 Seed global: {SEED}")
print(f"  ⚡ cuDNN benchmark: {torch.backends.cudnn.benchmark if torch.cuda.is_available() else 'N/A'}")
print(f"  ⏱️  T_NOTEBOOK_START recorded (for Kaggle timeout)")
print("=" * 65)

In [ ]:
"""
CELL 2: Global configuration — Hyperparameters and paths
=========================================================
We centralize ALL configuration here so it is easy to
modify in a single place and have it affect the entire notebook.

We use a Python @dataclass which is cleaner than a dict.

UNIFIED DISCOVERY:
  _find_all_dataset_h5() searches ALL of /kaggle/input/ (not just BASE_INPUT)
  to find consolidated datasets added with "Add Data".
  It is executed here so Config has the correct info from the start.
"""

# ═══════════════════════════════════════════════════════════════
# Unified auto-discovery of ALL valid HDF5 files
# ═══════════════════════════════════════════════════════════════

def _find_all_dataset_h5():
    """
    Discovers ALL valid HDF5 files with fields (density, params, dose).
    Searches ALL of /kaggle/input/ (or local equivalent), not just BASE_INPUT.

    This allows finding consolidated datasets that the user adds with
    "Add Data" en Kaggle (aparecen en /kaggle/input/<dataset-slug>/, no
    necesariamente bajo BASE_INPUT).

    Returns: list of (path, n_samples) sorted by n_samples descending
    """
    all_paths = []

    # Search paths: ALL Kaggle input + output
    search_roots = []
    if IN_KAGGLE:
        search_roots.append('/kaggle/input')  # ← SEARCHES ALL, not just BASE_INPUT
    elif os.path.isdir(BASE_INPUT):
        search_roots.append(BASE_INPUT)
    if os.path.isdir(BASE_OUTPUT):
        search_roots.append(BASE_OUTPUT)

    for root_dir in search_roots:
        for dirpath, _, filenames in os.walk(root_dir):
            for fname in filenames:
                if not fname.endswith('.h5'):
                    continue
                fpath = os.path.join(dirpath, fname)
                try:
                    with h5py.File(fpath, 'r') as hf:
                        # Verify required fields
                        if all(k in hf for k in ('density', 'params', 'dose')):
                            n = hf['density'].shape[0]
                            all_paths.append((fpath, n))
                except Exception:
                    pass

    all_paths.sort(key=lambda x: x[1], reverse=True)
    return all_paths

# ─── Discover datasets BEFORE creating Config ───
_auto_h5_all = _find_all_dataset_h5()
_auto_h5_total_samples = sum(n for _, n in _auto_h5_all)
if _auto_h5_all:
    print(f"\n  📊 Datasets descubiertos:")
    for p, n in _auto_h5_all:
        print(f"     {n:>6} samples │ {p}")

# ─── Auto-detectar checkpoint ───
_auto_checkpoint = None
if IN_KAGGLE:
    for _root in ['/kaggle/input']:
        for _dp, _, _fns in os.walk(_root):
            for _fn in _fns:
                if _fn == 'checkpoint_ebrt.pth':
                    _auto_checkpoint = os.path.join(_dp, _fn)
                    break
            if _auto_checkpoint:
                break
        if _auto_checkpoint:
            break
    if _auto_checkpoint:
        print(f"\n  🔄 Checkpoint found: {_auto_checkpoint}")
else:
    # Local: buscar en BASE_OUTPUT
    _local_ckpt = os.path.join('.', 'checkpoint_ebrt.pth')
    if os.path.exists(_local_ckpt):
        _auto_checkpoint = _local_ckpt
        print(f"\n  🔄 Checkpoint encontrado: {_auto_checkpoint}")


@dataclass
class Config:
    """Centralized experiment configuration."""

    # ═══════════════════════════════════════
    # DATA
    # ═══════════════════════════════════════
    # Working 3D volume (D, H, W = depth, height, width)
    vol_depth: int = 48
    vol_height: int = 64
    vol_width: int = 64

    # Number of scalar beam parameters (adapted to TrueBeam SVC 3.0 + HD MLC)
    n_beam_params: int = 10  # [beam_energy_norm, is_fff, field_x, field_y, ssd, gantry, collimator, patient_type, tumor_depth, tumor_size]

    # Pre-load to RAM: if True, loads entire HDF5 into memory
    # Eliminates I/O during training. Needs ~16 GB RAM for 10k samples.
    preload_to_ram: bool = True

    # Ruta al dataset HDF5 (fallback si _auto_h5_all no encuentra nada)
    dataset_h5: str = os.path.join(BASE_INPUT, 'ebrt_training_dataset.h5')
    # If True, generates synthetic data instead of using HDF5
    use_synthetic: bool = False

    # ═══════════════════════════════════════
    # MODEL — 3D TransUNet + FiLM
    # ═══════════════════════════════════════
    # Encoder CNN 3D
    encoder_channels: tuple = (1, 32, 64, 128, 256)
    # Meaning: 1 input channel (density) → 32 → 64 → 128 → 256 features
    # At each level, the volume is reduced ×2 in each dimension (via stride=2)
    # Level 0: (1, 48, 64, 64)  → (32, 48, 64, 64)  [ConvBlock, no downsample]
    # Level 1: (32, 48, 64, 64) → (64, 24, 32, 32)   [DownBlock ×2]
    # Level 2: (64, 24, 32, 32) → (128, 12, 16, 16)  [DownBlock ×2]
    # Level 3: (128, 12, 16, 16) → (256, 6, 8, 8)    [DownBlock ×2]

    # Transformer bottleneck
    trans_num_heads: int = 8        # Multi-head self-attention
    trans_num_layers: int = 4       # Number of Transformer blocks
    trans_mlp_ratio: float = 4.0    # Internal MLP expansion ratio (C → 4C → C)
    trans_dropout: float = 0.1      # Regularization in attention

    # Decoder with FiLM
    decoder_channels: tuple = (256, 128, 64, 32)
    # Symmetric to the encoder, but with skip connections + FiLM conditioning
    film_hidden_dim: int = 128  # Hidden dimension of the FiLM generator

    # Deep supervision
    deep_supervision: bool = True  # Auxiliary losses at decoder levels 1, 2, 3

    # ═══════════════════════════════════════
    # LOSS FUNCTION
    # ═══════════════════════════════════════
    loss_mse_weight: float = 1.0      # L_MSE dose-weighted
    loss_gradient_weight: float = 0.5  # L_grad: penalizes dose gradient errors
    loss_ssim_weight: float = 0.2      # L_SSIM: calidad estructural 3D
    loss_dvh_weight: float = 0.3       # L_DVH: difference in dose-volume histogram metrics
    loss_falloff_weight: float = 0.2   # L_falloff: extra precision in the distal fall-off zone
    loss_cosine_weight: float = 0.15   # L_cosine: cosine similarity between pred and target (global shape alignment)
    loss_l1_weight: float = 0.4        # L_L1: global MAE — paper Zhao et al. shows L1 > L2 for dose
    loss_transition_epochs: int = 60  # Epochs for smooth transition (protects pre-trained model)
    force_best_during_transition: bool = True  # During transition: force save of "best" model at each periodic checkpoint

    # ═══════════════════════════════════════
    # DATA AUGMENTATION (training only)
    # ═══════════════════════════════════════
    # Referencias: nnU-Net v2 (Isensee, Nature Methods 2021), Klarenberg 2025
    # (JACMP, EBRT breast), Shojaei 2025 (PMB, nnU-Net + sgDefAug),
    # Zhang 2024 (AIMLR, 3D dose radiomics), Foster 2025 (multi-task fluence+dose).
    #
    # ALL augmentations are checkpoint-compatible (they don't touch architecture).
    # Geometric augmentations apply the SAME transformation to density AND dose.
    # Beam params are NOT modified → deformations must be small,
    # within clinical uncertainty (±5-10mm position, ±3-5° IGRT rotation).

    augment_flip: bool = False          # DISABLED: flips invalidate beam params (gantry, isocenter)

    # ── Intensity (density only, ZERO risk with beam params) ──
    augment_noise_std: float = 0.02      # max σ of Gaussian noise (nnU-Net: var 0-0.1)
    augment_noise_prob: float = 0.5      # Probability of adding noise (nnU-Net: p=0.1)
    augment_intensity_range: tuple = (0.90, 1.10)  # Random scaling ±10% (nnU-Net: 0.75-1.25)
    augment_intensity_prob: float = 0.5  # Probability of applying intensity scaling

    augment_blur: bool = True             # Gaussian blur — simulates CT resolution variations
    augment_blur_sigma: tuple = (0.3, 1.0)  # Blur σ range (nnU-Net: 0.5-1.0)
    augment_blur_prob: float = 0.15       # Probability (nnU-Net: p=0.2)

    augment_gamma: bool = True            # Gamma transform — simula variaciones contraste
    augment_gamma_range: tuple = (0.85, 1.25)  # Rango γ (UQ paper: 0.9-1.1, nnU-Net: 0.7-1.5 → compromiso)
    augment_gamma_prob: float = 0.2       # Probability (increased 0.15→0.2)

    augment_brightness: bool = True       # Additive offset — simulates CT calibration drift
    augment_brightness_range: tuple = (-0.03, 0.03)  # ±3% (UQ paper: "intensity shifts", nnU-Net: ±0.3)
    augment_brightness_prob: float = 0.3  # Probability

    augment_lowres: bool = True           # Low resolution simulation — different CT slice thickness
    augment_lowres_scale: tuple = (1.0, 2.0)  # Downscale factor per axis (nnU-Net v2: 1.0-2.0)
    augment_lowres_prob: float = 0.15     # Probability (nnU-Net: p=0.2)
    augment_lowres_per_axis: bool = True  # Independent downscale per axis (simulates anisotropy)

    augment_coarse_dropout: bool = True   # Coarse Dropout 3D — simulates metallic artifacts / missing data
    augment_coarse_dropout_cube_size: tuple = (4, 8)  # Cube side range in voxels (4-8 vxl = 8-16mm at 2mm/vxl)
    augment_coarse_dropout_num_cubes: tuple = (1, 3)   # Range of number of cubes to remove
    augment_coarse_dropout_fill: float = 0.0  # Fill value (0=air, physically correct)
    augment_coarse_dropout_prob: float = 0.1  # Moderate probability (lowered from 0.15: too aggressive)

    # ── 3D Geometric (density + dose are transformed together) ──
    # Safe with beam params: small displacements (±6mm at 2mm/voxel)
    # are within clinical positioning uncertainty (±5-10mm).
    augment_translate: bool = True        # Random 3D translation
    augment_translate_range: int = 3      # ±N voxels maximum (±6mm at 2mm/voxel)
    augment_translate_prob: float = 0.5   # Probability of applying translation

    augment_elastic: bool = True          # Smooth 3D elastic deformation
    augment_elastic_alpha: float = 4.0    # Deformation magnitude (increased 2→4: ~4-8mm, Shojaei uses 40mm)
    augment_elastic_sigma: float = 5.0    # Gaussian smoothing σ (voxels, controla suavidad)
    augment_elastic_prob: float = 0.4     # Probability (increased 0.3→0.4)

    augment_rotate: bool = True           # Small 3D rotation
    augment_rotate_max_deg: float = 7.0   # ±7° (UQ paper: ±10°, nnU-Net: ±30° → compromise with FiLM)
    augment_rotate_prob: float = 0.3      # Conservador: p=0.3 (nnU-Net: p=0.2)
    # Justification: ±7° within IGRT setup error (±5-10° typical).
    # FiLM conditioning receives gantry angle → patient rotation ±7°
    # equivalent to ∆gantry ∓7° → acceptable mismatch (UQ paper uses ±10°).

    augment_scale: bool = True            # Isotropic scaling (zoom in/out)
    augment_scale_range: tuple = (0.95, 1.05)  # ±5% (Klarenberg: ±10%, nnU-Net: 0.7-1.4)
    augment_scale_prob: float = 0.2       # Probability (nnU-Net: p=0.2)
    # Justification: ±5% simulates patient size variation without changing
    # SSD/tumor_depth significantly for FiLM.

    # ═══════════════════════════════════════
    # TRAINING
    # ═══════════════════════════════════════
    # ── Adjusted for datasets of 5000-10000+ samples ──
    # With 5200 samples and batch=8: ~487 batches/epoch ≈ 17 min/epoch
    # With 10500 samples and batch=8: ~984 batches/epoch ≈ 35 min/epoch
    # If OOM occurs with batch=8, lower to 6
    batch_size: int = 8         # P100 16GB — increased from 4→8 for large datasets
    num_epochs: int = 1200     # ABSOLUTE maximum cap (multiple OneCycleLR cycles)
    onecycle_cycle_epochs: int = 200  # Epochs per cycle (round number, ~4.6 days on Kaggle with 10k)
    weight_decay: float = 5e-5  # L2 regularization (increased to reduce overfitting)
    lr_scheduler: str = 'onecycle'  # 'onecycle' (Super-Convergence + warm restarts)
    # ── OneCycleLR with automatic WARM RESTARTS ──
    # Each cycle: warmup → peak → cosine decay (onecycle_cycle_epochs epochs).
    # Upon completing a cycle, a new one is created with max_lr ×lr_restart_decay.
    # Within the cycle, the scheduler state is saved/restored between sessions.
    # Primer ciclo: pct_start=0.15 (warmup largo). Restarts: 0.05 (warmup corto).
    learning_rate: float = 3e-4        # max_lr of OneCycleLR (increased to explore new loss)
    onecycle_div_factor: float = 25.0    # LR inicial = max_lr / 25 = 4e-6
    onecycle_final_div_factor: float = 1e4  # LR final = max_lr / 1e4 = 1e-8
    onecycle_pct_start: float = 0.15     # 15% warmup first cycle (~22 epochs)
    restart_pct_start: float = 0.05      # 5% warmup in restart cycles (~7 ep)
    lr_restart_decay: float = 0.85  # max_lr ×0.85 per cycle (1.5e-4 → 1.28e-4 → 1.08e-4)
    patience: int = 4000          # Early stopping patience
    es_dose_acc_threshold: float = 0.3  # Minimum improvement threshold for DoseAcc3% to save (in pp)
    val_fraction: float = 0.15  # 15% for validation
    test_fraction: float = 0.10 # 10% for test (final evaluation)
    gradient_clip: float = 1.0  # Gradient clipping (estabilidad)
    num_workers: int = 4        # Workers for DataLoader (increased from 2→4)

    # ═══════════════════════════════════════
    # CHECKPOINT AND RESUMPTION
    # ═══════════════════════════════════════
    resume_from: str = _auto_checkpoint  # None = from scratch, path = resume
    checkpoint_name: str = 'checkpoint_ebrt.pth'  # Checkpoint filename
    split_indices_name: str = 'split_indices.json'  # Fixed train/val/test indices
    # Maximum TOTAL notebook time (from T_NOTEBOOK_START) before auto-saving.
    # Kaggle = 12h = 43200s. We subtract 900s (15 min) margin for saving + copying.
    # Timeout is measured from T_NOTEBOOK_START (imports cell), NOT from the
    # start of the training loop. This includes setup (~40 min) and never overruns.
    kaggle_time_limit: int = 43200 - 900  # 42300s = 11h 45min from notebook start

    # ═══════════════════════════════════════
    # OPTIMIZER (STAGE 2)
    # ═══════════════════════════════════════
    optim_lr: float = 0.01      # Learning rate for beam parameter optimization
    optim_steps: int = 500      # Gradient optimization steps
    optim_d95_target: float = 0.95  # Tumor D95 ≥ 95% of prescribed dose
    optim_oar_max_weight: float = 10.0  # Penalty for exceeding maximum OAR dose

    # ═══════════════════════════════════════
    # SYNTHETIC DATA (for demo without MC)
    # ═══════════════════════════════════════
    n_synthetic_samples: int = 200  # Sufficient to demonstrate the pipeline
    synthetic_noise_level: float = 0.02  # Noise to simulate MC uncertainty

    # ═══════════════════════════════════════
    # DROPOUT (for MC Dropout — uncertainty estimation)
    # ═══════════════════════════════════════
    # 3D Dropout in the Conv blocks of encoder/decoder.
    # During inference: activated (model.train()) to generate multiple
    # stochastic predictions → variance = epistemic uncertainty.
    # More robust than storing σ_MC (Gal & Ghahramani, ICML 2016).
    model_dropout: float = 0.1       # Tasa de dropout en ConvBlock3D
    mc_dropout_samples: int = 20     # N forward passes to estimate uncertainty

    # ═══════════════════════════════════════
    # SWA — Stochastic Weight Averaging (Izmailov et al., 2018)
    # ═══════════════════════════════════════
    # Averages model weights from the last N epochs to obtain
    # flatter solutions → better generalization and calibration.
    # Checkpoint compatible: swa_model is saved/restored separately.
    # Activated after swa_start_epoch (when the model has already converged).
    swa_enabled: bool = True          # Enable SWA
    swa_start_epoch: int = 400        # Start accumulating from this epoch
    swa_update_every: int = 5         # Update SWA every N epochs (smooths cyclic LR noise)

    @property
    def volume_size(self):
        """Tuple (D, H, W) for use throughout the notebook."""

        return (self.vol_depth, self.vol_height, self.vol_width)


# ═══════════════════════════════════════
# Instantiate global configuration
# ═══════════════════════════════════════
CFG = Config()

print("=" * 50)
print("\n⚙️  EXPERIMENT CONFIGURATION")
print("=" * 50)

for key, val in vars(CFG).items():
    if key.startswith('_'):
        continue  # Skip propiedades internas
    print(f"  {key:<30} = {val}")

print(f"\n  📐 Working volume: {CFG.volume_size}")
total_voxels = CFG.vol_depth * CFG.vol_height * CFG.vol_width
print(f"  📦 Total voxels: {total_voxels:,}")
mem_per_sample = total_voxels * 4 * 2 / 1e6  # input + output, float32
print(f"  💾 Memory per sample (input+output): ~{mem_per_sample:.1f} MB")
print(f"  💾 Batch de {CFG.batch_size}: ~{mem_per_sample * CFG.batch_size:.0f} MB")

print(f"\n  🏗️ Encoder: {' → '.join(map(str, CFG.encoder_channels))}")
print(f"  🔄 Transformer: {CFG.trans_num_layers} capas × {CFG.trans_num_heads} heads, mlp_ratio={CFG.trans_mlp_ratio}")
print(f"  🏗️ Decoder: {' → '.join(map(str, CFG.decoder_channels))} → 1 (dosis)")
print(f"  🎯 Dropout (MC Dropout): {CFG.model_dropout}")

if CFG.resume_from is not None:
    print(f"\n  🔄 RESUMPTION mode: training state will be restored")
else:
    print(f"\n  🆕 NEW mode: OneCycleLR ({CFG.onecycle_cycle_epochs} ep/cycle, max {CFG.num_epochs} ep)")

print("=" * 50)





In [ ]:
# =============================================================================
# CELL 2b: Available HDF5 data verification
# =============================================================================
# _auto_h5_all was already discovered in cell 2 by _find_all_dataset_h5()
# which searches ALL of /kaggle/input/ (not just BASE_INPUT).
#
# This cell only prints a detailed summary for visual verification.
# =============================================================================

print("─" * 60)
print("  AVAILABLE HDF5 DATA")
print("─" * 60)

if len(_auto_h5_all) == 0:
    print(f"\n  ⚠️ No valid HDF5 files found")
    print(f"     SYNTHETIC data will be used ({CFG.n_synthetic_samples} samples)")
    print(f"\n  To use real MC data in Kaggle:")
    print(f"     1. Ve a 'Add Data' en el notebook de Kaggle")
    print(f"     2. Search for and add the datasets 'ebrt-consolidated-part-XXX'")
    print(f"     3. Verify they appear in /kaggle/input/")
    if IN_KAGGLE:
        # List what is actually in /kaggle/input for diagnostics
        print(f"\n  📂 Current contents of /kaggle/input/:")
        try:
            for entry in sorted(os.listdir('/kaggle/input')):
                entry_path = os.path.join('/kaggle/input', entry)
                if os.path.isdir(entry_path):
                    n_files = len(os.listdir(entry_path))
                    print(f"     📁 {entry}/ ({n_files} archivos)")
                else:
                    print(f"     📄 {entry}")
        except Exception as e:
            print(f"     (error listando: {e})")
else:
    print(f"\n  ✅ {len(_auto_h5_all)} valid HDF5 file(s) — "
          f"{_auto_h5_total_samples} total samples")
    for i, (p, n) in enumerate(_auto_h5_all):
        relpath = os.path.relpath(p, '/kaggle/input') if IN_KAGGLE and p.startswith('/kaggle/input') \
            else os.path.basename(p)
        print(f"     [{i+1}] {relpath}: {n} samples")
    print(f"\n  use_synthetic = {CFG.use_synthetic}")

print("─" * 60)

---
# PART 2: DATA PIPELINE
---

## 2.1 — Where Do the Data Come From?

The data can come from **three sources** (in order of priority):

### Source A: Real Hospital DICOM Data
The hospital provides us with CT + RT-Structure + RT-Plan + RT-Dose for 3 anatomical sites (pelvis, head-and-neck, lung). The dose is calculated with AcurosXB in Dose-to-medium mode at 2×2×2 mm³ resolution. The RT-Plan includes all positions of the 120 HD Millennium MLC leaves.

### Source B: Monte Carlo HDF5 Dataset (from Notebook 1)
Notebook 1 generates a file `ebrt_training_dataset.h5` with this structure:
```
/density      (N, 1, Dz, Dy, Dx)  — Normalized density map (÷1.85)
/params       (N, 10)              — 10 normalized clinical parameters [0,1]
/dose         (N, 1, Dz, Dy, Dx)  — Absorbed dose in Gy (unnormalized)
```
> **Note**: MC uncertainty (σ_D/D) is not stored in the HDF5 — dose-weighted MSE already approximates the optimal weighting 1/σ². For prediction uncertainty estimation, MC Dropout is used (Gal & Ghahramani, ICML 2016).

### Source C: Synthetic Data (for demonstration)
If you don't have the data yet, we generate synthetic data that **mimics the real physics** of the TrueBeam using analytical models of megavoltage photon dose deposition with measured PDD parameters (Saravanakumar et al. 2025):

- **Density**: simplified phantom with 3 anatomies: pelvis (homogeneous), head-and-neck (dense bone + gradients), lung (tissue-air interface)
- **Dose**: analytical MV photon model: build-up + exponential attenuation + rectangular/FFF field profile + inverse square law
- The analytical model is NOT as accurate as Monte Carlo or AcurosXB, but for training the neural network it is sufficient to demonstrate that the architecture works

## 2.2 — Data Normalization: Critical Decisions

### Density
$$\rho_{norm} = \rho / \rho_{max} = \rho / 1.85 \quad \Rightarrow \quad \rho_{norm} \in [0, 1]$$

### Dose — Per-Sample Normalization
Each sample is normalized by its own maximum dose:
$$D_{norm} = D / D_{max,sample} \quad \Rightarrow \quad D_{norm} \in [0, 1]$$

**Why per-sample and not global?**
- The absolute dose varies by **orders of magnitude** between scenarios (a 6 MV beam deposits differently than a 15 MV one, small vs. large fields, FF vs FFF, etc.)
- If we normalize globally, low-energy samples would have values ~0.01 and the network would not learn their details
- The network learns the **shape** (relative spatial distribution) of the dose
- The absolute magnitude is recovered by multiplying by $D_{max}$

### Scalar Parameters (adapted to TrueBeam SVC 3.0 + HD MLC)
$$p_{norm} = (p - p_{min}) / (p_{max} - p_{min}) \quad \Rightarrow \quad p_{norm} \in [0, 1]$$

| # | Parameter | Real Range | $p_{min}$ | $p_{max}$ | Notes |
|---|---|---|---|---|---|
| 0 | Beam energy (ordinal) | 6/10/15 MV | 0.0 | 1.0 | 0.0=6MV, 0.5=10MV, 1.0=15MV |
| 1 | FFF (filter-free) | 0/1 | 0 | 1 | 0=FF (with filter), 1=FFF (without filter) |
| 2 | Field X — jaws (cm) | 1–40 | 1 | 40 | Defined by X jaws |
| 3 | Field Y — HD MLC (cm) | 1–22 | 1 | 22 | HD MLC maximum: 22 cm |
| 4 | SSD (cm) | 80–110 | 80 | 110 | Source-to-surface distance |
| 5 | Gantry angle (°) | 0–360 | 0 | 360 | Linac arm rotation |
| 6 | Collimator angle (°) | 0–360 | 0 | 360 | MLC/field rotation |
| 7 | Patient type (ordinal) | pelvis/head/lung | 0.0 | 1.0 | 0=pelvis, 0.5=head-and-neck, 1.0=lung |
| 8 | Tumor depth (cm) | 2–18 | 2 | 18 | Distance from surface |
| 9 | Tumor size (cm) | 1–8 | 1 | 8 | Approximate diameter |

**Note on MLC**: The individual positions of the 120 HD MLC leaves will be encoded in the future as a **2D fluence map** (extra input channel), not as scalar parameters. The MLC modulates beam intensity and defines the field shape — its complexity requires spatial representation.

## 2.3 — Data Augmentation Strategy

Data augmentation is **critical** in medical imaging where training data is scarce. Our pipeline applies **11 transforms** (inspired by nnU-Net, Isensee et al. 2021) that are physically consistent — both density and dose volumes receive the **same** geometric transform, while intensity transforms apply only to density:

### Geometric Transforms (applied identically to density + dose)

| # | Transform | Parameters | Physics Motivation |
|---|---|---|---|
| 1 | **Elastic deformation** | σ=3.0, α=50-200, p=0.4 | Simulates inter-patient anatomical variability (organ shape, soft tissue deformation) |
| 2 | **3D rotation** | ±10° per axis, p=0.5 | Equivalent to ±10° gantry/collimator uncertainty; teaches angular invariance |
| 3 | **Translation** | ±8 voxels, p=0.3 | Patient positioning uncertainty on the treatment couch |
| 4 | **3D scaling** | 0.85–1.15×, p=0.2 | Patient size variability |
| 5 | **Random flip** | 3 axes, p=0.5 each | Lateral symmetry (left↔right, anterior↔posterior) |

### Intensity Transforms (applied only to density)

| # | Transform | Parameters | Physics Motivation |
|---|---|---|---|
| 6 | **Gaussian blur** | σ=0.5–1.5, p=0.15 | Scanner resolution variability |
| 7 | **Intensity scaling** | 0.9–1.1×, p=0.5 | CT calibration differences between scanners |
| 8 | **Gaussian noise** | σ∈[0, 0.02], p=0.5 | CT noise / quantum mottle |
| 9 | **Gamma correction** | γ=0.7–1.5, p=0.2 | Non-linear density remapping |
| 10 | **Brightness shift** | ±0.1, p=0.3 | Systematic CT offset |
| 11 | **Low-resolution simulation** | 2–4× downsample+upsample, p=0.15 | Training with coarse CT scans |
| 12 | **Coarse dropout** | 2–4 cubes, 4–12 voxel side, p=0.1 | Robustness to missing data / artifacts |

> **Key design decision**: Augmentations are applied **on-the-fly** during training (not pre-computed). This saves disk space and provides virtually infinite variation. Validation and test sets receive **no augmentation** — they evaluate raw model performance. The  wrapper class makes this transparent.

In [ ]:
"""
CELL 3: Synthetic data generator for EBRT — TrueBeam SVC 3.0
=====================================================================
Generates (density, dose) pairs that mimic the physics of the
TrueBeam SVC 3.0 with HD MLC Millennium accelerator from the hospital.

MEASURED PDD DATA — Saravanakumar et al. (IJAMSCR, 2025):
  Commissioning of TrueBeam SVC 3.0 at Bangalore Baptist Hospital.
  Table 3: PDD for 10×10 cm² field at SSD=100 cm.

AVAILABLE BEAM QUALITIES:
  FF  (Flattening Filter):  6 MV, 10 MV, 15 MV
  FFF (Flattening Filter Free): 6 MV FFF, 10 MV FFF
  → NO hay 18 MV en este TrueBeam.
  → NO hay 15 MV FFF.

PHYSICS IMPLEMENTED (megavoltage photons):
  1. Photon PDD: build-up followed by exponential attenuation
     D(z) ∝ (1 - exp(-z/d_bup)) · exp(-μ·(z - d_max))
  2. Lateral profile: flat (FF) or forward-peaked (FFF)
  3. Inverse square law: (SSD + d_max)² / (SSD + d)²
  4. Attenuation by heterogeneous density (bone, lung)
  5. Rectangular field limited by HD MLC (max 40×22 cm²)
  6. 3 anatomies: pelvis, head-and-neck, lung

10 INPUT PARAMETERS:
  [beam_energy_norm, is_fff, field_x_cm, field_y_cm, ssd_cm,
   gantry_angle, collimator_angle, patient_type, tumor_depth, tumor_size]
"""

# ═══════════════════════════════════════════════════════════════
# PDD parameters table — MEASURED DATA from TrueBeam SVC 3.0
# Source: Saravanakumar et al. (2025), Tabla 3
# Campo 10×10 cm², SSD = 100 cm
# ═══════════════════════════════════════════════════════════════

_TRUEBEAM_PDD_DATA = {
    # (energy_norm, is_fff): (d_max_cm, mu_eff_cm-1, pdd_10%, pdd_20%, D20/D10, label)
    # mu_eff adjusted to reproduce measured PDD with our analytical model
    #
    # FF energies (with flattening filter):
    (0.0, 0): (1.50, 0.032, 66.22, 37.96, 0.666, '6 MV'),
    (0.5, 0): (2.34, 0.023, 73.37, 46.20, 0.738, '10 MV'),
    (1.0, 0): (2.95, 0.019, 76.56, 49.75, 0.763, '15 MV'),
    #
    # FFF energies (without flattening filter):
    # FFF has slightly lower d_max and higher surface dose
    (0.0, 1): (1.38, 0.037, 63.43, 34.52, 0.630, '6 MV FFF'),
    (0.5, 1): (2.18, 0.027, 71.04, 42.90, 0.705, '10 MV FFF'),
    # 15 MV FFF is NOT available on this TrueBeam
    # If requested, we use 15 MV FF data as fallback
    (1.0, 1): (2.95, 0.019, 76.56, 49.75, 0.763, '15 MV (FFF n/a → FF fallback)'),
}

# Additional HD MLC Millennium information
_HD_MLC_SPECS = {
    'n_leaf_pairs': 60,         # 60 pairs = 120 leaves
    'inner_pairs': 32,          # 32 pares centrales
    'inner_width_mm': 2.5,      # 2.5 mm proyectados al isocentro
    'outer_pairs': 28,          # 28 pares exteriores
    'outer_width_mm': 5.0,      # 5.0 mm proyectados al isocentro
    'max_field_x_cm': 40.0,     # Maximum X field (defined by jaws)
    'max_field_y_cm': 22.0,     # Maximum Y field (limited by HD MLC)
    'sad_cm': 100.0,            # Source-to-Axis Distance
}

# Available dose rates (MU/min) — from the paper
_DOSE_RATES = {
    '6 MV':     [100, 200, 300, 400, 500, 600],
    '10 MV':    [100, 200, 300, 400, 500, 600],
    '15 MV':    [100, 200, 300, 400, 500, 600],
    '6 MV FFF': [400, 600, 800, 1000, 1200, 1400],
    '10 MV FFF': [400, 800, 1200, 1600, 2000, 2400],
}


def _truebeam_dmax_and_mu(beam_energy_norm, is_fff):
    """
    Gets d_max and μ_eff for a TrueBeam beam quality.
    
    Interpolates between available energies (6, 10, 15 MV) according to
    beam_energy_norm ∈ [0, 1], and selects FF or FFF according to is_fff.
    
    Args:
        beam_energy_norm: 0.0 (6MV), 0.5 (10MV), 1.0 (15MV) — can interpolate
        is_fff: 0 (FF) or 1 (FFF). For intermediate values, interpolates.
    
    Returns:
        (d_max_cm, mu_eff_cm-1)
    """
    is_fff_binary = int(round(is_fff))  # Redondear a 0 o 1
    
    # Available energies (as normalized fractions)
    energy_keys = [0.0, 0.5, 1.0]
    
    # Linearly interpolate between energies
    en = np.clip(beam_energy_norm, 0.0, 1.0)
    
    for i in range(len(energy_keys) - 1):
        if en <= energy_keys[i + 1]:
            frac = (en - energy_keys[i]) / (energy_keys[i + 1] - energy_keys[i] + 1e-8)
            d0, mu0 = _TRUEBEAM_PDD_DATA[(energy_keys[i], is_fff_binary)][:2]
            d1, mu1 = _TRUEBEAM_PDD_DATA[(energy_keys[i + 1], is_fff_binary)][:2]
            return d0 + frac * (d1 - d0), mu0 + frac * (mu1 - mu0)
    
    # If it reaches here, use the last one
    d, mu = _TRUEBEAM_PDD_DATA[(energy_keys[-1], is_fff_binary)][:2]
    return d, mu


def generate_synthetic_phantom(D, H, W, params_norm):
    """
    Generates a simplified 3D phantom with 3 anatomical types for EBRT.

    The 3 patient types reflect the clinical sites of the hospital:
      - Pelvis (patient_type≈0): homogeneous soft tissue, few heterogeneities
      - Head-and-neck (patient_type≈0.5): dense bone (skull base), air cavities,
        steep gradients
      - Lung (patient_type≈1): we insert large air volumes (ρ≈0.26),
        tissue-air interfaces where electronic equilibrium is lost

    Args:
        D, H, W: volume dimensions
        params_norm: 10 normalized parameters [0,1]:
            [beam_energy_norm, is_fff, field_x, field_y, ssd, gantry,
             collimator, patient_type, tumor_depth, tumor_size]

    Returns:
        density: (D, H, W) float32 — normalized density map (/1.85)
        masks: dict with boolean masks of each region
    """
    density = np.ones((D, H, W), dtype=np.float32)  # Base: agua (ρ=1.0)

    # Normalized coordinates [0, 1] for Z, [-1, 1] for Y/X
    zz = np.linspace(0, 1, D)[:, None, None]
    yy = np.linspace(-1, 1, H)[None, :, None]
    xx = np.linspace(-1, 1, W)[None, None, :]

    # Denormalize parameters relevant to anatomy
    patient_type_n = params_norm[7]   # 0=pelvis, 0.5=head-and-neck, 1.0=lung
    tumor_depth_n = params_norm[8]    # [0, 1] → relative depth
    tumor_size_n = params_norm[9]     # [0, 1] → relative size

    # In EBRT tumors are at variable depths (2-18 cm in 40 cm)
    tumor_depth = 0.05 + tumor_depth_n * 0.40  # [0.05, 0.45] of the volume
    tumor_radius = 0.025 + tumor_size_n * 0.10  # [0.025, 0.125] of the volume

    # --- Tumor (esfera) ---
    tumor_dist = np.sqrt((zz - tumor_depth)**2 + yy**2 + xx**2)
    tumor_mask = tumor_dist < tumor_radius
    density[tumor_mask] = 1.05 / 1.85  # Soft tumor tissue

    # --- Specific anatomy according to patient type ---
    lung_l = np.zeros((D, H, W), dtype=bool)
    lung_r = np.zeros((D, H, W), dtype=bool)
    bone_mask = np.zeros((D, H, W), dtype=bool)

    if patient_type_n < 0.33:
        # ═══ PELVIS: heterogeneidad baja ═══
        # Only lateral pelvic bones (iliac) — thin layers
        bone_depth = 0.55 + 0.15 * np.random.random()
        bone_thickness = 0.02 + 0.01 * np.random.random()
        bone_mask = np.abs(zz - bone_depth) < bone_thickness / 2
        bone_mask = np.broadcast_to(bone_mask, (D, H, W)).copy()
        density[bone_mask] = 1.85 / 1.85  # Hueso compacto

    elif patient_type_n < 0.67:
        # ═══ HEAD-AND-NECK: steep gradients ═══
        # Dense bone (skull base, jaw) — thicker and variable
        bone_depth1 = 0.30 + 0.10 * np.random.random()
        bone_depth2 = 0.60 + 0.10 * np.random.random()
        bone_t1 = 0.04 + 0.02 * np.random.random()
        bone_t2 = 0.03 + 0.02 * np.random.random()
        bone_mask1 = np.abs(zz - bone_depth1) < bone_t1 / 2
        bone_mask2 = np.abs(zz - bone_depth2) < bone_t2 / 2
        bone_mask = np.broadcast_to(bone_mask1 | bone_mask2, (D, H, W)).copy()
        density[bone_mask] = 1.85 / 1.85

        # Air cavity (paranasal sinuses / pharynx)
        air_cavity = ((yy > -0.15) & (yy < 0.15) &
                      (xx > -0.15) & (xx < 0.15) &
                      (zz > 0.35) & (zz < 0.50))
        air_mask = np.broadcast_to(air_cavity, (D, H, W)).copy()
        density[air_mask] = 0.001 / 1.85  # Aire

    else:
        # ═══ LUNG: high heterogeneity ═══
        # Two large lungs with ρ = 0.26 g/cm³ (ICRP lung)
        # This is the most challenging site for dose calculations
        lung_l_mask = ((yy > 0.1) & (yy < 0.7) & (xx > 0.1) & (xx < 0.6) &
                       (zz > 0.15) & (zz < 0.50))
        lung_r_mask = ((yy > 0.1) & (yy < 0.7) & (xx < -0.1) & (xx > -0.6) &
                       (zz > 0.15) & (zz < 0.50))
        lung_l = np.broadcast_to(lung_l_mask, (D, H, W)).copy()
        lung_r = np.broadcast_to(lung_r_mask, (D, H, W)).copy()
        density[lung_l] = 0.26 / 1.85
        density[lung_r] = 0.26 / 1.85

        # Ribs (bone on both sides of the thorax)
        rib_depth = 0.20 + 0.05 * np.random.random()
        rib_mask = (np.abs(zz - rib_depth) < 0.015) & (np.abs(xx) > 0.3)
        bone_mask = np.broadcast_to(rib_mask, (D, H, W)).copy()
        density[bone_mask] = 1.85 / 1.85

    # --- External air (body contour) ---
    body_mask = (yy**2 + xx**2) < 0.85**2
    body_mask = np.broadcast_to(body_mask, (D, H, W)).copy()
    density[~body_mask] = 0.001 / 1.85

    # --- OAR (organ at risk — sphere adjacent to the tumor) ---
    oar_depth = tumor_depth + 0.12
    oar_dist = np.sqrt((zz - oar_depth)**2 + (yy - 0.0)**2 + (xx - 0.30)**2)
    oar_mask = oar_dist < 0.06
    oar_mask = np.broadcast_to(oar_mask, (D, H, W)).copy()

    masks = {
        'tumor': tumor_mask, 'bone': bone_mask, 'body': body_mask,
        'lung_l': lung_l, 'lung_r': lung_r, 'oar': oar_mask,
    }

    return density, masks


def generate_synthetic_dose(density, params_norm, D, H, W):
    """
    Generates a synthetic 3D dose map based on analytical model
    of the TrueBeam SVC 3.0 (FF and FFF megavoltage photons).

    The function models:
    1. Photon PDD with measured TrueBeam parameters (Saravanakumar 2025)
    2. Field profile rectangular (FF: flat, FFF: forward-peaked)
    3. Inverse square law (divergence from SAD=100cm)
    4. Attenuation modulated by heterogeneous density
    5. Field limited by HD MLC: max 40×22 cm²
    6. Gantry and collimator rotations

    Args:
        density: (D, H, W) normalized density map (/1.85)
        params_norm: 10 normalized parameters [0,1]:
            [beam_energy_norm, is_fff_n, field_x_n, field_y_n, ssd_n,
             gantry_n, collimator_n, patient_type_n, depth_n, size_n]

    Returns:
        dose: (D, H, W) float32 — dosis normalizada [0, 1]
    """
    (beam_energy_n, is_fff_n, field_x_n, field_y_n, ssd_n,
     gantry_n, collimator_n, patient_type_n, depth_n, size_n) = params_norm

    # Denormalize parameters to physical units
    d_max_cm, mu_eff = _truebeam_dmax_and_mu(beam_energy_n, is_fff_n)
    is_fff = is_fff_n > 0.5

    # Field size in cm — limited by HD MLC
    field_x_cm = 1.0 + field_x_n * 39.0        # [1, 40] cm (jaws)
    field_y_cm = 1.0 + field_y_n * 21.0         # [1, 22] cm (HD MLC)
    phantom_size_cm = 40.0
    field_x_norm = field_x_cm / phantom_size_cm
    field_y_norm = field_y_cm / phantom_size_cm

    ssd_cm = 80.0 + ssd_n * 30.0                # [80, 110] cm
    gantry_deg = gantry_n * 360.0                # [0, 360] grados
    collimator_deg = collimator_n * 360.0        # [0, 360] grados

    # Normalized coordinates
    zz = np.linspace(0, 1, D)[:, None, None]  # depth [0, 1]
    yy = np.linspace(-1, 1, H)[None, :, None]  # lateral Y [-1, 1]
    xx = np.linspace(-1, 1, W)[None, None, :]  # lateral X [-1, 1]

    # Depth in cm
    z_cm = zz * phantom_size_cm

    # --- 1. Photon PDD (measured TrueBeam data) ---
    d_max_norm = d_max_cm / phantom_size_cm

    # Build-up region (sigmoid suave)
    buildup = 1.0 - np.exp(-3.0 * zz / (d_max_norm + 1e-8))

    # Exponential attenuation past d_max
    mu_norm = mu_eff * phantom_size_cm
    attenuation = np.where(
        zz > d_max_norm,
        np.exp(-mu_norm * (zz - d_max_norm)),
        1.0
    )

    # Inverse square law
    ssd_norm = ssd_cm / phantom_size_cm
    inv_sq = ((ssd_norm + d_max_norm)**2) / ((ssd_norm + zz)**2 + 1e-8)

    pdd = buildup * attenuation * inv_sq

    # --- 2. Field profile ---
    gantry_rad = np.radians(gantry_deg)
    collimator_rad = np.radians(collimator_deg)
    cos_g = np.cos(gantry_rad)
    sin_g = np.sin(gantry_rad)

    # Simplified rotation of lateral profile by gantry angle
    xx_rot = xx * cos_g + zz * sin_g

    # Field penumbra: smooth sigmoid at the edges
    # Typical measured penumbra: 5-8 mm (paper: table 4)
    penumbra_width = 0.015  # ~6 mm in normalized units

    # Field rotation by collimator angle
    cos_c = np.cos(collimator_rad)
    sin_c = np.sin(collimator_rad)
    xx_field = xx * cos_c + yy * sin_c
    yy_field = -xx * sin_c + yy * cos_c

    field_x_half = field_x_norm / 2.0
    field_y_half = field_y_norm / 2.0

    # Profile in X: sigmoid on both edges
    profile_x = 1.0 / (1.0 + np.exp(-(field_x_half - np.abs(xx_field)) / penumbra_width))

    # Profile in Y: sigmoid on both edges
    profile_y = 1.0 / (1.0 + np.exp(-(field_y_half - np.abs(yy_field)) / penumbra_width))

    # Combined profile (rectangular)
    lateral_profile = profile_x * profile_y

    # --- 2b. FFF correction: forward-peaked profile ---
    # FFF beams do NOT have flattening filter → the profile has a peak
    # at the center that drops toward the edges. The shape is approximately conical.
    # Degree of unflatness: increases with energy and field size.
    if is_fff:
        # Off-axis ratio for FFF: central peak with Gaussian fall-off
        r_norm = np.sqrt(xx_field**2 + yy_field**2)
        # Unflatness is more pronounced in 10 FFF than in 6 FFF
        fff_sigma = 0.8 + 0.3 * beam_energy_n  # Greater fall-off for higher energy
        fff_profile = np.exp(-0.5 * (r_norm / fff_sigma)**2)
        # Normalize so that the center is 1.0
        fff_profile = fff_profile / (fff_profile.max() + 1e-8)
        lateral_profile = lateral_profile * fff_profile

    # --- 3. Attenuation by heterogeneous density ---
    density_real = density * 1.85  # Desnormalizar
    cumulative_density = np.cumsum(density_real, axis=0) / D
    density_factor = np.exp(-mu_norm * 0.3 * (cumulative_density - 1.0))

    # --- 4. Combine components ---
    dose = pdd * lateral_profile * density_factor

    # Normalizar a [0, 1]
    dose_max = dose.max()
    if dose_max > 0:
        dose = dose / dose_max

    # Add noise (simulates MC statistical uncertainty)
    noise = np.random.normal(0, CFG.synthetic_noise_level, dose.shape).astype(np.float32)
    dose = np.clip(dose + noise * dose, 0, 1)

    return dose.astype(np.float32)


print("✅ Synthetic EBRT generation functions defined — TrueBeam SVC 3.0")
print(f"   Beam qualities: {', '.join(d[5] for d in _TRUEBEAM_PDD_DATA.values())}")
print(f"   HD MLC: {_HD_MLC_SPECS['n_leaf_pairs']} pares, "
      f"centro {_HD_MLC_SPECS['inner_width_mm']}mm, "
      f"exterior {_HD_MLC_SPECS['outer_width_mm']}mm")
print(f"   Maximum field: {_HD_MLC_SPECS['max_field_x_cm']}×{_HD_MLC_SPECS['max_field_y_cm']} cm²")
print(f"   3 anatomies: pelvis (low), head-and-neck (gradients), lung (air)")

# --- Quick demo ---
# Params: [energy=10MV, FFF=no, fx=15cm, fy=15cm, ssd=100, gantry=0°,
#          collimator=0°, patient_type=lung, depth=0.3, size=0.5]
demo_params = np.array([0.5, 0.0, 0.36, 0.67, 0.67, 0.0, 0.0, 1.0, 0.3, 0.5], dtype=np.float32)
demo_density, demo_masks = generate_synthetic_phantom(CFG.vol_depth, CFG.vol_height, CFG.vol_width, demo_params)
demo_dose = generate_synthetic_dose(demo_density, demo_params, CFG.vol_depth, CFG.vol_height, CFG.vol_width)
print(f"\n   Demo: density shape={demo_density.shape}, range=[{demo_density.min():.3f}, {demo_density.max():.3f}]")
print(f"   Demo: dose shape={demo_dose.shape}, range=[{demo_dose.min():.4f}, {demo_dose.max():.4f}]")

In [ ]:
"""
CELL 4: Visualization of the sample synthetic data
======================================================
We visually verify that the synthetic phantom and dose
are physically reasonable before using them for training.
"""
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Central slice along each axis
z_mid = CFG.vol_depth // 2
y_mid = CFG.vol_height // 2
x_mid = CFG.vol_width // 2

# --- Row 1: Density ---
axes[0, 0].imshow(demo_density[z_mid, :, :], cmap='bone', vmin=0, vmax=1)
axes[0, 0].set_title(f'Density — Axial (Z={z_mid})', fontweight='bold')
axes[0, 0].set_xlabel('X'); axes[0, 0].set_ylabel('Y')

axes[0, 1].imshow(demo_density[:, y_mid, :], cmap='bone', vmin=0, vmax=1, aspect='auto')
axes[0, 1].set_title(f'Density — Coronal (Y={y_mid})', fontweight='bold')
axes[0, 1].set_xlabel('X'); axes[0, 1].set_ylabel('Z (depth)')

axes[0, 2].imshow(demo_density[:, :, x_mid], cmap='bone', vmin=0, vmax=1, aspect='auto')
axes[0, 2].set_title(f'Density — Sagittal (X={x_mid})', fontweight='bold')
axes[0, 2].set_xlabel('Y'); axes[0, 2].set_ylabel('Z (depth)')

# Density histogram
axes[0, 3].hist(demo_density.flatten(), bins=100, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 3].set_title('Density histogram', fontweight='bold')
axes[0, 3].set_xlabel('ρ / ρ_max'); axes[0, 3].set_ylabel('Voxels')
axes[0, 3].axvline(1.0/1.85, color='blue', linestyle='--', label='Agua')
axes[0, 3].axvline(1.0, color='red', linestyle='--', label='Hueso')
axes[0, 3].axvline(0.26/1.85, color='cyan', linestyle='--', label='Lung')
axes[0, 3].legend(fontsize=8)

# --- Row 2: Dose ---
isodose_levels = [0.1, 0.3, 0.5, 0.8, 0.95]
isodose_colors = ['blue', 'cyan', 'green', 'orange', 'red']

for i, (sl, title) in enumerate([
    (demo_dose[z_mid, :, :], f'Dose — Axial (Z={z_mid})'),
    (demo_dose[:, y_mid, :], f'Dosis — Coronal (Y={y_mid})'),
    (demo_dose[:, :, x_mid], f'Dose — Sagittal (X={x_mid})'),
]):
    im = axes[1, i].imshow(sl, cmap='hot', vmin=0, vmax=1,
                            aspect='auto' if i > 0 else 'equal')
    for level, color in zip(isodose_levels, isodose_colors):
        axes[1, i].contour(sl, levels=[level], colors=[color], linewidths=0.7, alpha=0.8)
    axes[1, i].set_title(title, fontweight='bold')
    plt.colorbar(im, ax=axes[1, i], shrink=0.8, label='D/D_max')

# PDD (percent depth dose)
pdd = demo_dose[:, y_mid, x_mid]
z_axis = np.arange(len(pdd))
axes[1, 3].plot(z_axis, pdd * 100, 'b-', linewidth=2)
axes[1, 3].set_title('PDD (Percent Depth Dose)', fontweight='bold')
axes[1, 3].set_xlabel('Depth (voxels)'); axes[1, 3].set_ylabel('% Maximum dose')
axes[1, 3].axhline(50, color='r', ls='--', alpha=0.5, label='50%')
axes[1, 3].axhline(90, color='orange', ls='--', alpha=0.5, label='90%')
axes[1, 3].grid(True, alpha=0.3); axes[1, 3].legend()

fig.suptitle('Demonstration Synthetic Data — Phantom + Analytical Dose (MV photon, rectangular field)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_OUTPUT, 'synthetic_demo.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Visual verification:")
print("   ✓ Density shows differentiated anatomical regions")
print("   ✓ Dose has build-up → maximum → exponential attenuation (characteristic MV photon PDD curve)")
print("   ✓ Isodose lines (contours) show lateral beam broadening")
print("   ✓ Bone (bright region in density) attenuates the radiation")

In [ ]:
"""
CELL 5: PyTorch Dataset — Real or synthetic loading (multi-HDF5)
===================================================================
La clase EBRTDataset hereda de torch.utils.data.Dataset y:
  1. Tries to load ALL real HDF5 files (from the consolidator)
  2. If they don't exist, generates synthetic data on-the-fly
  3. Applies the normalization described in the theory section
  4. Returns tensors ready for the network

Soporte multi-archivo:
  - The consolidator can generate multiple partials (ebrt-consolidated-part-NNN)
  - This class discovers and loads ALL valid HDF5 files automatically
  - Supports float16 (density) y float32 (dose) de forma transparente

Modo preload_to_ram=True:
  - Loads ALL data to RAM on initialization
  - RESIZES to target resolution (48×64×64) during loading
  - PRE-NORMALIZA la dosis per-sample
  - Result: ultra-fast __getitem__ (only numpy→tensor, no I/O or resize)
  - Memory: 10K samples × (48×64×64) × float32 × 2 ≈ 7.7 GB (fits in 32 GB)

AugmentedDataset:
  - Wrapper over any Dataset/Subset that applies data augmentation
  - Only used for training DataLoader (does not affect val/test)
  - Augmentations: noise + intensity + 3D translation + 3D elastic deformation
  - Checkpoint compatible (does not change architecture)
"""

class EBRTDataset(Dataset):
    """
    Dataset for EBRT surrogate training.

    Modos:
      - 'h5': loads real data from one or more HDF5 files
      - 'synthetic': generates analytical data on the fly

    Each sample is a dict with:
      - 'density':  (1, D, H, W) tensor — normalized density map
      - 'params':   (10,) tensor — 10 beam/anatomy parameters normalized [0,1]
      - 'dose':     (1, D, H, W) tensor — dosis normalizada [0,1]
      - 'dose_max': scalar — scale factor to recover absolute dose
    """

    def __init__(self, cfg: Config, mode: str = 'auto', indices=None):
        """
        Args:
            cfg: global configuration
            mode: 'h5', 'synthetic', o 'auto' (intenta h5, sino synthetic)
            indices: subset of indices (for train/val split)
        """
        self.cfg = cfg
        self.indices = indices

        # Decidir modo
        if mode == 'auto':
            has_h5 = (len(_auto_h5_all) > 0) if '_auto_h5_all' in globals() else \
                     (os.path.exists(cfg.dataset_h5) and not cfg.use_synthetic)
            if has_h5 and not cfg.use_synthetic:
                mode = 'h5'
            else:
                mode = 'synthetic'

        self.mode = mode
        self._ram_data = None  # Cache en RAM
        self._h5_handles = []  # Lista de handles abiertos (multi-file lazy)
        self._h5_index_map = []  # Mapeo: global_idx → (file_idx, local_idx)

        if self.mode == 'h5':
            self._init_h5()
        else:
            self._init_synthetic()

    def _init_h5(self):
        """Initializes loading from one or multiple HDF5 files."""
        # Discover all valid HDF5 files
        if '_auto_h5_all' in globals() and len(_auto_h5_all) > 0:
            h5_files = _auto_h5_all  # [(path, n_samples), ...]
        elif os.path.exists(self.cfg.dataset_h5):
            # Fallback: single file (backward compatibility)
            with h5py.File(self.cfg.dataset_h5, 'r') as hf:
                n = hf['density'].shape[0]
            h5_files = [(self.cfg.dataset_h5, n)]
        else:
            raise FileNotFoundError(f"No valid HDF5 file found")

        self._h5_paths = [p for p, _ in h5_files]
        self._h5_n_samples = [n for _, n in h5_files]
        self.n_total_in_files = sum(self._h5_n_samples)

        # Get metadata from the first file
        with h5py.File(self._h5_paths[0], 'r') as hf:
            self.vol_shape = tuple(hf.attrs.get('volume_shape',
                                   hf['density'].shape[2:]))
        self.target_shape = (self.cfg.vol_depth, self.cfg.vol_height, self.cfg.vol_width)

        # Build mapping from global indices to (file, local_index)
        self._h5_index_map = []
        for file_idx, n in enumerate(self._h5_n_samples):
            for local_idx in range(n):
                self._h5_index_map.append((file_idx, local_idx))

        if self.indices is not None:
            self.n_samples = len(self.indices)
        else:
            self.n_samples = self.n_total_in_files

        n_files = len(self._h5_paths)
        if n_files > 1:
            print(f"  📦 Multi-HDF5: {n_files} archivos, "
                  f"{self.n_total_in_files} total samples")
        else:
            print(f"  📂 HDF5: {self.n_total_in_files} samples")

        # Pre-load to RAM if enabled
        if self.cfg.preload_to_ram:
            self._preload_to_ram()
        else:
            # Open all files for lazy I/O
            self._h5_handles = [h5py.File(p, 'r') for p in self._h5_paths]
            print(f"     HDF5 resolution: {self.vol_shape} → target: {self.target_shape}")
            print(f"     Mode: lazy I/O (no pre-loading)")

    def _preload_to_ram(self):
        """
        Pre-loads data from ALL HDF5 files to RAM, resizing to
        target resolution and pre-normalizing dose per-sample.

        Supports float16 (density from new consolidator) and float32
        (previous format) transparently — everything is converted
        a float32 en RAM.
        """
        actual_indices = self.indices if self.indices is not None else \
            list(range(self.n_total_in_files))
        n = len(actual_indices)
        needs_resize = (self.vol_shape[0] != self.target_shape[0] or
                        self.vol_shape[1] != self.target_shape[1] or
                        self.vol_shape[2] != self.target_shape[2])

        D, H, W = self.target_shape
        # Estimar memoria
        mem_estimate_gb = n * 2 * D * H * W * 4 / 1e9  # density + dose_norm
        mem_params_gb = n * self.cfg.n_beam_params * 4 / 1e9
        mem_dmax_gb = n * 4 / 1e9
        total_gb = mem_estimate_gb + mem_params_gb + mem_dmax_gb

        print(f"  ⏳ Pre-loading {n} samples in RAM "
              f"(de {len(self._h5_paths)} archivo(s))...")
        print(f"     Resolution: {self.vol_shape} → {self.target_shape} "
              f"({'resize' if needs_resize else 'ya coincide'})")
        print(f"     Memoria estimada: {total_gb:.1f} GB")

        t0 = time.time()

        # Pre-allocate arrays at TARGET resolution
        density_ram = np.empty((n, 1, D, H, W), dtype=np.float32)
        dose_ram = np.empty((n, 1, D, H, W), dtype=np.float32)
        params_ram = np.empty((n, self.cfg.n_beam_params), dtype=np.float32)
        dmax_ram = np.empty(n, dtype=np.float32)

        # Process file by file
        ram_idx = 0
        for file_idx, h5_path in enumerate(self._h5_paths):
            n_in_file = self._h5_n_samples[file_idx]

            # Determine which global indices belong to this file
            file_global_start = sum(self._h5_n_samples[:file_idx])
            file_global_end = file_global_start + n_in_file

            # Filter actual_indices for this file
            file_local_indices = []
            file_ram_positions = []
            for ram_pos, global_idx in enumerate(actual_indices):
                if file_global_start <= global_idx < file_global_end:
                    local_idx = global_idx - file_global_start
                    file_local_indices.append(local_idx)
                    file_ram_positions.append(ram_pos)

            if not file_local_indices:
                continue

            with h5py.File(h5_path, 'r') as hf:
                relpath = os.path.basename(h5_path)
                print(f"     Loading {relpath}: {len(file_local_indices)} samples...")

                CHUNK = 100
                for chunk_start in range(0, len(file_local_indices), CHUNK):
                    chunk_end = min(chunk_start + CHUNK, len(file_local_indices))
                    chunk_local = file_local_indices[chunk_start:chunk_end]
                    chunk_ram = file_ram_positions[chunk_start:chunk_end]

                    # Sort for efficient HDF5 reading
                    sort_order = sorted(range(len(chunk_local)),
                                       key=lambda i: chunk_local[i])
                    sorted_local = [chunk_local[i] for i in sort_order]

                    # Read from HDF5 — auto-converts float16→float32 via numpy
                    d_chunk = np.float32(hf['density'][sorted_local])
                    p_chunk = np.float32(hf['params'][sorted_local])
                    dose_chunk = np.float32(hf['dose'][sorted_local])

                    # Des-ordenar al orden original
                    inv_order = [0] * len(sort_order)
                    for new_pos, old_pos in enumerate(sort_order):
                        inv_order[old_pos] = new_pos
                    d_chunk = d_chunk[inv_order]
                    p_chunk = p_chunk[inv_order]
                    dose_chunk = dose_chunk[inv_order]

                    for j in range(chunk_end - chunk_start):
                        ram_pos = chunk_ram[j]
                        density_j = d_chunk[j]
                        dose_j = dose_chunk[j]

                        if needs_resize:
                            density_j = self._resize_volume(density_j,
                                                           self.target_shape)
                            dose_j = self._resize_volume(dose_j,
                                                        self.target_shape)

                        dose_max = dose_j.max()
                        if dose_max > 0:
                            dose_j = dose_j / dose_max
                        else:
                            dose_max = 1.0

                        density_ram[ram_pos] = density_j
                        dose_ram[ram_pos] = dose_j
                        params_ram[ram_pos] = p_chunk[j]
                        dmax_ram[ram_pos] = dose_max

                    ram_idx += (chunk_end - chunk_start)
                    if ram_idx % 500 == 0 or ram_idx == n:
                        elapsed = time.time() - t0
                        rate = ram_idx / elapsed if elapsed > 0 else 0
                        print(f"     {ram_idx}/{n} ({rate:.0f}/s)...", end='\r')

        self._ram_data = {
            'density': density_ram,
            'dose': dose_ram,
            'params': params_ram,
            'dose_max': dmax_ram,
        }

        actual_gb = (density_ram.nbytes + dose_ram.nbytes +
                     params_ram.nbytes + dmax_ram.nbytes) / 1e9
        print(f"\n  ✅ Pre-carga completa: {time.time()-t0:.1f}s, {actual_gb:.1f} GB en RAM")

    def _init_synthetic(self):
        """Initializes synthetic generation."""
        self.n_samples = self.cfg.n_synthetic_samples
        if self.indices is not None:
            self.n_samples = len(self.indices)

        # Pre-generate random parameters for reproducibility
        rng = np.random.RandomState(42)
        self._all_params = rng.uniform(0, 1, (self.cfg.n_synthetic_samples, self.cfg.n_beam_params)).astype(np.float32)
        print(f"  🔧 Synthetic dataset: {self.n_samples} samples")
        print(f"     Volume shape: ({self.cfg.vol_depth}, {self.cfg.vol_height}, {self.cfg.vol_width})")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.mode == 'h5':
            if self._ram_data is not None:
                return self._get_ram(idx)
            # Map to real index if using a subset
            real_idx = self.indices[idx] if self.indices is not None else idx
            return self._get_h5(real_idx)
        else:
            real_idx = self.indices[idx] if self.indices is not None else idx
            return self._get_synthetic(real_idx)

    def _get_ram(self, idx):
        """Loads a sample from RAM cache (no I/O or resize)."""
        # .copy() eliminado: .to(device) en train_one_epoch ya crea copia,
        # and DataLoader collate (torch.stack) also creates new tensors.
        # No in-place operations on inputs → safe to share memory.
        return {
            'density': torch.from_numpy(self._ram_data['density'][idx]),
            'params': torch.from_numpy(self._ram_data['params'][idx]),
            'dose': torch.from_numpy(self._ram_data['dose'][idx]),
            'dose_max': torch.tensor(self._ram_data['dose_max'][idx]),
        }

    def _get_h5(self, idx):
        """Loads a sample from HDF5 (lazy I/O, supports multi-file)."""
        file_idx, local_idx = self._h5_index_map[idx]
        hf = self._h5_handles[file_idx]

        # Read and convert to float32 (supports float16 transparently)
        density = np.float32(hf['density'][local_idx])  # (1, D, H, W)
        params = np.float32(hf['params'][local_idx])     # (n_beam_params,)
        dose = np.float32(hf['dose'][local_idx])         # (1, D, H, W)

        # Normalizar dosis per-sample
        dose_max = dose.max()
        if dose_max > 0:
            dose_norm = dose / dose_max
        else:
            dose_norm = dose
            dose_max = 1.0

        # Resize if necessary (MC data may have different shape)
        target_shape = (self.cfg.vol_depth, self.cfg.vol_height, self.cfg.vol_width)
        if density.shape[1:] != target_shape:
            density = self._resize_volume(density, target_shape)
            dose_norm = self._resize_volume(dose_norm, target_shape)

        return {
            'density': torch.from_numpy(density).float(),
            'params': torch.from_numpy(params).float(),
            'dose': torch.from_numpy(dose_norm).float(),
            'dose_max': torch.tensor(dose_max).float(),
        }

    def _get_synthetic(self, idx):
        """Generates a synthetic sample."""
        params = self._all_params[idx]
        D, H, W = self.cfg.vol_depth, self.cfg.vol_height, self.cfg.vol_width

        # Generar fantoma y dosis
        density, _ = generate_synthetic_phantom(D, H, W, params)
        dose = generate_synthetic_dose(density, params, D, H, W)

        # Normalizar dosis per-sample
        dose_max = dose.max()
        if dose_max > 0:
            dose_norm = dose / dose_max
        else:
            dose_norm = dose
            dose_max = 1.0

        # Add channel dimension: (D,H,W) → (1,D,H,W)
        density_t = torch.from_numpy(density[np.newaxis]).float()
        dose_t = torch.from_numpy(dose_norm[np.newaxis]).float()

        return {
            'density': density_t,
            'params': torch.from_numpy(params).float(),
            'dose': dose_t,
            'dose_max': torch.tensor(dose_max).float(),
        }

    @staticmethod
    def _resize_volume(vol, target_shape):
        """Resizes a 3D volume with trilinear interpolation."""
        vol_t = torch.from_numpy(vol).float().unsqueeze(0)  # (1, C, D, H, W)
        resized = F.interpolate(vol_t, size=target_shape, mode='trilinear',
                                align_corners=False)
        return resized.squeeze(0).numpy()

    def close(self):
        """Closes all open HDF5 files."""
        if hasattr(self, 'h5f') and self.h5f is not None:
            self.h5f.close()
        for hf in getattr(self, '_h5_handles', []):
            try:
                hf.close()
            except Exception:
                pass
        self._h5_handles = []


# ═══════════════════════════════════════════════════════════════
# 3D geometric augmentation helper functions
# ═══════════════════════════════════════════════════════════════

def _gaussian_smooth_3d(field, sigma, truncate=3):
    """
    3D separable Gaussian smoothing (pure PyTorch, no scipy).

    Applies 1D Gaussian convolutions along each spatial axis
    sequentially (separable). This is O(N³·k) instead of O(N³·k³)
    where k is the kernel size.

    Args:
        field: tensor (D, H, W) — displacement field a suavizar
        sigma: standard deviation of the Gaussian kernel (in voxels)
        truncate: kernel radius = truncate × σ (default: 3σ)

    Returns:
        smoothed tensor (D, H, W)
    """
    if sigma <= 0:
        return field
    radius = int(truncate * sigma + 0.5)
    x = torch.arange(-radius, radius + 1, dtype=field.dtype)
    g = torch.exp(-0.5 * (x / sigma) ** 2)
    g = g / g.sum()

    vol = field.unsqueeze(0).unsqueeze(0)  # (1, 1, D, H, W)

    for d in range(3):
        # 1D kernel along axis d: shape (1, 1, kD, kH, kW)
        k_shape = [1, 1, 1, 1, 1]
        k_shape[d + 2] = g.shape[0]
        kernel = g.reshape(k_shape)

        # F.pad: (W_left, W_right, H_left, H_right, D_left, D_right)
        pad = [0] * 6
        pad_idx = 2 * (2 - d)  # d=0→4(D_left), d=1→2(H_left), d=2→0(W_left)
        pad[pad_idx] = radius
        pad[pad_idx + 1] = radius

        vol = F.conv3d(F.pad(vol, pad, mode='replicate'), kernel)

    return vol.squeeze(0).squeeze(0)


def _translate_3d(density, dose, max_shift):
    """
    Random 3D translation with zero-padding.

    Shifts density and dose with the SAME random offset on each axis.
    Borders are filled with zero (physically correct:
    density=0 → air/exterior, dose=0 → no dose outside the field).

    ±3 voxels at 2mm/voxel = ±6mm, within the clinical uncertainty
    of patient positioning (±5-10mm typical with IGRT).
    El error inducido en SSD/tumor_depth es <4% → despreciable.

    Args:
        density: (1, D, H, W) tensor
        dose: (1, D, H, W) tensor
        max_shift: maximum displacement in voxels

    Returns:
        (density, dose) trasladados
    """
    for dim in range(3):
        s = torch.randint(-max_shift, max_shift + 1, (1,)).item()
        if s == 0:
            continue
        spatial_dim = dim + 1  # dim 0 = canal
        density = density.roll(s, dims=spatial_dim)
        dose = dose.roll(s, dims=spatial_dim)

        # Fill with zeros the region that "wrapped" from the roll
        slices = [slice(None)] * 4
        if s > 0:
            slices[spatial_dim] = slice(0, s)
        else:
            slices[spatial_dim] = slice(s, None)
        density[tuple(slices)] = 0.0
        dose[tuple(slices)] = 0.0

    return density, dose


def _elastic_deform_3d(density, dose, alpha, sigma):
    """
    Smooth random 3D elastic deformation.

    Generates random displacement fields, smooths them with a Gaussian
    filter (to make them smooth/quasi-diffeomorphic), and applies the
    SAME deformation to density and dose via grid_sample.

    This simulates natural anatomical variations (breathing, bladder
    filling, organ position) without invalidating beam parameters.
    With alpha=4.0, sigma=5.0 maximum displacements are ~2-4 voxels
    (4-8mm), within clinical uncertainty.
    Ref: Shojaei 2025 uses deformations up to 40mm with good results.

    Args:
        density: (1, D, H, W) tensor
        dose: (1, D, H, W) tensor
        alpha: deformation magnitude in voxels (before smoothing)
        sigma: Gaussian smoothing σ in voxels (larger = smoother)

    Returns:
        (density, dose) deformed with the same transformation
    """
    D, H, W = density.shape[1:]

    # Random displacement fields → smooth → scale
    dd = _gaussian_smooth_3d(torch.randn(D, H, W), sigma) * alpha
    dh = _gaussian_smooth_3d(torch.randn(D, H, W), sigma) * alpha
    dw = _gaussian_smooth_3d(torch.randn(D, H, W), sigma) * alpha

    # Base grid in normalized coordinates [-1, 1]
    grid_d = torch.linspace(-1, 1, D)
    grid_h = torch.linspace(-1, 1, H)
    grid_w = torch.linspace(-1, 1, W)
    mesh_d, mesh_h, mesh_w = torch.meshgrid(grid_d, grid_h, grid_w, indexing='ij')

    # Convert displacement from voxels to normalized coordinates
    # 1 voxel = 2.0 / (S - 1) in normalized coords [-1, 1] with align_corners=True
    norm_dw = dw * 2.0 / max(W - 1, 1)
    norm_dh = dh * 2.0 / max(H - 1, 1)
    norm_dd = dd * 2.0 / max(D - 1, 1)

    # F.grid_sample 5D: grid[..., 0]=x(W), [..1]=y(H), [..2]=z(D)
    grid = torch.stack([
        mesh_w + norm_dw,
        mesh_h + norm_dh,
        mesh_d + norm_dd,
    ], dim=-1).unsqueeze(0)  # (1, D, H, W, 3)

    # Apply SAME deformation to density and dose
    density_out = F.grid_sample(
        density.unsqueeze(0).float(), grid,
        mode='bilinear', padding_mode='border', align_corners=True
    ).squeeze(0)

    dose_out = F.grid_sample(
        dose.unsqueeze(0).float(), grid,
        mode='bilinear', padding_mode='border', align_corners=True
    ).squeeze(0)

    return density_out, dose_out

def _rotate_3d(density, dose, max_deg):
    """
    Small random 3D rotation (density + dose together).

    Applies random rotations around the 3 principal axes:
      - Axis D (axial rotation): ±max_deg
      - Axis H (coronal tilt): ±max_deg
      - Axis W (sagittal nod): ±max_deg

    Uses Euler angles ZYX and F.grid_sample to interpolate.
    With max_deg=5° the maximum geometric error is ~2-3mm at the
    edges of the volume (48×64×64 at 2mm/voxel), within
    IGRT setup uncertainty (±3-5° typical).
    Ref: Klarenberg 2025 uses rotations up to 10° en CBCT-to-CT.

    Args:
        density: (1, D, H, W) tensor
        dose: (1, D, H, W) tensor
        max_deg: maximum rotation angle in degrees (per axis)

    Returns:
        (density, dose) rotated with the same transformation
    """
    D, H, W = density.shape[1:]

    # Random angles in radians
    angles_deg = torch.empty(3).uniform_(-max_deg, max_deg)
    a, b, c = (angles_deg * 3.14159265 / 180.0).unbind(0)  # ZYX Euler

    # Rotation matrices per axis
    cos_a, sin_a = torch.cos(a), torch.sin(a)
    cos_b, sin_b = torch.cos(b), torch.sin(b)
    cos_c, sin_c = torch.cos(c), torch.sin(c)

    # R = Rz(a) @ Ry(b) @ Rx(c)
    R = torch.tensor([
        [cos_a*cos_b,  cos_a*sin_b*sin_c - sin_a*cos_c,  cos_a*sin_b*cos_c + sin_a*sin_c],
        [sin_a*cos_b,  sin_a*sin_b*sin_c + cos_a*cos_c,  sin_a*sin_b*cos_c - cos_a*sin_c],
        [-sin_b,       cos_b*sin_c,                       cos_b*cos_c],
    ])

    # Base grid in normalized coordinates [-1, 1]
    grid_d = torch.linspace(-1, 1, D)
    grid_h = torch.linspace(-1, 1, H)
    grid_w = torch.linspace(-1, 1, W)
    mesh_d, mesh_h, mesh_w = torch.meshgrid(grid_d, grid_h, grid_w, indexing='ij')

    # Apply inverse rotation (for grid_sample: where to sample the input)
    R_inv = R.T  # R es ortogonal → R⁻¹ = Rᵀ
    new_w = R_inv[0, 0]*mesh_w + R_inv[0, 1]*mesh_h + R_inv[0, 2]*mesh_d
    new_h = R_inv[1, 0]*mesh_w + R_inv[1, 1]*mesh_h + R_inv[1, 2]*mesh_d
    new_d = R_inv[2, 0]*mesh_w + R_inv[2, 1]*mesh_h + R_inv[2, 2]*mesh_d

    # grid_sample expects (N, D, H, W, 3) with [x=W, y=H, z=D]
    grid = torch.stack([new_w, new_h, new_d], dim=-1).unsqueeze(0)

    density_out = F.grid_sample(
        density.unsqueeze(0).float(), grid,
        mode='bilinear', padding_mode='zeros', align_corners=True
    ).squeeze(0)

    dose_out = F.grid_sample(
        dose.unsqueeze(0).float(), grid,
        mode='bilinear', padding_mode='zeros', align_corners=True
    ).squeeze(0)

    return density_out, dose_out


def _scale_3d(density, dose, scale_range):
    """
    Random isotropic 3D scaling (density + dose together).

    Aplica un factor de escala uniforme s ∈ [lo, hi]:
      - s > 1 → zoom in (larger anatomy, samples central part)
      - s < 1 → zoom out (smaller anatomy, borders with padding)

    With scale_range=(0.95, 1.05) the change is ±5%, corresponding to
    ~3mm of variation in a 128mm volume. This simulates
    inter-patient anatomical variations (body size, CTV expansion).
    Ref: nnU-Net v2 uses scaling of 0.7-1.4 for segmentation.
    We use a much more conservative range to avoid invalidating beam params.

    Args:
        density: (1, D, H, W) tensor
        dose: (1, D, H, W) tensor
        scale_range: (lo, hi) — scale factor range

    Returns:
        (density, dose) scaled with the same transformation
    """
    D, H, W = density.shape[1:]

    s = torch.empty(1).uniform_(scale_range[0], scale_range[1]).item()

    # Base grid in normalized coordinates [-1, 1]
    grid_d = torch.linspace(-1, 1, D)
    grid_h = torch.linspace(-1, 1, H)
    grid_w = torch.linspace(-1, 1, W)
    mesh_d, mesh_h, mesh_w = torch.meshgrid(grid_d, grid_h, grid_w, indexing='ij')

    # Scale = divide coordinates by s (zoom in if s>1 → more central sampling)
    grid = torch.stack([
        mesh_w / s,
        mesh_h / s,
        mesh_d / s,
    ], dim=-1).unsqueeze(0)  # (1, D, H, W, 3)

    # Apply SAME scale to density and dose
    density_out = F.grid_sample(
        density.unsqueeze(0).float(), grid,
        mode='bilinear', padding_mode='zeros', align_corners=True
    ).squeeze(0)

    dose_out = F.grid_sample(
        dose.unsqueeze(0).float(), grid,
        mode='bilinear', padding_mode='zeros', align_corners=True
    ).squeeze(0)

    return density_out, dose_out

def _simulate_lowres_3d(density, scale_range, per_axis=True):
    """
    Simulates CT acquisition with lower resolution (nnU-Net v2).

    Reduces density resolution (downsample → upsample) to
    simulate patients scanned with different slice thickness or
    different in-plane resolution. This forces the network to be
    robusta ante variaciones de calidad de imagen.

    Only applies to density (not dose): dose is TPS output,
    always at the same resolution.

    Args:
        density: (1, D, H, W) tensor
        scale_range: (lo, hi) downscale factor per axis (1.0=no change, 2.0=half res)
        per_axis: if True, each axis has independent factor (simulates anisotropy)

    Returns:
        density degraded and restored to original resolution
    """
    _, D, H, W = density.shape

    if per_axis:
        # Independent factor per axis → simulates anisotropic acquisition
        sd = torch.empty(1).uniform_(scale_range[0], scale_range[1]).item()
        sh = torch.empty(1).uniform_(scale_range[0], scale_range[1]).item()
        sw = torch.empty(1).uniform_(scale_range[0], scale_range[1]).item()
    else:
        s = torch.empty(1).uniform_(scale_range[0], scale_range[1]).item()
        sd = sh = sw = s

    # Reduced resolution (at least 2 voxels per axis)
    new_D = max(int(round(D / sd)), 2)
    new_H = max(int(round(H / sh)), 2)
    new_W = max(int(round(W / sw)), 2)

    # If no real change, skip
    if (new_D, new_H, new_W) == (D, H, W):
        return density

    # Downsample → Upsample (using F.interpolate with batch dim)
    low = F.interpolate(
        density.unsqueeze(0), size=(new_D, new_H, new_W),
        mode='trilinear', align_corners=False
    )
    restored = F.interpolate(
        low, size=(D, H, W),
        mode='trilinear', align_corners=False
    ).squeeze(0)

    return restored


def _coarse_dropout_3d(density, cube_size_range, num_cubes_range, fill_value=0.0):
    """
    Coarse Dropout 3D — simulates metallic artifacts or missing data.

    Places random cubes of constant value (fill_value) in the volume
    of density. Simulates regions where the CT has artifacts (prostheses,
    implants, surgical clips) that corrupt the signal. Forces the network
    to interpolate dose in regions without density information.

    Only applies to density (not dose): real dose has no artifacts
    (is TPS output with manual correction).

    Refs: nnU-Net v2 (CoarseDropout), Shojaei 2025 (data corruption).

    Args:
        density: (1, D, H, W) tensor
        cube_size_range: (min_side, max_side) cube size in voxels
        num_cubes_range: (min_n, max_n) number of cubes to place
        fill_value: valor de relleno (0.0 = aire)

    Returns:
        density with cubes removed
    """
    _, D, H, W = density.shape
    min_side, max_side = cube_size_range
    min_n, max_n = num_cubes_range

    n_cubes = torch.randint(min_n, max_n + 1, (1,)).item()

    for _ in range(n_cubes):
        # Random cube size (can be non-cubic for more variety)
        sd = torch.randint(min_side, max_side + 1, (1,)).item()
        sh = torch.randint(min_side, max_side + 1, (1,)).item()
        sw = torch.randint(min_side, max_side + 1, (1,)).item()

        # Random position (cube can partially extend outside → clamp)
        d0 = torch.randint(0, max(D - sd + 1, 1), (1,)).item()
        h0 = torch.randint(0, max(H - sh + 1, 1), (1,)).item()
        w0 = torch.randint(0, max(W - sw + 1, 1), (1,)).item()

        density[:, d0:d0+sd, h0:h0+sh, w0:w0+sw] = fill_value

    return density



class AugmentedDataset(Dataset):
    """
    Augmentation wrapper for the TRAINING set.

    Wraps a Dataset o Subset y aplica augmentaciones on-the-fly
    that do NOT change the model architecture → checkpoint compatible.

    Augmentation pipeline (in order of application):

      GEOMETRIC (density + dose together, same transformation):
        1. Random 3D translation (±N voxels) — patient position uncertainty
        2. Small 3D rotation (±5°) — IGRT setup error [NEW: Klarenberg 2025]
        3. Isotropic scaling (±5%) — anatomy variation [NEW: nnU-Net v2]
        4. Smooth 3D elastic deformation — inter-patient anatomical variation

      INTENSITY (density only):
        5. Gaussian blur (σ 0.3-1.0) — CT resolution variation [nnU-Net v2]
        6. Low resolution simulation — different CT slice thickness [NEW: nnU-Net v2]
        7. Gamma transform (γ 0.85-1.25) — contrast variation [nnU-Net v2, UQ paper]
        8. Gaussian noise additive (random σ 0-0.02, p=0.5)
        9. Intensity scaling (±10%, p=0.5)
       10. Additive brightness (±0.03) — CT calibration drift [NEW: UQ paper]
       11. Coarse Dropout 3D — metallic artifacts / missing data [NEW: nnU-Net v2]

      DESACTIVADAS:
       12. 3D Flips — invalidate beam geometry

    Beam params are NOT modified: deformations are small,
    within the clinical positioning uncertainty (±5-10mm IGRT,
    ±3-5° setup error). Refs: nnU-Net v2 (Isensee 2021), Klarenberg 2025,
    Shojaei 2025, Zhang 2024, Foster 2025.

    Why a separate wrapper?
      - persistent_workers=True freezes the dataset state in each worker.
        A .training flag in EBRTDataset would not propagate correctly.
      - With a wrapper, train_loader uses AugmentedDataset and val/test_loader
        use the original Subset, with no possibility of bugs.
    """

    def __init__(self, base_dataset, cfg: Config):
        """
        Args:
            base_dataset: base Dataset or Subset (without augmentation)
            cfg: Config with all augmentation parameters
        """
        self.base = base_dataset

        # ── Intensidad ──
        self.noise_std = cfg.augment_noise_std
        self.noise_prob = cfg.augment_noise_prob
        self.intensity_lo, self.intensity_hi = cfg.augment_intensity_range
        self.intensity_prob = cfg.augment_intensity_prob
        self.do_blur = cfg.augment_blur
        self.blur_sigma_lo, self.blur_sigma_hi = cfg.augment_blur_sigma
        self.blur_prob = cfg.augment_blur_prob
        self.do_gamma = cfg.augment_gamma
        self.gamma_lo, self.gamma_hi = cfg.augment_gamma_range
        self.gamma_prob = cfg.augment_gamma_prob

        # ── Geometric ──
        self.do_translate = cfg.augment_translate
        self.translate_range = cfg.augment_translate_range
        self.translate_prob = cfg.augment_translate_prob
        self.do_elastic = cfg.augment_elastic
        self.elastic_alpha = cfg.augment_elastic_alpha
        self.elastic_sigma = cfg.augment_elastic_sigma
        self.elastic_prob = cfg.augment_elastic_prob
        self.do_rotate = cfg.augment_rotate
        self.rotate_max_deg = cfg.augment_rotate_max_deg
        self.rotate_prob = cfg.augment_rotate_prob
        self.do_scale = cfg.augment_scale
        self.scale_range = cfg.augment_scale_range
        self.scale_prob = cfg.augment_scale_prob
        self.do_brightness = cfg.augment_brightness
        self.brightness_lo, self.brightness_hi = cfg.augment_brightness_range
        self.brightness_prob = cfg.augment_brightness_prob
        self.do_lowres = cfg.augment_lowres
        self.lowres_scale = cfg.augment_lowres_scale
        self.lowres_prob = cfg.augment_lowres_prob
        self.lowres_per_axis = cfg.augment_lowres_per_axis
        self.do_coarse_dropout = cfg.augment_coarse_dropout
        self.coarse_dropout_cube_size = cfg.augment_coarse_dropout_cube_size
        self.coarse_dropout_num_cubes = cfg.augment_coarse_dropout_num_cubes
        self.coarse_dropout_fill = cfg.augment_coarse_dropout_fill
        self.coarse_dropout_prob = cfg.augment_coarse_dropout_prob
        self.do_flip = cfg.augment_flip

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        sample = self.base[idx]

        # Clone density and dose to avoid modifying shared RAM arrays
        density = sample['density'].clone()
        dose = sample['dose'].clone()

        # ═══════════════════════════════════════
        # GEOMETRIC (density + dose together)
        # ═══════════════════════════════════════

        # ── 1. Random 3D translation ──
        if self.do_translate and torch.rand(1).item() < self.translate_prob:
            density, dose = _translate_3d(density, dose, self.translate_range)

        # ── 2. Small 3D rotation (±5°) ── [NEW]
        if self.do_rotate and torch.rand(1).item() < self.rotate_prob:
            density, dose = _rotate_3d(density, dose, self.rotate_max_deg)

        # ── 3. Isotropic scaling (±5%) ── [NEW]
        if self.do_scale and torch.rand(1).item() < self.scale_prob:
            density, dose = _scale_3d(density, dose, self.scale_range)

        # ── 4. 3D elastic deformation ──
        if self.do_elastic and torch.rand(1).item() < self.elastic_prob:
            density, dose = _elastic_deform_3d(
                density, dose, self.elastic_alpha, self.elastic_sigma
            )

        # ═══════════════════════════════════════
        # INTENSITY (density only)
        # ═══════════════════════════════════════

        # ── 5. Gaussian blur ── [NEW]
        if self.do_blur and torch.rand(1).item() < self.blur_prob:
            sigma = torch.empty(1).uniform_(self.blur_sigma_lo, self.blur_sigma_hi).item()
            density = _gaussian_smooth_3d(
                density.squeeze(0), sigma
            ).unsqueeze(0)

        # ── 6. Low resolution simulation ── [NEW: nnU-Net v2]
        if self.do_lowres and torch.rand(1).item() < self.lowres_prob:
            density = _simulate_lowres_3d(
                density, self.lowres_scale, self.lowres_per_axis
            )

        # ── 7. Gamma transform ── [NEW]
        if self.do_gamma and torch.rand(1).item() < self.gamma_prob:
            gamma = torch.empty(1).uniform_(self.gamma_lo, self.gamma_hi).item()
            # density is normalized ~[0,1] → pow(gamma) is safe
            # Pre-clamp to avoid NaN with negative values (after noise)
            density = density.clamp(min=0.0).pow(gamma)

        # ── 8. Additive Gaussian noise (density only) ──
        if self.noise_std > 0 and torch.rand(1).item() < self.noise_prob:
            # random σ ∈ [0, noise_std] → variety of noise levels
            actual_std = torch.empty(1).uniform_(0, self.noise_std).item()
            density = density + torch.randn_like(density) * actual_std

        # ── 9. Random intensity scaling (density only) ──
        if torch.rand(1).item() < self.intensity_prob:
            scale = torch.empty(1).uniform_(self.intensity_lo, self.intensity_hi).item()
            density = density * scale

        # ── 10. Additive brightness offset ── [NEW: UQ paper + nnU-Net v2]
        if self.do_brightness and torch.rand(1).item() < self.brightness_prob:
            offset = torch.empty(1).uniform_(self.brightness_lo, self.brightness_hi).item()
            density = density + offset

        # ── 11. Coarse Dropout 3D ── [NEW: nnU-Net v2]
        if self.do_coarse_dropout and torch.rand(1).item() < self.coarse_dropout_prob:
            density = _coarse_dropout_3d(
                density, self.coarse_dropout_cube_size,
                self.coarse_dropout_num_cubes, self.coarse_dropout_fill
            )

        # ── 12. Random 3D flips (DISABLED by default) ──
        if self.do_flip:
            for spatial_dim in [1, 2, 3]:  # D, H, W (dim 0=canal)
                if torch.rand(1).item() < 0.5:
                    density = density.flip(spatial_dim)
                    dose = dose.flip(spatial_dim)

        # Clamp to physically reasonable range (densities ≥ 0)
        density = density.clamp(min=0.0)

        sample['density'] = density
        sample['dose'] = dose

        return sample


# ── Augmentation summary ──
print("✅ EBRTDataset class defined (multi-HDF5)")
print(f"   Pre-load to RAM: {'Yes (with resize + normalization)' if CFG.preload_to_ram else 'No (lazy I/O)'}")
print("   Supports float16 (density) and float32 (dose) automatically")

print("✅ AugmentedDataset defined (wrapper for training)")
print("   ── Augmentation pipeline (refs: nnU-Net v2, Klarenberg 2025, Shojaei 2025) ──")
print("   GEOMETRIC (density+dose together):")
if CFG.augment_translate:
    print(f"     1. 3D Translation: ±{CFG.augment_translate_range} vxl, p={CFG.augment_translate_prob}")
if CFG.augment_rotate:
    print(f"     2. 3D Rotation:   ±{CFG.augment_rotate_max_deg}°, p={CFG.augment_rotate_prob} [NEW]")
if CFG.augment_scale:
    print(f"     3. 3D Scaling:   {CFG.augment_scale_range}, p={CFG.augment_scale_prob} [NEW]")
if CFG.augment_elastic:
    print(f"     4. Elastic:      α={CFG.augment_elastic_alpha}, σ={CFG.augment_elastic_sigma}, p={CFG.augment_elastic_prob}")
print("   INTENSITY (density only):")
if CFG.augment_blur:
    print(f"     5. Gaussian blur: σ={CFG.augment_blur_sigma}, p={CFG.augment_blur_prob}")
if CFG.augment_lowres:
    print(f"     6. Low-res sim:   scale={CFG.augment_lowres_scale}, p={CFG.augment_lowres_prob} [NEW]")
if CFG.augment_gamma:
    print(f"     7. Gamma:         γ={CFG.augment_gamma_range}, p={CFG.augment_gamma_prob}")
print(f"     8. Gauss noise:   σ∈[0,{CFG.augment_noise_std}], p={CFG.augment_noise_prob}")
print(f"     9. Intensidad:    {CFG.augment_intensity_range}, p={CFG.augment_intensity_prob}")
if CFG.augment_brightness:
    print(f"    10. Brightness:    {CFG.augment_brightness_range}, p={CFG.augment_brightness_prob} [NEW]")
if CFG.augment_coarse_dropout:
    print(f"    11. CoarseDropout: cubes={CFG.augment_coarse_dropout_cube_size}, n={CFG.augment_coarse_dropout_num_cubes}, p={CFG.augment_coarse_dropout_prob} [NEW]")
print(f"   DESACTIVADAS:")
print(f"    12. 3D Flips: {'Yes' if CFG.augment_flip else 'No (invalidates beam params)'}")




In [ ]:
"""
CELL 6: Create training, validation, and test DataLoaders
================================================================
Dividimos el dataset en train (75%), validation (15%) y test (10%).

SPLIT INTELIGENTE:
  1. If there is NO previous split → creates a new one with the correct proportions
  2. If there IS a previous split and the dataset grew:
     a) If it grew >30% → PROPORTIONAL RE-SPLIT: the new samples are
        distributed among train/val/test maintaining the original %% ratios.
        Old val/test are preserved (never lost), but new ones are added
        so that validation is representative of the complete dataset.
     b) If it grew ≤30% → new ones go only to train (val/test split remains
        statistically representative)
  3. force_new_split=True → ignores any previous split y crea uno
     completely new. USE when changing the data generator.

Indices are saved to disk (split_indices.json) for reproducibility
between Kaggle sessions.

KEY OPTIMIZATION (v2):
  The dataset is loaded to RAM ONCE. The train/val/test subsets
  are created as views (torch.utils.data.Subset) over the same
  arrays in memory, without re-reading the HDF5 files. This reduces the time
  de carga de ~7 horas a ~35 minutos.
"""

# ══════════════════════════════════════════════════════════════
# ⚠️ CONTROL: Force a new split from scratch
# Enable when: (1) you change the MC generator, (2) you want a clean split
# If True, ignores any previous split_indices.json
force_new_split = False  # ← Change to False after the first successful training
# ══════════════════════════════════════════════════════════════

# ─── Crear dataset completo ───
# With preload_to_ram=True, this SINGLE call loads ALL data to RAM
# (~9.6 GB for 6080 samples). The train/val/test subsets will be views
# over these same arrays (Subset), without re-reading the HDF5 files.
full_dataset = EBRTDataset(CFG, mode='auto')
n_total = len(full_dataset)

# ─── Search for previous split ───
split_path = os.path.join(BASE_OUTPUT, CFG.split_indices_name)

split_found = None
if not force_new_split:
    # Search in /kaggle/input/ (uploaded with the checkpoint)
    if IN_KAGGLE:
        for dirpath, _, filenames in os.walk('/kaggle/input'):
            if CFG.split_indices_name in filenames:
                split_found = os.path.join(dirpath, CFG.split_indices_name)
                break
    if split_found is None and os.path.exists(split_path):
        split_found = split_path
else:
    print("  ⚠️ force_new_split=True → ignoring any previous split")

# ─── Split logic ───
if split_found is not None:
    # Load previous split
    with open(split_found, 'r') as f:
        saved_split = json.load(f)

    old_n = saved_split.get('n_total_at_creation', len(saved_split['train']) +
                            len(saved_split['val']) + len(saved_split['test']))
    n_new = n_total - old_n

    if n_new <= 0:
        # Dataset did not grow → use split as is
        train_indices = saved_split['train']
        val_indices = saved_split['val']
        test_indices = saved_split['test']
        print(f"  📂 Split loaded from: {split_found}")
        print(f"     Dataset unchanged ({n_total} samples)")
        print(f"     Train: {len(train_indices)} | Val: {len(val_indices)} | Test: {len(test_indices)}")

    elif n_new / old_n > 0.30:
        # Dataset grew >30% → PROPORTIONAL RE-SPLIT
        # Preserve old val/test + add proportion of new ones
        old_set = set(saved_split['train']) | set(saved_split['val']) | set(saved_split['test'])
        new_indices = [i for i in range(n_total) if i not in old_set]
        np.random.seed(SEED + old_n)  # Reproducible seed but different from original
        np.random.shuffle(new_indices)

        # Repartir nuevas proporcionalmente
        n_new_test = int(len(new_indices) * CFG.test_fraction)
        n_new_val = int(len(new_indices) * CFG.val_fraction)
        n_new_train = len(new_indices) - n_new_val - n_new_test

        new_train = new_indices[:n_new_train]
        new_val = new_indices[n_new_train:n_new_train + n_new_val]
        new_test = new_indices[n_new_train + n_new_val:]

        train_indices = saved_split['train'] + new_train
        val_indices = saved_split['val'] + new_val
        test_indices = saved_split['test'] + new_test

        print(f"  📂 Split loaded from: {split_found}")
        print(f"  🔄 PROPORTIONAL RE-SPLIT (dataset grew {n_new/old_n:.0%})")
        print(f"     New samples: {len(new_indices)} → "
              f"Train +{n_new_train}, Val +{n_new_val}, Test +{n_new_test}")
        print(f"     Train: {len(saved_split['train'])} + {n_new_train} = {len(train_indices)}")
        print(f"     Val:   {len(saved_split['val'])} + {n_new_val} = {len(val_indices)}")
        print(f"     Test:  {len(saved_split['test'])} + {n_new_test} = {len(test_indices)}")

    else:
        # Dataset grew ≤30% → new ones go only to train (val/test still representative)
        old_set = set(saved_split['train']) | set(saved_split['val']) | set(saved_split['test'])
        new_indices = [i for i in range(n_total) if i not in old_set]
        train_indices = saved_split['train'] + new_indices
        val_indices = saved_split['val']
        test_indices = saved_split['test']

        print(f"  📂 Split loaded from: {split_found}")
        print(f"     Small growth ({n_new/old_n:.0%}) → new ones go only to train")
        print(f"     Train: {len(saved_split['train'])} + {len(new_indices)} = {len(train_indices)}")
        print(f"     Val:   {len(val_indices)} (unchanged)")
        print(f"     Test:  {len(test_indices)} (unchanged)")

else:
    # Create completely new split
    all_indices = list(range(n_total))
    np.random.seed(SEED)
    np.random.shuffle(all_indices)

    n_test = int(n_total * CFG.test_fraction)
    n_val = int(n_total * CFG.val_fraction)

    train_indices = all_indices[:n_total - n_val - n_test]
    val_indices = all_indices[n_total - n_val - n_test:n_total - n_test]
    test_indices = all_indices[n_total - n_test:]

    print(f"  🆕 NEW split created (seed={SEED}):")
    print(f"     Train: {len(train_indices)} ({100*len(train_indices)/n_total:.0f}%) | "
          f"Val: {len(val_indices)} ({100*len(val_indices)/n_total:.0f}%) | "
          f"Test: {len(test_indices)} ({100*len(test_indices)/n_total:.0f}%)")

# Save split to disk (always, for the next session)
split_data = {
    'train': train_indices,
    'val': val_indices,
    'test': test_indices,
    'n_total_at_creation': n_total,
    'seed': SEED,
}
with open(split_path, 'w') as f:
    json.dump(split_data, f)
print(f"  💾 Split saved en: {split_path}")

n_train = len(train_indices)
n_val = len(val_indices)
n_test = len(test_indices)

# ─── KEY OPTIMIZATION: Subset instead of re-loading ───
# full_dataset already has ALL data in RAM (preload_to_ram=True).
# Instead of creating 3 new datasets (which would re-read ALL HDF5
# → ~7 horas de I/O redundante!), usamos torch.utils.data.Subset
# which simply remaps indices over the same arrays in RAM.
# Result: 0 seconds of additional I/O, same total memory.
train_subset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)
test_dataset = Subset(full_dataset, test_indices)

# ─── DATA AUGMENTATION (training only) ───
# Wrap train_subset with AugmentedDataset to apply augmentations
# on-the-fly during training. Val/test do NOT carry augmentation.
# AugmentedDataset clones density before modifying → does not corrupt shared RAM.
train_dataset = AugmentedDataset(train_subset, CFG)

print(f"\n  ⚡ Subsets created as VIEWS over dataset in RAM (0 additional I/O)")
print(f"  🎲 Data augmentation activa en training:")
print(f"     • Gaussian noise: σ={CFG.augment_noise_std}")
print(f"     • Intensity scaling: [{CFG.augment_intensity_range[0]}, {CFG.augment_intensity_range[1]}]")
if CFG.augment_translate:
    print(f"     • 3D Translation: ±{CFG.augment_translate_range} voxels (prob={CFG.augment_translate_prob})")
if CFG.augment_elastic:
    print(f"     • Elastic deformation: α={CFG.augment_elastic_alpha}, σ={CFG.augment_elastic_sigma} (prob={CFG.augment_elastic_prob})")

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,           # Shuffle each epoch
    num_workers=CFG.num_workers,  # Parallel data pre-loading
    pin_memory=(DEVICE.type == 'cuda'),
    drop_last=True,         # Discard last incomplete batch (BatchNorm stability)
    persistent_workers=True,  # Reuse workers between epochs (saves overhead)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=True,
)

# Verify a sample
sample = next(iter(train_loader))
print(f"\n📊 DATALOADERS CREATED")
print(f"   Train: {n_train} samples → {len(train_loader)} batches of {CFG.batch_size}")
print(f"   Val:   {n_val} samples → {len(val_loader)} batches of {CFG.batch_size}")
print(f"   Test:  {n_test} samples → {len(test_loader)} batches of {CFG.batch_size}")
print(f"\n   Shape de un batch:")
print(f"   • density: {sample['density'].shape} — (B, C=1, D, H, W)")
print(f"   • params:  {sample['params'].shape}  — (B, N_params)")
print(f"   • dose:    {sample['dose'].shape} — (B, C=1, D, H, W)")
print(f"   • dose_max: {sample['dose_max'].shape} — (B,)")
print(f"\n   Memory per batch: {sample['density'].nelement() * 4 / 1e6:.1f} MB (density) + "
      f"{sample['dose'].nelement() * 4 / 1e6:.1f} MB (dose)")
print(f"   num_workers: {CFG.num_workers} | persistent_workers: True")
print(f"   Pre-carga RAM: {CFG.preload_to_ram}")

---
# PART 3: MODEL ARCHITECTURE — 3D TransUNet with FiLM
---

## 3.1 — Architecture Overview

The architecture combines three fundamental ideas from modern deep learning:

### A) 3D U-Net (Ronneberger et al., 2015 — adapted to 3D by Çiçek et al., 2016)

The U-Net is the standard in medical image segmentation. Its encoder-decoder structure with **skip connections** preserves fine spatial detail:

```
                                    Skip connections
                                 ┌─────────────────────┐
                                 │                     │
  Input (1,48,64,64)             │                     │     Output (1,48,64,64)
      │                          │                     │         ▲
      ▼                          │                     │         │
  ┌────────┐    ┌────────┐      │   ┌────────┐    ┌────────┐   │
  │Conv3D  │───▶│Conv3D  │──────┼──▶│FiLM    │───▶│Conv3D  │───┘
  │32 feat │    │64 feat │      │   │+UpConv │    │32 feat │
  │↓×2     │    │↓×2     │      │   │↑×2     │    │↑×2     │
  └────────┘    └────────┘      │   └────────┘    └────────┘
                    │           │       ▲
                    ▼           │       │
               ┌────────┐      │   ┌────────┐
               │Conv3D  │──────┘──▶│FiLM    │
               │128 feat│          │+UpConv │
               │↓×2     │          │↑×2     │
               └────────┘          └────────┘
                    │                  ▲
                    ▼                  │
               ┌────────┐         ┌────────┐
               │Conv3D  │────────▶│FiLM    │
               │256 feat│         │+UpConv │
               │↓×2     │         │↑×2     │
               └────────┘         └────────┘
                    │                  ▲
                    ▼                  │
              ┌──────────────┐        │
              │  TRANSFORMER │────────┘
              │  BOTTLENECK  │
              │  (3,4,4)=48  │
              │  tokens      │
              └──────────────┘
                    ▲
                    │
              Beam parameters
              (energy, angle, ...)
              injected via FiLM
```

### B) Transformer in the Bottleneck (Chen et al., 2021 — TransUNet)

Why a Transformer here and not more CNN?

**Problem**: Photons interact along their **entire** path. A bone at 5cm depth absorbs radiation that no longer reaches deeper tissues. This is a **long-range** dependency.

**CNNs**: have a local receptive field. A 3×3×3 convolution only "sees" immediate neighbors. Even with pooling, the effective receptive field grows slowly.

**Transformers**: The **self-attention** mechanism allows each position to attend to ALL other positions in the volume. Thus, a tumor voxel at 2cm can directly "query" a bone voxel at 6cm.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where $Q, K, V$ are linear projections of the bottleneck tokens.

### C) FiLM Conditioning (Perez et al., 2018)

**Feature-wise Linear Modulation** modulates the decoder features according to beam parameters:

$$\text{FiLM}(F, \gamma, \beta) = \gamma \odot F + \beta$$

Where:
- $F$ is the feature tensor at a decoder level
- $\gamma, \beta$ are learned vectors from beam parameters via a shared MLP
- $\odot$ is element-wise multiplication per channel

**Why FiLM and not simply concatenating the parameters?**
- Concatenating parameters as channels is **additive**: it cannot modulate multiplicatively. A high-energy beam doesn't just *add* dose, it changes the *shape* of the distribution.
- FiLM is **multiplicative + additive**: $\gamma$ scales features (amplifies/suppresses), $\beta$ shifts them. This is exactly what the physics requires.
- FiLM is applied at **each level** of the decoder, allowing modulation at different spatial scales.

### D) Instance Normalization (instead of Batch Normalization)

With batches of 2-4 3D volumes, BatchNorm statistics are very noisy. Instance Normalization computes mean and variance **per sample and per channel** independently, remaining stable with any batch size.

### E) Deep Supervision

We add auxiliary losses at intermediate decoder outputs (after upsampling to each resolution). This:
- Provides more direct gradients to early decoder layers
- Acts as regularization
- Auxiliary loss weights decrease: $\lambda_{aux} = 0.5^{level}$

### F) MC Dropout for Uncertainty Estimation (Gal & Ghahramani, 2016)

Standard neural networks output a single point prediction. In clinical dosimetry, we also need to know **how confident** the model is. MC Dropout provides epistemic uncertainty by:

1. Keeping `Dropout3D` active during inference (`model.train()` mode, but with no gradient computation)
2. Running $T$ forward passes (default: 20) with different dropout masks
3. Computing the **mean** prediction (better than a single pass) and the **variance** per voxel

$$\hat{D}(\mathbf{r}) = \frac{1}{T}\sum_{t=1}^T D_t(\mathbf{r}), \qquad \text{Var}(\mathbf{r}) = \frac{1}{T}\sum_{t=1}^T (D_t(\mathbf{r}) - \hat{D}(\mathbf{r}))^2$$

High variance regions indicate areas where the model is uncertain (e.g., tissue interfaces, field edges, heterogeneous anatomy). This information is used in Stage 2 (optimizer) to quantify the confidence of the optimal plan.

### G) 3D Separable Convolutions

To reduce the parameter count while maintaining spatial expressiveness, encoder blocks use **depthwise separable convolutions**: a depthwise convolution (per-channel spatial filtering) followed by a pointwise 1×1×1 convolution (channel mixing). This reduces parameters by a factor of ~$k^3/C_{out}$ compared to standard convolutions.

### H) Weight Initialization

Bad initialization can cause the network not to converge (especially deep 3D architectures). All Conv3D layers are initialized with **Kaiming normal** (He et al., 2015), and biases are set to zero. The FiLM γ and β projections are initialized so that $\gamma \approx 1$ and $\beta \approx 0$ at the start — this means the network initially ignores beam parameters and learns a "base" dose distribution, then gradually learns to modulate it.

In [ ]:
"""
CELL 7: Model building blocks
============================================
We define the reusable components that form the network.
Each block is a documented nn.Module.
"""

# ═══════════════════════════════════════════════════════════
# BLOCK 1: ConvBlock3D — Double 3D convolution with normalization
# ═══════════════════════════════════════════════════════════

class ConvBlock3D(nn.Module):
    """
    Basic 3D convolution block, used in encoder and decoder.

    Structure:
        Conv3D(3×3×3) → InstanceNorm → LeakyReLU →
        Conv3D(3×3×3) → InstanceNorm → LeakyReLU

    Why two consecutive convolutions?
        - A single 3×3×3 convolution has a receptive field of 3³=27 voxels
        - Two consecutive: effective receptive field 5³=125 voxels
        - More representation capacity without increasing stride

    Why LeakyReLU and not ReLU?
        - ReLU kills neurons (gradient=0 for x<0)
        - LeakyReLU(0.01) allows small gradients → better training
        - Standard in medical imaging networks (nnU-Net, MONAI)
    """

    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


# ═══════════════════════════════════════════════════════════
# BLOCK 2: DownBlock3D — Encoder level (downsample + conv)
# ═══════════════════════════════════════════════════════════

class DownBlock3D(nn.Module):
    """
    Encoder block: reduces resolution ×2 and increases features.

    Structure:
        MaxPool3D(2) → ConvBlock3D(in_ch → out_ch)

    Why MaxPool and not stride=2 in Conv?
        - MaxPool preserves the strongest activations (edges, interfaces)
        - stride=2 can lose information if the kernel doesn't cover well
        - Both work; MaxPool is more traditional in U-Net
    """

    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.conv = ConvBlock3D(in_ch, out_ch, dropout)

    def forward(self, x):
        x = self.pool(x)
        x = self.conv(x)
        return x


# ═══════════════════════════════════════════════════════════
# BLOCK 3: FiLMGenerator — Generates γ and β from beam parameters
# ═══════════════════════════════════════════════════════════

class FiLMGenerator(nn.Module):
    """
    Generates FiLM coefficients (γ, β) from beam parameters.

    Structure:
        params (10,) → Linear → ReLU → Linear → ReLU → Linear → (2·C,)
                                                                  ↓
                                                        γ = [:C],  β = [C:]

    Why an MLP and not a simple linear layer?
        - The relationship between beam parameters and feature modulation
          is NON-LINEAR. Example: doubling the energy does not double the dose
          in all voxels, it changes it in a complex way. The difference
          FF vs FFF modifies the entire lateral profile, not just the magnitude.
        - A 3-layer MLP can learn non-linear transformations.

    A FiLMGenerator is created for EACH decoder level.
    
    The 10 input parameters include:
        beam_energy, is_fff, field_x, field_y, ssd, gantry,
        collimator, patient_type, tumor_depth, tumor_size
    """

    def __init__(self, n_params: int, n_channels: int, hidden_dim: int = 128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(n_params, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, 2 * n_channels),  # γ y β concatenados
        )
        # Initialize γ≈1, β≈0 (identity at the start of training)
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)
        # γ bias initialized to 1 so that FiLM(F) ≈ F at start
        with torch.no_grad():
            self.mlp[-1].bias[:n_channels] = 1.0

        self.n_channels = n_channels

    def forward(self, params):
        """
        Args:
            params: (B, n_params) — normalized beam parameters

        Returns:
            gamma: (B, C, 1, 1, 1) — factor multiplicativo
            beta:  (B, C, 1, 1, 1) — additive factor
        """
        out = self.mlp(params)  # (B, 2C)
        gamma = out[:, :self.n_channels].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)  # (B, C, 1, 1, 1)
        beta = out[:, self.n_channels:].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return gamma, beta


# ═══════════════════════════════════════════════════════════
# BLOCK 4: UpBlock3D + FiLM — Decoder level
# ═══════════════════════════════════════════════════════════

class UpBlockFiLM3D(nn.Module):
    """
    Decoder block with FiLM conditioning.

    Structure:
        1. Upsample ×2 (trilinear interpolation)
        2. Concatenate with skip connection from encoder
        3. ConvBlock3D (reduces channels after concatenation)
        4. Apply FiLM: γ ⊙ features + β

    Why trilinear interpolation instead of ConvTranspose3D?
        - ConvTranspose3D can produce checkerboard artifacts
          due to kernel overlap
        - Trilinear interpolation + Conv3D is smoother and more stable
        - It is the modern standard (nnU-Net, MONAI, etc.)
    """

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int,
                 n_params: int, film_hidden: int, dropout: float = 0.0):
        """
        Args:
            in_ch: input channels (from the lower level)
            skip_ch: skip connection channels (from the encoder)
            out_ch: canales de salida
            n_params: number of beam parameters
            film_hidden: hidden dimension of the FiLM generator
        """
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='trilinear', align_corners=False)
        self.conv = ConvBlock3D(in_ch + skip_ch, out_ch, dropout)
        self.film = FiLMGenerator(n_params, out_ch, film_hidden)

    def forward(self, x, skip, params):
        """
        Args:
            x: (B, in_ch, D, H, W) — features from the lower level
            skip: (B, skip_ch, D*2, H*2, W*2) — skip connection from encoder
            params: (B, n_params) — beam parameters

        Returns:
            (B, out_ch, D*2, H*2, W*2) — features upsampled y moduladas
        """
        x = self.up(x)  # (B, in_ch, D*2, H*2, W*2)

        # Handle size differences (due to padding/indivisibility)
        if x.shape != skip.shape:
            diff_d = skip.shape[2] - x.shape[2]
            diff_h = skip.shape[3] - x.shape[3]
            diff_w = skip.shape[4] - x.shape[4]
            x = F.pad(x, [diff_w // 2, diff_w - diff_w // 2,
                          diff_h // 2, diff_h - diff_h // 2,
                          diff_d // 2, diff_d - diff_d // 2])

        x = torch.cat([x, skip], dim=1)  # (B, in_ch + skip_ch, ...)
        x = self.conv(x)                  # (B, out_ch, ...)

        # Apply FiLM conditioning
        gamma, beta = self.film(params)    # (B, out_ch, 1, 1, 1) each
        x = gamma * x + beta              # Feature-wise modulation

        return x


print("✅ Building blocks defined:")
print("   • ConvBlock3D:     Conv3D×2 + InstanceNorm + LeakyReLU")
print("   • DownBlock3D:     MaxPool + ConvBlock3D")
print("   • FiLMGenerator:   MLP(params) → (γ, β)")
print("   • UpBlockFiLM3D:   Upsample + Concat(skip) + ConvBlock3D + FiLM")

In [ ]:
"""
CELDA 8: Transformer Bottleneck
=================================
We implement the Transformer block that is placed at the bottom
of the U-Net, between the deepest encoder and the decoder.

What does the Transformer do here?
  - Receives the compressed 3D volume (256, 3, 4, 4) = 48 tokens
  - Each token is a "patch" of the volume with 256 features
  - Self-attention allows each patch to attend to all others
  - Captures global dependencies (e.g.: bone → dosimetric shadow)
  - Returns the volume with the same dimensions but enriched
    with global information

Components:
  1. PatchEmbedding: (B, C, D, H, W) → (B, N, embed_dim) — tokeniza
  2. PositionalEncoding: adds learnable positional information
  3. TransformerBlock ×L: self-attention + MLP
  4. Reshape back: (B, N, embed_dim) → (B, C, D, H, W)
"""

class LearnablePositionalEncoding3D(nn.Module):
    """
    Learnable positional encoding for 3D tokens.

    Why learnable and not sinusoidal?
      - El encoding sinusoidal (Vaswani 2017) asume periodicidad,
        but human anatomy is NOT periodic
      - A learnable encoding can capture spatial relationships
        arbitrary (e.g.: the "left lung" position has no
        periodic relationship with "right lung")
      - With only 48 tokens (3×4×4), the parameter cost is minimal
    """

    def __init__(self, n_tokens: int, embed_dim: int):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, n_tokens, embed_dim) * 0.02)

    def forward(self, x):
        return x + self.pos_embed


class MultiHeadSelfAttention3D(nn.Module):
    """
    Multi-Head Self-Attention for 3D tokens.

    Attention = softmax(Q·K^T / √d_k) · V

    Multi-Head: dividimos embed_dim en num_heads cabezas independientes.
    Each head can learn different attention patterns:
      - Head 1: attention to similar densities
      - Head 2: attention to spatial proximity
      - Head 3: attention to material interfaces
      - etc.
    """

    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert embed_dim % num_heads == 0, \
            f"embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})"

        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x):
        """
        Args:
            x: (B, N, C) — N tokens of dimension C

        Returns:
            (B, N, C) — tokens transformados
        """
        B, N, C = x.shape

        # Compute Q, K, V in a single operation
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, H, N, d_k)
        q, k, v = qkv.unbind(0)             # 3 × (B, H, N, d_k)

        # Scaled attention
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, H, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # Apply attention to values
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)  # (B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class TransformerBlock(nn.Module):
    """
    A standard Transformer block (Pre-Norm).

    Structure:
        LayerNorm → MHSA → residual →
        LayerNorm → MLP  → residual

    Why Pre-Norm (LayerNorm before attention)?
        - Post-Norm (original) has gradient problems in deep networks
        - Pre-Norm is more stable and does not need aggressive learning rate warmup
        - It is the standard in ViT, BERT, GPT, etc.

    The internal MLP expands and contracts the features:
        C → 4C → C (ratio=4.0)
    This allows the model to learn non-linear transformations
    more complex than a single linear layer.
    """

    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: float = 4.0,
                 dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention3D(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)

        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),  # GELU is smoother than ReLU, standard in Transformers
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))  # Residual + Attention
        x = x + self.mlp(self.norm2(x))   # Residual + MLP
        return x


class TransformerBottleneck(nn.Module):
    """
    Bottleneck completo: tokenizar → Transformer → reshape.

    Flujo:
        (B, C, D, H, W)  →  flatten  →  (B, D*H*W, C)  →
        + positional encoding  →
        TransformerBlock × L  →
        reshape  →  (B, C, D, H, W)
    """

    def __init__(self, in_channels: int, spatial_dims: tuple,
                 num_heads: int = 8, num_layers: int = 4,
                 mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()
        self.spatial_dims = spatial_dims  # (D, H, W) of the bottleneck
        n_tokens = spatial_dims[0] * spatial_dims[1] * spatial_dims[2]

        self.pos_enc = LearnablePositionalEncoding3D(n_tokens, in_channels)
        self.blocks = nn.ModuleList([
            TransformerBlock(in_channels, num_heads, mlp_ratio, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(in_channels)

    def forward(self, x):
        """
        Args:
            x: (B, C, D, H, W)
        Returns:
            (B, C, D, H, W)
        """
        B, C, D, H, W = x.shape

        # Tokenizar: (B, C, D, H, W) → (B, N, C)
        x = rearrange(x, 'b c d h w -> b (d h w) c')

        # Add positional encoding
        x = self.pos_enc(x)

        # Pass through Transformer blocks
        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # Reconstruct volume: (B, N, C) → (B, C, D, H, W)
        x = rearrange(x, 'b (d h w) c -> b c d h w', d=D, h=H, w=W)

        return x


print("✅ Transformer Bottleneck defined:")
print("   • LearnablePositionalEncoding3D: learnable positions for 3D tokens")
print("   • MultiHeadSelfAttention3D: multi-head attention O(N²)")
print("   • TransformerBlock: Pre-Norm with MHSA + MLP + residuals")
print("   • TransformerBottleneck: tokenization + L blocks + reshape")

In [ ]:
"""
CELL 9: Full model — TransUNetFiLM3D
=============================================
We assemble all blocks into the complete network.
This is the main class that is trained and used for inference.
"""

class TransUNetFiLM3D(nn.Module):
    """
    3D TransUNet with FiLM Conditioning for EBRT dose prediction.

    Entrada:
        - density: (B, 1, D, H, W)  — normalized patient density map
        - params:  (B, 10)          — 10 TrueBeam SVC 3.0 beam parameters normalized [0,1]
                                       [energy, FFF, field_x, field_y, SSD, gantry, collimator,
                                        tipo_paciente, prof_tumor, tam_tumor]

    Salida:
        - dose:    (B, 1, D, H, W)  — predicted dose distribution [0, 1]
        - aux:     lista de predicciones intermedias (si deep_supervision=True)

    Arquitectura:
        ENCODER (4 levels of 3D CNN with downsampling)
            → TRANSFORMER BOTTLENECK (global self-attention over 3D tokens)
            → DECODER with FiLM (4 levels of upsampling + FiLM conditioning)
            → Final head (Conv 1×1×1 → Sigmoid)
    """

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        enc_ch = cfg.encoder_channels  # (1, 32, 64, 128, 256)
        dec_ch = cfg.decoder_channels  # (256, 128, 64, 32)

        # ═══════════════════════════════════════
        # ENCODER (with dropout for MC Dropout)
        # ═══════════════════════════════════════
        drop = cfg.model_dropout  # 3D Dropout for MC Dropout inference

        # Level 0: input (1 channel) → 32 features (NO downsampling)
        self.enc0 = ConvBlock3D(enc_ch[0], enc_ch[1], dropout=drop)

        # Levels 1-3: each one downsamples ×2
        self.enc1 = DownBlock3D(enc_ch[1], enc_ch[2], dropout=drop)  # 32 → 64
        self.enc2 = DownBlock3D(enc_ch[2], enc_ch[3], dropout=drop)  # 64 → 128
        self.enc3 = DownBlock3D(enc_ch[3], enc_ch[4], dropout=drop)  # 128 → 256

        # ═══════════════════════════════════════
        # TRANSFORMER BOTTLENECK
        # ═══════════════════════════════════════
        # Compute spatial dimensions of the bottleneck
        # Input: (D, H, W) → after 3 downsamples ×2: (D/8, H/8, W/8)
        bt_d = cfg.vol_depth // 8
        bt_h = cfg.vol_height // 8
        bt_w = cfg.vol_width // 8
        self.bottleneck_dims = (bt_d, bt_h, bt_w)

        self.transformer = TransformerBottleneck(
            in_channels=enc_ch[4],  # 256
            spatial_dims=self.bottleneck_dims,
            num_heads=cfg.trans_num_heads,
            num_layers=cfg.trans_num_layers,
            mlp_ratio=cfg.trans_mlp_ratio,
            dropout=cfg.trans_dropout,
        )

        # ═══════════════════════════════════════
        # DECODER with FiLM
        # ═══════════════════════════════════════
        self.dec3 = UpBlockFiLM3D(dec_ch[0], enc_ch[3], dec_ch[1],
                                   cfg.n_beam_params, cfg.film_hidden_dim, dropout=drop)
        self.dec2 = UpBlockFiLM3D(dec_ch[1], enc_ch[2], dec_ch[2],
                                   cfg.n_beam_params, cfg.film_hidden_dim, dropout=drop)
        self.dec1 = UpBlockFiLM3D(dec_ch[2], enc_ch[1], dec_ch[3],
                                   cfg.n_beam_params, cfg.film_hidden_dim, dropout=drop)

        # ═══════════════════════════════════════
        # CABEZA DE SALIDA
        # ═══════════════════════════════════════
        # Conv 1×1×1 → 1-channel dose map
        self.head = nn.Sequential(
            nn.Conv3d(dec_ch[3], dec_ch[3], kernel_size=1),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(dec_ch[3], 1, kernel_size=1),
            nn.Sigmoid(),  # Normalized dose ∈ [0, 1]
        )

        # ═══════════════════════════════════════
        # DEEP SUPERVISION (auxiliary outputs)
        # ═══════════════════════════════════════
        if cfg.deep_supervision:
            self.aux_heads = nn.ModuleList([
                nn.Sequential(
                    nn.Conv3d(ch, 1, kernel_size=1),
                    nn.Sigmoid()
                )
                for ch in [dec_ch[1], dec_ch[2]]  # Niveles 3→2 y 2→1
            ])

        self._initialize_weights()

    def _initialize_weights(self):
        """
        Weight initialization — Kaiming for Conv, Xavier for Linear.

        Why does initialization matter?
          - Bad initialization can cause the network not to converge
          - Kaiming (He) is optimal for networks with ReLU/LeakyReLU
          - Xavier (Glorot) is optimal for linear layers without activation
        """
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.InstanceNorm3d, nn.LayerNorm)):
                if m.weight is not None:
                    nn.init.ones_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, density, params):
        """
        Forward pass completo.

        Args:
            density: (B, 1, D, H, W) — normalized density map
            params:  (B, 10) — beam parameters normalized [0,1]

        Returns:
            dose: (B, 1, D, H, W) — predicted dose distribution [0,1]
            aux_outputs: lista de predicciones intermedias (deep supervision)
        """
        # ═══ ENCODER ═══
        e0 = self.enc0(density)    # (B, 32, D, H, W)
        e1 = self.enc1(e0)         # (B, 64, D/2, H/2, W/2)
        e2 = self.enc2(e1)         # (B, 128, D/4, H/4, W/4)
        e3 = self.enc3(e2)         # (B, 256, D/8, H/8, W/8)

        # ═══ TRANSFORMER BOTTLENECK ═══
        t = self.transformer(e3)   # (B, 256, D/8, H/8, W/8) — enriched with global info

        # ═══ DECODER with FiLM ═══
        d3 = self.dec3(t, e2, params)    # (B, 128, D/4, H/4, W/4)
        d2 = self.dec2(d3, e1, params)   # (B, 64, D/2, H/2, W/2)
        d1 = self.dec1(d2, e0, params)   # (B, 32, D, H, W)

        # ═══ FINAL HEAD ═══
        dose = self.head(d1)       # (B, 1, D, H, W)

        # ═══ DEEP SUPERVISION ═══
        aux_outputs = []
        if self.cfg.deep_supervision and self.training:
            # Auxiliary predictions at intermediate resolutions
            aux3 = self.aux_heads[0](d3)  # (B, 1, D/4, H/4, W/4)
            aux2 = self.aux_heads[1](d2)  # (B, 1, D/2, H/2, W/2)
            aux_outputs = [aux3, aux2]

        return dose, aux_outputs

    def count_parameters(self):
        """Counts and categorizes model parameters."""
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)

        encoder_params = sum(p.numel() for n, p in self.named_parameters() if 'enc' in n)
        transformer_params = sum(p.numel() for n, p in self.named_parameters() if 'transformer' in n)
        decoder_params = sum(p.numel() for n, p in self.named_parameters() if 'dec' in n)
        film_params = sum(p.numel() for n, p in self.named_parameters() if 'film' in n)
        head_params = sum(p.numel() for n, p in self.named_parameters() if 'head' in n)

        return {
            'total': total, 'trainable': trainable,
            'encoder': encoder_params, 'transformer': transformer_params,
            'decoder': decoder_params, 'film': film_params, 'head': head_params,
        }


print("✅ TransUNetFiLM3D defined — main surrogate model")

In [ ]:
"""
CELL 10: Instantiate model and verify dimensions
=====================================================
We create the model, verify that dimensions are correct
with a test forward pass, and show the parameter distribution.
"""

# Instanciar
model = TransUNetFiLM3D(CFG).to(DEVICE)

# ── torch.compile: CUDA kernel fusion for 5-15% speedup ──
# Requiere PyTorch ≥ 2.0 y CUDA Capability ≥ 7.0 (Triton backend).
# P100 = CC 6.0 → NOT compatible with Triton → automatically disabled.
# On GPUs ≥ V100 (CC 7.0+) it is enabled for kernel fusion.
_use_compile = (
    DEVICE.type == 'cuda'
    and hasattr(torch, 'compile')
    and torch.cuda.get_device_capability(DEVICE)[0] >= 7
)
if _use_compile:
    try:
        model = torch.compile(model, mode='default')
        print("  ⚡ torch.compile enabled (CUDA kernel fusion)")
    except Exception as e:
        print(f"  ⚠️ torch.compile failed: {e}")
        print("     Continuing without compilation (no impact on results)")
elif DEVICE.type == 'cuda':
    _cc = torch.cuda.get_device_capability(DEVICE)
    print(f"  ℹ️ torch.compile disabled (GPU CC {_cc[0]}.{_cc[1]} < 7.0, Triton no soportado)")
    print("     No impact on results — only ~5-15% speed loss")

# Test forward pass
with torch.no_grad():
    test_density = torch.randn(1, 1, CFG.vol_depth, CFG.vol_height, CFG.vol_width).to(DEVICE)
    test_params = torch.rand(1, CFG.n_beam_params).to(DEVICE)
    test_dose, test_aux = model(test_density, test_params)

print("🏗️  MODEL: TransUNetFiLM3D")
print("=" * 60)
print(f"\n  📥 Input density:   {test_density.shape}")
print(f"  📥 Input params:    {test_params.shape}")
print(f"  📤 Output dose:     {test_dose.shape}")
print(f"  📤 Output range:    [{test_dose.min():.4f}, {test_dose.max():.4f}] (Sigmoid → [0,1])")
if test_aux:
    for i, aux in enumerate(test_aux):
        print(f"  📤 Aux output {i}:    {aux.shape}")

# Count parameters
param_counts = model.count_parameters()
print(f"\n  📊 PARAMETERS:")
print(f"  {'Component':<20} {'Parameters':>12} {'%':>6}")
print(f"  {'-'*40}")
for name in ['encoder', 'transformer', 'decoder', 'film', 'head']:
    count = param_counts[name]
    pct = 100 * count / param_counts['total']
    print(f"  {name.capitalize():<20} {count:>12,} {pct:>5.1f}%")
print(f"  {'─'*40}")
print(f"  {'TOTAL':<20} {param_counts['total']:>12,}")
print(f"  {'Trainable':<20} {param_counts['trainable']:>12,}")

# Memory estimation
mem_model = param_counts['total'] * 4 / 1e6  # float32
mem_grad = mem_model * 3  # Gradients + optimizer states (Adam)
print(f"\n  💾 Model memory:           ~{mem_model:.0f} MB")
print(f"  💾 + gradients + Adam:   ~{mem_grad:.0f} MB")
print(f"  💾 + activaciones (est.):  ~{mem_grad * 2:.0f} MB")

# Verify bottleneck
print(f"\n  🔄 Bottleneck Transformer:")
print(f"     Dimensiones espaciales: {model.bottleneck_dims}")
n_tokens = model.bottleneck_dims[0] * model.bottleneck_dims[1] * model.bottleneck_dims[2]
print(f"     Number of tokens: {n_tokens}")
print(f"     Complejidad self-attention: O({n_tokens}²) = O({n_tokens**2:,})")
print(f"     {CFG.trans_num_layers} layers × {CFG.trans_num_heads} heads")

del test_density, test_params, test_dose, test_aux
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

print("\n✅ Model verified — dimensions correct")

In [ ]:
# ============================================================================
#  MC DROPOUT — PREDICTION UNCERTAINTY ESTIMATION
# ============================================================================
#  Method: Gal & Ghahramani, "Dropout as a Bayesian Approximation:
#          Representing Model Uncertainty in Deep Learning" (ICML, 2016)
#
#  Instead of storing the Monte Carlo uncertainty σ_D/D from the
#  simulations (which only reflects MC statistical noise, not the
#  model uncertainty), we use MC Dropout:
#
#  1. Keep Dropout3D active during inference (model.train())
#  2. We run N stochastic forward passes with torch.no_grad()
#  3. The MEAN of the N predictions = best dose estimate
#  4. The STD of the N predictions = model uncertainty map
#
#  Advantages over stored MC uncertainty:
#  - No requiere almacenamiento extra (~30% ahorro en disco)
#  - Captures EPISTEMIC uncertainty (what the model doesn't know)
#  - MC uncertainty only captures RANDOM noise from the simulation
#  - Applicable to any new data without needing to re-simulate
# ============================================================================

@torch.no_grad()
def mc_dropout_predict(
    model: nn.Module,
    density: torch.Tensor,     # (B, 1, D, H, W)
    params: torch.Tensor,      # (B, n_params)
    n_samples: int | None = None,
    device: torch.device | None = None,
) -> dict[str, torch.Tensor]:
    """
    Prediction with MC Dropout for epistemic uncertainty estimation.

    Runs `n_samples` forward passes with active dropout to obtain
    a distribution of predictions per voxel.

    Args:
        model:     TransUNetFiLM3D model (must have Dropout3d layers).
        density:   Input density tensor (B, 1, D, H, W).
        params:    Irradiation parameters (B, n_params).
        n_samples: Number of stochastic passes. Default: CFG.mc_dropout_samples.
        device:    Compute device. Default: inferred from the model.

    Returns:
        dict with keys:
            'mean':  (B, 1, D, H, W) — Best dose estimate (mean).
            'std':   (B, 1, D, H, W) — Epistemic uncertainty (std).
            'cv':    (B, 1, D, H, W) — Coefficient of variation σ/μ (relative uncertainty).
            'all':   (n_samples, B, 1, D, H, W) — All individual predictions.
    """
    if n_samples is None:
        n_samples = CFG.mc_dropout_samples

    if device is None:
        device = next(model.parameters()).device

    density = density.to(device)
    params = params.to(device)

    # ── Save original model state ──
    was_training = model.training

    # ── Activate ONLY dropout (not BatchNorm in train mode) ──
    # We set the model to eval to freeze BatchNorm, then
    # manually reactivate only Dropout3d modules
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout3d):
            m.train()

    # ── N stochastic passes ──
    predictions = []
    for _ in range(n_samples):
        pred, _aux = model(density, params)
        predictions.append(pred.cpu())

    # ── Restore original state ──
    model.train(was_training)

    # ── Statistics ──
    all_preds = torch.stack(predictions, dim=0)   # (N, B, 1, D, H, W)
    mean_pred = all_preds.mean(dim=0)              # (B, 1, D, H, W)
    std_pred = all_preds.std(dim=0)                # (B, 1, D, H, W)

    # Coefficient of variation (avoid div/0 in regions without dose)
    cv_pred = std_pred / (mean_pred.abs() + 1e-8)

    return {
        'mean': mean_pred,
        'std': std_pred,
        'cv': cv_pred,
        'all': all_preds,
    }


def print_mc_dropout_summary(results: dict, dose_threshold_frac: float = 0.1):
    """Prints uncertainty summary MC Dropout."""
    mean = results['mean']
    std = results['std']
    cv = results['cv']
    n_samples = results['all'].shape[0]

    # Mask of region with significant dose (>threshold% of Dmax)
    dmax = mean.max()
    high_dose_mask = mean > (dose_threshold_frac * dmax)

    print(f"\n{'='*60}")
    print(f"  📊 MC DROPOUT — EPISTEMIC UNCERTAINTY")
    print(f"{'='*60}")
    print(f"  Stochastic passes: {n_samples}")
    print(f"  Dropout rate: {CFG.model_dropout:.1%}")
    print(f"\n  --- Full region ---")
    print(f"  σ media:    {std.mean():.6f}")
    print(f"  Max σ:      {std.max():.6f}")
    print(f"  Mean CV:    {cv.mean():.4f} ({cv.mean()*100:.2f}%)")
    print(f"\n  --- Dose region >{dose_threshold_frac:.0%} Dmax ---")
    if high_dose_mask.any():
        print(f"  Voxels:     {high_dose_mask.sum():,}")
        print(f"  σ media:    {std[high_dose_mask].mean():.6f}")
        print(f"  Mean CV:    {cv[high_dose_mask].mean():.4f} ({cv[high_dose_mask].mean()*100:.2f}%)")
        print(f"  Max CV:     {cv[high_dose_mask].max():.4f} ({cv[high_dose_mask].max()*100:.2f}%)")
    else:
        print(f"  (no voxels above threshold)")
    print(f"{'='*60}")


print("✅ MC Dropout — uncertainty inference functions ready")
print(f"   → mc_dropout_predict(model, density, params, n_samples={CFG.mc_dropout_samples})")
print(f"   → Model dropout rate: {CFG.model_dropout:.1%}")

---
# PART 4: LOSS FUNCTION — PHYSICS-INFORMED
---

## 4.1 — Why Is MSE Not Enough?

The standard **Mean Squared Error** treats all voxels equally:

$$\mathcal{L}_{MSE} = \frac{1}{N}\sum_{i}(D_{pred,i} - D_{true,i})^2$$

Problems:
1. **Does not weight by clinical relevance**: a 5% error in a low-dose region (bremsstrahlung tail) matters much less than a 5% error in the tumor
2. **Does not penalize gradient errors**: the slope of the dose curve (the "sharpness" of the distal fall-off) is clinically critical. MSE can yield a smooth profile that measures well on average but is clinically unacceptable
3. **Does not capture structure**: two distributions can have similar MSE but one with artifacts and the other smooth

## 4.2 — Our Loss: 5 Components

$$\mathcal{L}_{total} = \lambda_1 \mathcal{L}_{MSE}^{w} + \lambda_2 \mathcal{L}_{grad} + \lambda_3 \mathcal{L}_{SSIM} + \lambda_4 \mathcal{L}_{DVH} + \lambda_5 \mathcal{L}_{falloff}$$

| Component | $\lambda$ | What does it penalize? | Why is it necessary? |
|---|---|---|---|
| $\mathcal{L}_{MSE}^{w}$ | 1.0 | Point-to-point error (dose-weighted) | Baseline — general reconstruction |
| $\mathcal{L}_{grad}$ | 0.5 | Difference in spatial dose gradients | Preserves isodose boundaries and PDD slope |
| $\mathcal{L}_{SSIM}$ | 0.2 | Difference in structure (luminance, contrast, similarity) | Perceptual quality, avoids artifacts |
| $\mathcal{L}_{DVH}$ | 0.3 | Difference in DVH metrics ($D_{95}$, $D_{max}$, $D_{mean}$) | Direct clinical relevance |
| $\mathcal{L}_{falloff}$ | 0.2 | Error in the penumbra and fall-off region | Critical zone for OAR protection |

### Detail of Each Component

#### $\mathcal{L}_{MSE}^{w}$ — Weighted MSE
$$\mathcal{L}_{MSE}^{w} = \frac{1}{N}\sum_{i} w_i \cdot (D_{pred,i} - D_{true,i})^2$$

The weights $w_i$ are proportional to the dose:
$$w_i = 1 + \alpha \cdot D_{true,i}$$

Thus, voxels with high dose (near the beam) are weighted more than background voxels. $\alpha$ is calibrated so that $\langle w \rangle \approx 2$.

#### $\mathcal{L}_{grad}$ — 3D Gradient Loss
$$\mathcal{L}_{grad} = \frac{1}{N}\sum_{i}\left|\nabla D_{pred,i} - \nabla D_{true,i}\right|^2$$

The gradient is computed by finite differences: $\partial D/\partial x \approx D_{i+1} - D_{i-1}$ along each axis.

#### $\mathcal{L}_{SSIM}$ — Structural Similarity 3D
$$\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

Computed by 3D sliding window. The loss is $1 - \text{SSIM}$.

#### $\mathcal{L}_{DVH}$ — DVH-Based Loss
$$\mathcal{L}_{DVH} = |D_{95,pred} - D_{95,true}| + |D_{max,pred} - D_{max,true}| + |D_{mean,pred} - D_{mean,true}|$$

Differentiable thanks to `torch.quantile` and `torch.topk`.

#### $\mathcal{L}_{falloff}$ — Dose Fall-off Loss
Penalizes differences in the zone where the dose drops from 80% to 20% of the maximum (the "distal penumbra").

In [ ]:
"""
CELL 11: Physics-Informed Loss Function Implementation
================================================================
Each component is an independent function. They are combined in
PhysicsInformedLoss which manages weights and summation.

Components (7):
  1. weighted_mse_loss: dose-weighted MSE (high-dose voxels are more important)
  2. gradient_loss_3d: penalizes errors in spatial dose gradients
  3. ssim_loss_3d: 3D SSIM for structural quality
  4. dvh_loss: dose-volume histogram difference
  5. dose_falloff_loss: extra precision in the penumbra zone
  6. cosine_similarity_loss: global shape alignment (new)
"""

def weighted_mse_loss(pred, target, dose_weight_alpha=5.0):
    """
    Weighted MSE: high-dose voxels are more important.

    The weighting is: w_i = 1 + α·D_true_i
    This amplifies the error in the high dose gradient zone (where
    the beam is) without completely ignoring the background zone.

    Args:
        pred:   (B, 1, D, H, W)
        target: (B, 1, D, H, W)
        dose_weight_alpha: amplification factor

    Returns:
        scalar loss
    """
    weights = 1.0 + dose_weight_alpha * target  # (B, 1, D, H, W)
    sq_error = (pred - target) ** 2
    weighted_sq_error = weights * sq_error
    return weighted_sq_error.mean()


def gradient_loss_3d(pred, target):
    """
    Penalizes differences in spatial dose gradients.

    Computes ∂D/∂x, ∂D/∂y, ∂D/∂z by central finite differences
    and measures the MSE between predicted and real gradients.

    Why does this matter?
      - The dose gradient determines the "sharpness" of the isodose lines
      - In EBRT, the field penumbra and the depth fall-off
        must be reproduced accurately
      - A model that smooths the penumbra can have good MSE but
        would be clinically unacceptable

    In EBRT with variable gantry, all three axes contribute equally
    (isotropic gradients).

    Args/Returns: same as weighted_mse_loss
    """
    # Gradients along each axis by finite differences
    # ∂D/∂z (depth)
    grad_pred_z = pred[:, :, 1:, :, :] - pred[:, :, :-1, :, :]
    grad_true_z = target[:, :, 1:, :, :] - target[:, :, :-1, :, :]

    # ∂D/∂y
    grad_pred_y = pred[:, :, :, 1:, :] - pred[:, :, :, :-1, :]
    grad_true_y = target[:, :, :, 1:, :] - target[:, :, :, :-1, :]

    # ∂D/∂x
    grad_pred_x = pred[:, :, :, :, 1:] - pred[:, :, :, :, :-1]
    grad_true_x = target[:, :, :, :, 1:] - target[:, :, :, :, :-1]

    # MSE between predicted and real gradients (per axis)
    loss_z = F.mse_loss(grad_pred_z, grad_true_z)
    loss_y = F.mse_loss(grad_pred_y, grad_true_y)
    loss_x = F.mse_loss(grad_pred_x, grad_true_x)

    # Uniform weight: in EBRT with variable gantry, gradients are
    # equally important in all axes (there is no privileged axis)
    # AVERAGE (not sum) so the effective weight is λ=0.5, not ~1.5
    return (loss_z + loss_y + loss_x) / 3.0


def ssim_loss_3d(pred, target, window_size=5, data_range=1.0, cached_window=None):
    """
    Structural Similarity Index (SSIM) Loss en 3D.

    SSIM measures the perceptual similarity between two images/volumes
    based on three components:
      1. Luminance: do they have similar means?
      2. Contrast: do they have similar variances?
      3. Structure: are they correlated?

    Computed with a 3D Gaussian sliding window.

    Args:
        pred, target: (B, 1, D, H, W) en [0, 1]
        window_size: window size (odd)
        data_range: value range (1.0 if normalized)

    Returns:
        1 - SSIM (so it is a loss to minimize)
    """
    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2

    # 3D Gaussian window
    def gaussian_window_3d(size, sigma=1.5):
        coords = torch.arange(size, dtype=torch.float32) - size // 2
        g = torch.exp(-coords**2 / (2 * sigma**2))
        g = g / g.sum()
        window = g[:, None, None] * g[None, :, None] * g[None, None, :]
        return window.unsqueeze(0).unsqueeze(0)  # (1, 1, K, K, K)

    # IMPORTANT: disable autocast to force true FP32
    # torch.amp.autocast overrides .float() inside its context,
    # so we need enabled=False for F.conv3d to use FP32
    with torch.amp.autocast('cuda', enabled=False):
        pred = pred.float()
        target = target.float()

        # Use pre-cached window if provided (avoids recreating each batch)
        if cached_window is not None:
            window = cached_window.to(pred.device)
        else:
            window = gaussian_window_3d(window_size).to(pred.device)
        pad = window_size // 2

        mu_pred = F.conv3d(pred, window, padding=pad)
        mu_target = F.conv3d(target, window, padding=pad)

        mu_pred_sq = mu_pred ** 2
        mu_target_sq = mu_target ** 2
        mu_pred_target = mu_pred * mu_target

        sigma_pred_sq = F.conv3d(pred ** 2, window, padding=pad) - mu_pred_sq
        sigma_target_sq = F.conv3d(target ** 2, window, padding=pad) - mu_target_sq
        sigma_pred_target = F.conv3d(pred * target, window, padding=pad) - mu_pred_target

        # Clamp variances to avoid negative values from numerical errors
        sigma_pred_sq = torch.clamp(sigma_pred_sq, min=0.0)
        sigma_target_sq = torch.clamp(sigma_target_sq, min=0.0)

        ssim_map = ((2 * mu_pred_target + C1) * (2 * sigma_pred_target + C2)) / \
                   ((mu_pred_sq + mu_target_sq + C1) * (sigma_pred_sq + sigma_target_sq + C2))

        return 1.0 - ssim_map.mean()

def dvh_loss(pred, target, n_points=20):
    """
    Loss based on Dose-Volume Histogram (DVH) metrics.

    The DVH is THE clinical evaluation tool. Penalizing
    DVH differences ensures that clinical indicators
    (D95, D_max, D_mean) are correct even if the dose map
    has small point-to-point differences.

    Vectorized: avoids Python loop (20 kernel launches → 1).

    Args:
        pred, target: (B, 1, D, H, W) en [0, 1]

    Returns:
        loss escalar
    """
    dose_levels = torch.linspace(0.05, 0.95, n_points, device=pred.device)
    # Reshape for broadcasting: (n_points, 1, 1, 1, 1, 1)
    levels = dose_levels.view(-1, 1, 1, 1, 1, 1)
    steepness = 50.0

    # pred/target: (B, 1, D, H, W) → (1, B, 1, D, H, W) for broadcast with levels
    vol_pred = torch.sigmoid(steepness * (pred.unsqueeze(0) - levels)).mean(dim=(1, 2, 3, 4, 5))
    vol_true = torch.sigmoid(steepness * (target.unsqueeze(0) - levels)).mean(dim=(1, 2, 3, 4, 5))

    return ((vol_pred - vol_true) ** 2).mean()

def dose_falloff_loss(pred, target):
    """
    Penalizes errors in the beam fall-off zone (penumbra).

    It is clinically critical in EBRT:
      - Defines the edge of the treatment field
      - The penumbra must be accurate to protect adjacent OARs
      - Errors in this zone can lead to tumor underdosing
        or critical organ overdosing

    Implementation: mask that selects voxels in the range [0.2, 0.8]
    and applies MSE with extra weight.

    Args:
        pred, target: (B, 1, D, H, W) en [0, 1]
    """
    # Smooth mask of the fall-off zone
    falloff_mask = torch.sigmoid(20 * (target - 0.15)) * torch.sigmoid(20 * (0.85 - target))

    sq_error = (pred - target) ** 2
    weighted = falloff_mask * sq_error
    # Normalize by the number of voxels in the zone
    n_falloff = falloff_mask.sum() + 1e-8
    return weighted.sum() / n_falloff


def cosine_similarity_loss(pred, target):
    """
    Loss based on cosine similarity between pred and target.

    Measures the global shape alignment between dose volumes,
    regardless of scale (complementary to MSE).

    Cosine similarity = cos(θ) between the flattened vectors of pred and target.
    Loss = 1 - cos(θ), minimized when the vectors point in the same direction.

    Why is it useful?
      - MSE penalizes point-to-point errors, but can converge to a
        solution with the correct scale but the wrong "shape".
      - Cosine similarity forces the model to reproduce the SPATIAL
        DISTRIBUTION of the dose, not just its average magnitude.
      - Complements SSIM: SSIM is local (window 5×5×5), cosine is global.

    Args:
        pred, target: (B, 1, D, H, W) en [0, 1]

    Returns:
        scalar loss in [0, 2] (0 = identical vectors, 2 = opposite)
    """
    # Flatten to (B, N) where N = D*H*W (channel is already 1)
    p_flat = pred.flatten(1)
    t_flat = target.flatten(1)
    # F.cosine_similarity returns (B,), we average over the batch
    return 1.0 - F.cosine_similarity(p_flat, t_flat, dim=1).mean()


class PhysicsInformedLoss(nn.Module):
    """
    Combined loss with 6 components + deep supervision.

    Combines all terms with configurable weights:
        L_total = λ₁·L_MSE_w + λ₂·L_grad + λ₃·L_SSIM + λ₄·L_DVH + λ₅·L_falloff + λ₆·L_cosine
                  + Σ 0.5^k · L_aux_k (deep supervision)
    """

    def __init__(self, cfg: Config):
        super().__init__()
        self.w_mse = cfg.loss_mse_weight
        self.w_grad = cfg.loss_gradient_weight
        self.w_ssim = cfg.loss_ssim_weight
        self.w_dvh = cfg.loss_dvh_weight
        self.w_falloff = cfg.loss_falloff_weight
        self.w_cosine = cfg.loss_cosine_weight
        self.w_l1 = cfg.loss_l1_weight
        self.deep_supervision = cfg.deep_supervision

        # ── Smooth loss transition (protects pre-trained model) ──
        # When resuming training, new components (L1, gradient/3)
        # are introduced gradually to avoid 'shock' in the weights.
        self.transition_epochs = cfg.loss_transition_epochs
        self._transition_start = None  # Epoch where transition starts
        self._current_epoch = 0

        # Pre-cache 3D Gaussian window for SSIM (avoids recreating each batch)
        # register_buffer: moves to GPU with .to(device) and saves in state_dict
        ssim_window_size = 5
        coords = torch.arange(ssim_window_size, dtype=torch.float32) - ssim_window_size // 2
        g = torch.exp(-coords**2 / (2 * 1.5**2))
        g = g / g.sum()
        window = g[:, None, None] * g[None, :, None] * g[None, None, :]
        self.register_buffer('_ssim_window', window.unsqueeze(0).unsqueeze(0))  # (1,1,K,K,K)

    def set_epoch(self, epoch):
        """Updates current epoch to compute transition factor."""
        self._current_epoch = epoch

    def start_transition(self, start_epoch):
        """Activates smooth transition from start_epoch.
        Call this when resuming from a checkpoint trained with a different loss.
        L1 ramps from 0→full and the gradient compensates sum→mean gradually."""
        self._transition_start = start_epoch
        print(f"  🔄 Loss transition activated: epochs {start_epoch}→{start_epoch + self.transition_epochs}")
        print(f"     L1: 0.0 → {self.w_l1:.2f} | Gradient: ×3 → ×1 (sum→mean compensation)")

    def _get_transition_factor(self):
        """Factor 0→1 over transition_epochs. If no transition, returns 1.0."""
        if self._transition_start is None:
            return 1.0
        elapsed = self._current_epoch - self._transition_start
        if elapsed < 0:
            return 0.0
        return min(1.0, elapsed / max(self.transition_epochs, 1))

    def forward(self, pred, target, aux_preds=None):
        """
        Args:
            pred: (B, 1, D, H, W) — main prediction
            target: (B, 1, D, H, W) — ground truth
            aux_preds: lista de predicciones intermedias (deep supervision)

        Returns:
            total_loss: scalar
            loss_dict: dictionary with each component (for logging)
        """
        # Ensure target has the same dims as pred
        if target.shape != pred.shape:
            target = F.interpolate(target, size=pred.shape[2:], mode='trilinear',
                                   align_corners=False)

        # Compute each component
        l_mse = weighted_mse_loss(pred, target)
        l_grad = gradient_loss_3d(pred, target)
        l_ssim = ssim_loss_3d(pred, target, cached_window=self._ssim_window)
        l_dvh = dvh_loss(pred, target)
        l_falloff = dose_falloff_loss(pred, target)
        l_cosine = cosine_similarity_loss(pred, target)
        l_l1 = F.l1_loss(pred, target)  # Global MAE — paper Zhao et al. (MD Anderson)

        # Weighted total loss (with smooth transition if active)
        t = self._get_transition_factor()

        # Gradient: compensates sum→mean change (/3.0)
        # Old loss: w_grad * sum_of_3 = w_grad * 3 * mean
        # New loss: w_grad * mean
        # Factor: 3*(1-t) + 1*t = 3 - 2t, va de ×3 (old) a ×1 (new)
        grad_factor = 3.0 - 2.0 * t

        # L1: ramp from 0 → full weight
        l1_factor = t

        total = (self.w_mse * l_mse +
                 self.w_grad * grad_factor * l_grad +
                 self.w_ssim * l_ssim +
                 self.w_dvh * l_dvh +
                 self.w_falloff * l_falloff +
                 self.w_cosine * l_cosine +
                 self.w_l1 * l1_factor * l_l1)

        # Deep supervision: auxiliary losses at lower resolutions
        if self.deep_supervision and aux_preds:
            for k, aux_pred in enumerate(aux_preds):
                # Downsample target to aux resolution
                aux_target = F.interpolate(target, size=aux_pred.shape[2:],
                                           mode='trilinear', align_corners=False)
                aux_loss = F.mse_loss(aux_pred, aux_target)
                weight = 0.5 ** (k + 1)  # 0.5, 0.25, 0.125, ...
                total = total + weight * aux_loss

        loss_dict = {
            'total': total.item(),
            'mse_w': l_mse.item(),
            'gradient': l_grad.item(),
            'ssim': l_ssim.item(),
            'dvh': l_dvh.item(),
            'falloff': l_falloff.item(),
            'cosine': l_cosine.item(),
            'l1': l_l1.item(),
            'transition': t,
        }

        return total, loss_dict

# Instanciar
criterion = PhysicsInformedLoss(CFG)

# Quick test
with torch.no_grad():
    test_p = torch.rand(1, 1, 16, 16, 16)
    test_t = torch.rand(1, 1, 16, 16, 16)
    test_loss, test_dict = criterion(test_p, test_t)
print("✅ PhysicsInformedLoss defined with 7 components:")
print(f"   λ_MSE={CFG.loss_mse_weight}, λ_grad={CFG.loss_gradient_weight}, "
      f"λ_falloff={CFG.loss_falloff_weight}, λ_cosine={CFG.loss_cosine_weight}, "
      f"λ_L1={CFG.loss_l1_weight}")

print(f"\n   Test loss: {test_loss.item():.4f}")
for k, v in test_dict.items():
    print(f"   • {k}: {v:.4f}")

---
# PART 5: TRAINING
---

## 5.1 — Training Strategy

### Optimizer: AdamW
AdamW (Adam with decoupled Weight Decay, Loshchilov & Hutter 2019) is the standard for Transformers. Unlike Adam, the weight decay in AdamW does not interact with the adaptive learning rate, providing better regularization.

### Learning Rate Schedule: OneCycleLR with Automatic Warm Restarts

```
LR
 ↑
max_lr───╭──╮                    ╭──╮
 |       ╱    ╲                  ╱    ╲      ← max_lr × 0.7
 |      ╱      ╲                ╱      ╲
 |     ╱  cos   ╲              ╱  cos   ╲
 |    ╱  anneal  ╲            ╱  anneal  ╲___
 |   ╱            ╲____      ╱
 ├──┊───────────────┊───────┊───────────────┊──→ Epochs
 0  22              150     157             300
    ┊ Cycle 0 (15%)         ┊ Cycle 1 (5%)
    ┊ save/restore          ┊ save/restore
    ┊ between sessions        ┊ between sessions
```

- **OneCycleLR + Warm Restarts** (Smith & Topin 2018 + Loshchilov & Hutter 2017): each 150-epoch cycle runs warmup → peak → cosine decay
- **Auto-restart**: upon completing a cycle, another starts with max_lr reduced ×0.7 and short warmup (5%), allowing progressive refinement
- **Save/restore within cycle**: the scheduler state is saved in the checkpoint. When resuming mid-cycle, it continues exactly where it left off
- **Fast-forward for legacy**: if the checkpoint has no scheduler_state, the correct advancement is simulated
- **`force_new_cycle`**: manual flag to force a fresh OneCycleLR cycle — resets the scheduler, early stopping thresholds, and suspends the collapse detector during the first new cycle (since metrics will start low during warmup)

### Dual Early Stopping (Loss + DoseAcc3%)

Standard early stopping monitors only the validation loss. But our composite loss is the weighted sum of 7 components — when some improve and others worsen, the total can stagnate even though the model is improving *clinically*. Our dual criterion solves this:

**Improvement is detected if EITHER condition holds:**
1. `val_loss < best_loss - min_delta` — the composite loss improved
2. `DoseAcc3% > best_dose_acc + 0.3pp` — the clinical accuracy metric improved

> **Real example**: val_loss went from 0.00508 → 0.00514 (worsened), while DoseAcc3% went from 71.9% → 73.6% (improved). With loss-only early stopping, training would have stopped. With dual criteria, it correctly continued.

The `min_epochs` parameter (default: 25) prevents early stopping from firing during warmup or collapse recovery.

### Collapse Detection and Auto-Recovery

If DoseAcc3% drops >15 percentage points below the best recorded value, the model is considered to have "collapsed" (catastrophic forgetting, numerical instability, or bad learning rate). The system:
1. Restores weights from the best checkpoint
2. Resets the early stopping counter and best DoseAcc3%
3. Logs the collapse event (max 3 per session to avoid infinite loops)
4. The detector is **suspended** during the first cycle after a `force_new_cycle` (metrics will naturally start low during warmup)

### Stochastic Weight Averaging (SWA) — Izmailov et al. (2018)

After `swa_start_epoch`, the network weights are averaged over subsequent epochs (every `swa_update_every` epochs). SWA typically improves generalization by 0.5–1% and produces flatter minima. The SWA model is:
- Accumulated alongside the main model (no extra forward passes)
- Has its BatchNorm statistics updated at the end of training using the full training set
- Saved as a separate checkpoint (`swa_transunet_film.pth`)
- Compatible with checkpoint save/restore (SWA state dict and counter are persisted)

### Smooth Loss Transition

When adding new loss components (e.g., L1) mid-training, the `PhysicsInformedLoss` supports a smooth transition that ramps from 0 → full weight over configurable epochs, preventing sudden loss landscape changes from destabilizing a pre-trained model (see Part 4, §4.4).

### DoseAcc — Dose-Zone Clinical Metric

During validation, **DoseAcc** is computed: the percentage of voxels in the **clinical dose zone** (dose > 10% of $D_{max}$) where the prediction agrees with the target within X% (default: 3% and 5%). This mimics the gamma-index concept but is faster to compute during training. Additionally, the relative $D_{max}$ error and mean error in the dose zone are tracked.

### Mixed Precision (AMP)
If a GPU with FP16 support is available, we use Automatic Mixed Precision:
- Forward pass in FP16 (faster, less memory)
- Loss and backward in FP32 (numerical stability)
- ~2× speedup without quality loss

### Gradient Clipping
We clip the gradient norm to 1.0 to prevent gradient explosion (common in Transformers).

In [ ]:
"""
    CELL 12: Training and validation functions
===================================================
The training loop with all optimizations.
Incluye:
  - CosineWarmupScheduler → OneCycleLR: per-batch LR schedule with super-convergence
  - EarlyStopping: overfitting prevention (with min_epochs)
  - Checkpoint: save/load complete state
  - Physical metrics: dose accuracy (dose zone), D_max error, MAE
"""

def create_onecycle_scheduler(optimizer, cfg, steps_per_epoch,
                              cycle_epochs=None, max_lr=None, pct_start=None):
    """
    Creates a OneCycleLR for a training cycle.

    Supports warm restarts: upon completing a cycle, a new one is created with
    adjusted parameters (decayed max_lr, reduced warmup).

    Within a cycle, the scheduler state is saved in the checkpoint and
    restored on resumption (mid-cycle continuation without redundant warmup).

    Args:
        optimizer: AdamW
        cfg: global Config
        steps_per_epoch: batches per epoch
        cycle_epochs: epochs of this cycle (default: cfg.onecycle_cycle_epochs)
        max_lr: maximum LR of the cycle (default: cfg.learning_rate)
        pct_start: warmup fraction (default: cfg.onecycle_pct_start)
    """
    cycle_epochs = cycle_epochs or cfg.onecycle_cycle_epochs
    max_lr = max_lr or cfg.learning_rate
    pct_start = pct_start or cfg.onecycle_pct_start
    total_steps = cycle_epochs * steps_per_epoch

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        total_steps=total_steps,
        pct_start=pct_start,
        div_factor=cfg.onecycle_div_factor,
        final_div_factor=cfg.onecycle_final_div_factor,
        anneal_strategy='cos',
        three_phase=False,
    )

    warmup_epochs = int(cycle_epochs * pct_start)
    print(f"  📈 OneCycleLR (cycle of {cycle_epochs} ep):")
    print(f"     max_lr={max_lr:.2e} | steps={total_steps}")
    print(f"     Warmup: {warmup_epochs} ep → peak → "
          f"cosine decay {cycle_epochs - warmup_epochs} ep")

    return scheduler




class EarlyStopping:
    """
    Stops training if there is no improvement in loss NOR in clinical metrics.

    Improvement criteria (OR — it is sufficient for one to be met):
      1. val_loss < best_loss - min_delta  (improvement in the composite loss)
      2. DoseAcc3% > best_dose_acc + dose_acc_threshold pp  (clinical improvement)

    Why a dual criterion?
      val_loss is the weighted sum of 7 components (MSE, gradient, SSIM,
      DVH, falloff, cosine, L1). When some improve and others worsen, the
      total sum can stagnate or even increase, but the clinical metrics
      (DoseAcc3%, D_max_err) DO improve because they reflect the REAL quality of
      the dose distribution in the clinical zone (>10% D_max).

      Real example: val_loss goes from 0.00508→0.00514 while DoseAcc3%
      goes from 71.9%→73.6% — the model improves clinically but the
      composite loss does not reflect it due to compensation between components.

    dose_acc_threshold:
      0.3pp captures real incremental improvements without saving from noise.
      Before it was 1.0pp, too high for the refinement phase where
      improvements are 0.3-0.5pp per session.

    Functionality:
      - Saves the best model (best checkpoint)
      - Compares with best recorded loss AND best DoseAcc3%
      - If no improvement in ANY criterion in 'patience' epochs → stop
      - min_epochs: does not activate stop before this epoch
        (avoids stopping prematurely during warmup / collapse phase)
    """

    def __init__(self, patience: int, min_delta: float = 5e-7,
                 save_path: str = 'best_model.pth', min_epochs: int = 25,
                 dose_acc_threshold: float = 0.3):
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.min_epochs = min_epochs
        self.dose_acc_threshold = dose_acc_threshold
        self.counter = 0
        self.best_loss = float('inf')
        self.best_dose_acc = 0.0   # Best DoseAcc3% recorded
        self.should_stop = False

    def __call__(self, val_loss, model, epoch=0, physics_metrics=None):
        improved = False

        # Criterion 1: improvement in val_loss (classic criterion)
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            improved = True

        # Criterion 2: clinical improvement — DoseAcc3% rises ≥threshold pp
        # Captures real improvements that val_loss does not detect because
        # its 7 components compensate each other.
        if physics_metrics is not None:
            dose_acc = physics_metrics.get('dose_acc_3pct', 0.0)
            if dose_acc > self.best_dose_acc + self.dose_acc_threshold:
                self.best_dose_acc = dose_acc
                improved = True

        if improved:
            self.counter = 0
            # Save best model
            torch.save({
                'model_state_dict': model.state_dict(),
                'val_loss': val_loss,
            }, self.save_path)
        else:
            self.counter += 1
            # Only activate early stopping after min_epochs
            if self.counter >= self.patience and epoch >= self.min_epochs:
                self.should_stop = True

    def load_best(self, model):
        """Restores the weights from the best checkpoint."""
        checkpoint = torch.load(self.save_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(checkpoint['model_state_dict'])
        return checkpoint['val_loss']


def save_full_checkpoint(path, model, optimizer, scaler, epoch,
                         best_loss, history, early_stop_counter,
                         scheduler=None, cycle_len=None, steps_per_epoch=None,
                         swa_model=None, swa_n=0):
    """
    Saves the COMPLETE state for resuming training.

    Includes everything needed to continue exactly where it left off:
    - Model weights
    - Optimizer state (Adam moments)
    - GradScaler state (FP16 scale factor)
    - OneCycleLR state (exact position within the current cycle)
    - cycle_len and steps_per_epoch: to detect incompatibility on resumption
    - SWA model state (averaged weights)
    - Current epoch, best loss, complete history
    """
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'epoch': epoch,
        'best_loss': best_loss,
        'history': history,
        'early_stop_counter': early_stop_counter,
        # ── Scheduler metadata to detect incompatibility ──
        'cycle_len': cycle_len,
        'steps_per_epoch': steps_per_epoch,
        # ── SWA state ──
        'swa_state_dict': swa_model.module.state_dict() if swa_model is not None else None,
        'swa_n': swa_n,
    }
    torch.save(checkpoint, path)
    print(f"  💾 Resumption checkpoint: {path} (epoch {epoch})")


def load_full_checkpoint(path, model, optimizer, scaler, device):
    """
    Loads complete state for resuming training.

    Includes scheduler_state to restore the OneCycleLR
    to the exact point within the current cycle.

    Returns:
        start_epoch, best_loss, history, early_stop_counter, scheduler_state,
        saved_cycle_len, saved_steps_per_epoch, swa_state_dict, swa_n
    """
    print(f"  📂 Loading checkpoint: {path}")
    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if scaler is not None and checkpoint['scaler_state_dict'] is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    epoch = checkpoint['epoch'] + 1  # Continue from the next
    best_loss = checkpoint['best_loss']
    history = checkpoint['history']
    es_counter = checkpoint.get('early_stop_counter', 0)
    scheduler_state = checkpoint.get('scheduler_state_dict', None)

    # ── Scheduler metadata (None if old checkpoint) ──
    saved_cycle_len = checkpoint.get('cycle_len', None)
    saved_steps_per_epoch = checkpoint.get('steps_per_epoch', None)

    # ── SWA state (None if old checkpoint without SWA) ──
    swa_state_dict = checkpoint.get('swa_state_dict', None)
    swa_n_loaded = checkpoint.get('swa_n', 0)

    print(f"     Resuming from epoch {epoch}")
    print(f"     Previous best val_loss: {best_loss:.6f}")
    print(f"     History: {len(history['train_loss'])} epochs recorded")
    if saved_cycle_len is not None:
        print(f"     Checkpoint saved with cycle_len={saved_cycle_len}, "
              f"steps_per_epoch={saved_steps_per_epoch}")
    else:
        print(f"     ⚠️ Old checkpoint: no saved cycle_len/steps_per_epoch")
    if scheduler_state is not None:
        step = scheduler_state.get('_step_count', '?')
        print(f"     OneCycleLR: restoring state (step {step})")
    else:
        print(f"     OneCycleLR: no previous state → warmup from the start")
    if swa_state_dict is not None:
        print(f"     SWA: restoring averaged weights (n={swa_n_loaded})")
    else:
        print(f"     SWA: no previous state (will accumulate from swa_start_epoch)")

    return (epoch, best_loss, history, es_counter, scheduler_state,
            saved_cycle_len, saved_steps_per_epoch, swa_state_dict, swa_n_loaded)




def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None,
                    scheduler=None):
    """
    Trains one complete epoch.

    Args:
        model: TransUNetFiLM3D
        loader: DataLoader de train
        criterion: PhysicsInformedLoss
        optimizer: AdamW
        device: 'cuda' o 'cpu'
        scaler: GradScaler for mixed precision (None if CPU)
        scheduler: OneCycleLR (se actualiza per-batch). None si no se usa.

    Returns:
        avg_loss: average loss for the epoch
        loss_components: dictionary of mean components
    """
    model.train()
    total_loss = 0.0
    component_sums = {}
    n_batches = 0

    for batch in loader:
        density = batch['density'].to(device, non_blocking=True)
        params = batch['params'].to(device, non_blocking=True)
        target = batch['dose'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()

        if scaler is not None:
            # Mixed precision (GPU)
            # The loss is computed inside autocast; ssim_loss_3d
            # disables autocast internally for FP32 stability
            with torch.amp.autocast('cuda'):
                pred, aux = model(density, params)
                loss, loss_dict = criterion(pred, target, aux)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                try:
                    scheduler.step()
                except ValueError:
                    pass  # OneCycleLR cycle completed; will be recreated in the loop
        else:
            # Full precision (CPU)
            pred, aux = model(density, params)
            loss, loss_dict = criterion(pred, target, aux)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip)
            optimizer.step()
            if scheduler is not None:
                try:
                    scheduler.step()
                except ValueError:
                    pass  # OneCycleLR cycle completed; will be recreated in the loop

        total_loss += loss.item()
        for k, v in loss_dict.items():
            component_sums[k] = component_sums.get(k, 0.0) + v
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    avg_components = {k: v / max(n_batches, 1) for k, v in component_sums.items()}
    return avg_loss, avg_components


@torch.no_grad()
def validate(model, loader, criterion, device):
    """
    Evaluates the model on the validation set.
    Computes loss + representative physical metrics.

    IMPORTANT — Metrics in DOSE ZONE:
      DoseAcc and MAE_dose metrics are computed ONLY on voxels
      where target > 10% of D_max for each sample. This excludes the
      ~99.5% of background (dose ≈ 0) that would artificially inflate
      DoseAcc and dilute MAE.

    @torch.no_grad() disables gradient computation,
    ahorrando memoria y acelerando la inferencia.

    Returns:
        avg_loss: loss media
        avg_components: dict of loss components
        physics_metrics: dict with physical metrics
    """
    model.eval()
    total_loss = 0.0
    component_sums = {}
    n_batches = 0
    use_amp = (device.type == 'cuda')

    # Physical metrics accumulators
    total_mae = 0.0              # Global MAE (all voxels, for compatibility)
    total_mae_dose = 0.0         # MAE only in dose zone (>10% D_max)
    total_dmax_err = 0.0         # Relative error at maximum dose point
    total_dose_acc_3pct = 0.0    # % voxels IN DOSE ZONE with error < 3% of max
    total_dose_acc_5pct = 0.0    # % voxels IN DOSE ZONE with error < 5% of max
    n_valid_batches_dose = 0     # Batches with valid dose zone

    for batch in loader:
        density = batch['density'].to(device, non_blocking=True)
        params = batch['params'].to(device, non_blocking=True)
        target = batch['dose'].to(device, non_blocking=True)

        if use_amp:
            with torch.amp.autocast('cuda'):
                pred, aux = model(density, params)
                loss, loss_dict = criterion(pred, target, aux)
        else:
            pred, aux = model(density, params)
            loss, loss_dict = criterion(pred, target, aux)

        total_loss += loss.item()
        for k, v in loss_dict.items():
            component_sums[k] = component_sums.get(k, 0.0) + v

        # ─── Physical metrics (per batch) ───
        # Working in float32 for precision
        p = pred.float()
        t = target.float()

        # Global normalized MAE [0,1] (all voxels, for compatibility)
        total_mae += (p - t).abs().mean().item()

        # D_max error: |max(pred) - max(target)| / max(target)
        pred_max = p.flatten(1).max(dim=1).values  # (B,)
        tgt_max = t.flatten(1).max(dim=1).values   # (B,)
        dmax_err = ((pred_max - tgt_max).abs() / tgt_max.clamp(min=1e-8)).mean().item()
        total_dmax_err += dmax_err

        # ─── Metrics ONLY in significant dose zone ───
        # Mask: voxels where target > 10% of D_max for each sample
        # This excludes ~99.5% of background that inflates DoseAcc and dilutes MAE
        tgt_max_vol = tgt_max.view(-1, 1, 1, 1, 1)  # (B, 1, 1, 1, 1)
        dose_mask = t > 0.10 * tgt_max_vol           # (B, 1, D, H, W) bool

        n_dose_voxels = dose_mask.sum().item()
        if n_dose_voxels > 0:
            # MAE only in dose zone
            abs_err = (p - t).abs()
            mae_dose = (abs_err * dose_mask.float()).sum().item() / n_dose_voxels
            total_mae_dose += mae_dose

            # Dose accuracy: % of voxels IN DOSE ZONE with error < threshold
            pass_3 = ((abs_err < 0.03 * tgt_max_vol) & dose_mask).sum().item()
            pass_5 = ((abs_err < 0.05 * tgt_max_vol) & dose_mask).sum().item()
            dose_acc_3 = pass_3 / n_dose_voxels
            dose_acc_5 = pass_5 / n_dose_voxels
            total_dose_acc_3pct += dose_acc_3
            total_dose_acc_5pct += dose_acc_5
            n_valid_batches_dose += 1

        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    avg_components = {k: v / max(n_batches, 1) for k, v in component_sums.items()}

    # Use n_valid_batches_dose for dose zone metrics
    n_dose = max(n_valid_batches_dose, 1)
    physics_metrics = {
        'mae': total_mae / max(n_batches, 1),
        'mae_dose': total_mae_dose / n_dose,
        'dmax_err_pct': 100.0 * total_dmax_err / max(n_batches, 1),
        'dose_acc_3pct': 100.0 * total_dose_acc_3pct / n_dose,
        'dose_acc_5pct': 100.0 * total_dose_acc_5pct / n_dose,
    }

    return avg_loss, avg_components, physics_metrics

print("✅ Training functions defined:")
print("   • EarlyStopping: dual criterion (val_loss + DoseAcc3%, with min_epochs)")
print("   • save/load_full_checkpoint: complete state for resumption (with cycle_len/steps_per_epoch)")
print("   • OneCycleLR: cycles with automatic warm restarts (save/restore)")
print("   • train_one_epoch(): one complete training step")
print("   • validate(): evaluation + physical metrics (dose zone)")
print("   ⚡ DoseAcc and MAE_dose are computed ONLY on zone with dose >10% D_max")

In [ ]:
"""
CELL 13: MAIN TRAINING LOOP
============================================
All together: optimizer, scheduler, early stopping, logging.

Features:
  - OneCycleLR with WARM RESTARTS: cycles of 200 epochs with decayed max_lr.
    Upon completing a cycle → new cycle with max_lr ×0.85 and short warmup (5%).
    Within the cycle, scheduler state is saved/restored between sessions.
  - force_new_cycle: force a new OneCycleLR cycle (resets scheduler,
    ES thresholds, and suspends collapse detector during the first new cycle)
  - Automatic scheduler incompatibility detection (cycle_len or
    steps_per_epoch changed → automatically force new cycle)
  - Automatic timeout: runs all possible epochs until the Kaggle timeout
  - Resume training from previous checkpoint
  - Auto-save before Kaggle timeout (measured from T_NOTEBOOK_START)
  - Physical metrics each epoch (dose accuracy in dose zone, D_max error, MAE)
  - Early stopping with min_epochs (does not stop before epoch 25)
  - skip_training: if True, skips the loop and loads the best model directly
    → Useful for sessions dedicated to post-training evaluation/optimization
  - Collapse detection: if DoseAcc3% drops >15pp from best, restores weights
    (suspended during the first new cycle to allow time for warmup)
"""

# ═══════════════════════════════════════════════════════════
# SKIP TRAINING?
# ═══════════════════════════════════════════════════════════
# Set to True to load existing checkpoint and skip directly
# to evaluation / optimization cells (Stage 2).
# Typical multi-session workflow:
#   Session 1: skip_training=False → trains ~60 epochs → timeout → saves checkpoint
#   Session 2: skip_training=False → resumes → completes training (or timeout again)
#   Session 3: skip_training=True  → loads best model → runs evaluation + Stage 2
skip_training = False

# ═══════════════════════════════════════════════════════════
# ⚠️ RE-TRAIN MODEL (load weights, restart everything else)
# ═══════════════════════════════════════════════════════════
# Set to True to LOAD model weights from the checkpoint
# but RESTART everything else from epoch 0:
#   - Model weights: ✅ PRESERVED from checkpoint
#   - Optimizer: NEW (Adam moments at zero)
#   - Scheduler: NEW (OneCycleLR from the start of cycle 0)
#   - History: empty
#   - Early stopping: counter to 0
# Useful when you change hyperparameters (LR, cycles, etc.) and want to
# retrain the already-learned model with a new configuration.
force_fresh_training = False  # ← Resume from checkpoint (preserves optimizer/scheduler/history)

# ═══════════════════════════════════════════════════════════
# 🔄 FORCE NEW CYCLE (without resetting epoch or optimizer)
# ═══════════════════════════════════════════════════════════
# Set to True to force a NEW OneCycleLR CYCLE on resumption:
#   - Scheduler: NEW (warmup from the start of the new cycle)
#   - ES thresholds: RESET (best_loss=inf, best_dose_acc=0)
#   - Collapse detector: SUSPENDED during the first new cycle
#   - Optimizer: SE CONSERVA (momentos de Adam intactos)
#   - History: PRESERVED (continuation of training)
# Useful when you changed the dataset (more samples, augmentation),
# cycle_len, or anything that invalidates the saved scheduler state.
# Also activated AUTOMATICALLY if incompatibility is detected
# between the saved scheduler and the current configuration.
force_new_cycle = False  # ← Disabled (reset was already applied in session 12)

# ═══════════════════════════════════════════════════════════
# SETUP
# ═══════════════════════════════════════════════════════════

# Optimizer AdamW
optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
    betas=(0.9, 0.999),  # First and second order moments
)

# Mixed precision (only if GPU available)
use_amp = DEVICE.type == 'cuda'
scaler = torch.amp.GradScaler('cuda') if use_amp else None

# SWA — Stochastic Weight Averaging (Izmailov et al., 2018)
# Accumulates weight average → flatter solutions → better generalization.
# Activated after swa_start_epoch; checkpoint compatible.
swa_model = None
swa_n = 0  # Counter of accumulated SWA updates
if CFG.swa_enabled:
    swa_model = torch.optim.swa_utils.AveragedModel(model)
    print(f"  🔄 SWA enabled: start_epoch={CFG.swa_start_epoch}, "
          f"update_every={CFG.swa_update_every} epochs")

# Save paths
best_model_path = os.path.join(BASE_OUTPUT, 'best_transunet_film.pth')
checkpoint_path = os.path.join(BASE_OUTPUT, CFG.checkpoint_name)
swa_model_path = os.path.join(BASE_OUTPUT, 'swa_transunet_film.pth')

# ═══════════════════════════════════════════════════════════
# LOAD CHECKPOINT (if it exists and not forcing fresh)
# ═══════════════════════════════════════════════════════════

# Defaults (overwritten if full checkpoint is loaded)
start_epoch = 0
best_loss_so_far = float('inf')
history = {
    'train_loss': [], 'val_loss': [], 'lr': [],
    'epoch_time': [], 'train_components': [],
    'val_components': [], 'physics_metrics': [],
    'n_restarts': 0,
}
es_counter_loaded = 0
_sched_state = None
_saved_cycle_len = None
_saved_steps_per_epoch = None
session_number = 0

if force_fresh_training:
    # Load ONLY model weights from checkpoint (not optimizer/scheduler)
    if CFG.resume_from is not None:
        checkpoint = torch.load(CFG.resume_from, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        prev_epoch = checkpoint.get('epoch', '?')
        prev_loss = checkpoint.get('best_loss', '?')
        print(f"  🔄 force_fresh_training=True → Model weights LOADED from checkpoint")
        print(f"     Model trained until epoch {prev_epoch} (best_loss={prev_loss})")
        print(f"     Optimizer, scheduler, historial → REINICIADOS a cero")
        del checkpoint
    else:
        print("  ⚠️ force_fresh_training=True but no checkpoint → random weights")
    print("     Data split (train/val/test) is NOT modified")
elif CFG.resume_from is not None:
    start_epoch, best_loss_so_far, history, es_counter_loaded, _sched_state, \
        _saved_cycle_len, _saved_steps_per_epoch, _swa_state_dict, _swa_n_loaded = \
        load_full_checkpoint(CFG.resume_from, model, optimizer, scaler, DEVICE)
    # ── Restaurar SWA state si existe ──
    if swa_model is not None and _swa_state_dict is not None:
        swa_model.module.load_state_dict(_swa_state_dict)
        swa_n = _swa_n_loaded
        print(f"  🔄 SWA restored: {swa_n} accumulated updates")
    # Ensure history has the physical metrics key
    if 'physics_metrics' not in history:
        history['physics_metrics'] = [{}] * len(history['train_loss'])
    # Calculate session number from previous resumptions
    session_number = history.get('n_restarts', 0) + 1
    # Activate smooth transition (persist across sessions via history)
    _ts = history.get('transition_start_epoch', None)
    if _ts is not None:
        # Transition already started in previous session → continue
        criterion.start_transition(_ts)
        criterion.set_epoch(start_epoch)  # So it calculates the correct factor
        t = criterion._get_transition_factor()
        print(f"  🔄 Transition RESUMED from epoch {_ts} (now {t*100:.0f}% completed)")
    else:
        # First time → start transition and save in history
        criterion.start_transition(start_epoch)
        history['transition_start_epoch'] = start_epoch

if skip_training:
    # ═══════════════════════════════════════════════════════════
    # SKIP MODE: load best model and skip to post-training
    # ═══════════════════════════════════════════════════════════
    print("=" * 70)
    print("  ⏭️  SKIP TRAINING — Loading best existing model")
    print("=" * 70)
    timeout_triggered = False
    if os.path.exists(best_model_path):
        state = torch.load(best_model_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(state)
        print(f"  ✅ Model loaded from: {best_model_path}")
    elif CFG.resume_from is not None:
        print(f"  ✅ Model loaded from checkpoint: {CFG.resume_from}")
    else:
        print(f"  ⚠️  No trained model found. Subsequent cells may fail.")
    print(f"  📊 Epochs in history: {len(history['train_loss'])}")
    if history['physics_metrics'] and history['physics_metrics'][-1]:
        lp = history['physics_metrics'][-1]
        print(f"  📊 Last DoseAcc3%: {lp.get('dose_acc_3pct', 0):.1f}%")
    print("=" * 70)

else:
    # ═══════════════════════════════════════════════════════════
    # NORMAL MODE: TRAINING
    # ═══════════════════════════════════════════════════════════

    # ─── Calculate available epochs in this session ───
    steps_per_epoch = len(train_loader)
    setup_elapsed = time.time() - T_NOTEBOOK_START
    time_for_training = CFG.kaggle_time_limit - setup_elapsed

    # Estimate time per epoch (use history if available, otherwise ~2 min/epoch as default)
    if history['epoch_time']:
        epoch_time_estimate = np.median(history['epoch_time'][-5:])  # Median of last 5
    else:
        # Conservative estimate: ~2s per batch (includes forward, backward, metrics)
        epoch_time_estimate = steps_per_epoch * 1.5 + 30  # +30s validation overhead

    # Epochs that fit in available time (with margin of 1 epoch for save)
    epochs_this_session = max(3, int(time_for_training / epoch_time_estimate) - 1)
    # Do not exceed the global cap
    epochs_this_session = min(epochs_this_session, CFG.num_epochs - start_epoch)
    # end_epoch = global cap (the timeout safety net is the real limit)
    end_epoch = CFG.num_epochs

    # ─── OneCycleLR with WARM RESTARTS ───
    # Each cycle = onecycle_cycle_epochs epochs. Upon completing → new cycle.
    cycle_len = CFG.onecycle_cycle_epochs
    cycle_number = start_epoch // cycle_len
    pos_in_cycle = start_epoch % cycle_len

    # ─── Automatic scheduler incompatibility detection ───
    # If cycle_len or steps_per_epoch changed since the last checkpoint,
    # the saved scheduler state is incompatible → force new cycle.
    _sched_incompatible = False
    if _sched_state is not None and not force_new_cycle:
        if _saved_cycle_len is not None and _saved_steps_per_epoch is not None:
            saved_total = _saved_cycle_len * _saved_steps_per_epoch
            current_total = cycle_len * steps_per_epoch
            if saved_total != current_total:
                _sched_incompatible = True
                print(f"\n  ⚠️  INCOMPATIBILIDAD DETECTADA en scheduler:")
                print(f"      Checkpoint: cycle_len={_saved_cycle_len} × "
                      f"steps_per_epoch={_saved_steps_per_epoch} = {saved_total} total_steps")
                print(f"      Current:    cycle_len={cycle_len} × "
                      f"steps_per_epoch={steps_per_epoch} = {current_total} total_steps")
                print(f"      → Automatically forcing new cycle")
        else:
            # Old checkpoint without metadata → assume incompatible
            _sched_incompatible = True
            print(f"\n  ⚠️  Old checkpoint without cycle_len/steps_per_epoch → "
                  f"forcing new cycle for safety")

    # ─── Apply force_new_cycle (manual or auto-detected) ───
    _starting_new_cycle = force_new_cycle or _sched_incompatible
    if _starting_new_cycle and start_epoch > 0:
        # Calculate logical cycle number based on current epoch
        cycle_number = start_epoch // cycle_len
        pos_in_cycle = 0  # Start from the beginning of the new cycle
        _sched_state = None  # Descartar scheduler state incompatible

        print(f"\n  🔄 NEW CYCLE FORCED at epoch {start_epoch}:")
        print(f"     Logical cycle #{cycle_number + 1} (cycle_len={cycle_len})")
        print(f"     Scheduler: NEW (warmup from start)")
        print(f"     ES thresholds: RESET")
        print(f"     Collapse detector: SUSPENDED {cycle_len} epochs")

    # LR and warmup of the current cycle
    effective_max_lr = CFG.learning_rate * (CFG.lr_restart_decay ** cycle_number)
    effective_pct = CFG.onecycle_pct_start if cycle_number == 0 else CFG.restart_pct_start

    scheduler = create_onecycle_scheduler(
        optimizer, CFG, steps_per_epoch,
        cycle_epochs=cycle_len, max_lr=effective_max_lr, pct_start=effective_pct
    )
    current_cycle_end = (cycle_number + 1) * cycle_len
    # If the new cycle starts mid-way, adjust the cycle end
    if _starting_new_cycle and start_epoch > 0:
        current_cycle_end = start_epoch + cycle_len

    # Restore scheduler position within the cycle
    if pos_in_cycle > 0:
        if _sched_state is not None:
            try:
                scheduler.load_state_dict(_sched_state)
                print(f"  📈 OneCycleLR restored (cycle #{cycle_number+1}, pos {pos_in_cycle})")
            except Exception:
                print(f"  ⚠️ Scheduler state incompatible → fast-forward {pos_in_cycle} ep")
                scheduler = create_onecycle_scheduler(
                    optimizer, CFG, steps_per_epoch,
                    cycle_epochs=cycle_len, max_lr=effective_max_lr, pct_start=effective_pct)
                for _ in range(pos_in_cycle * steps_per_epoch):
                    try:
                        scheduler.step()
                    except ValueError:
                        break  # Scheduler already reached its limit
        else:
            print(f"  ⏩ Fast-forward scheduler: {pos_in_cycle} ep in cycle #{cycle_number+1}")
            for _ in range(pos_in_cycle * steps_per_epoch):
                try:
                    scheduler.step()
                except ValueError:
                    break  # Scheduler already reached its limit
    elif cycle_number > 0 and not _starting_new_cycle:
        print(f"  🔄 Warm restart: cycle #{cycle_number+1}, max_lr={effective_max_lr:.2e}")

    # Early stopping (with min_epochs to avoid stopping prematurely)
    # With force_fresh_training: optimizer starts from zero and OneCycleLR
    # needs at least 1 complete cycle to raise and lower the LR.
    # min_epochs must cover at least 1 cycle to not kill training
    # during the warmup phase (where the model temporarily worsens).
    es_min_epochs = max(25, cycle_len) if force_fresh_training else 25
    early_stop = EarlyStopping(
        patience=CFG.patience,
        save_path=best_model_path,
        min_epochs=es_min_epochs,
        dose_acc_threshold=CFG.es_dose_acc_threshold,
    )

    # ─── Reset ES thresholds if new cycle forced ───
    if _starting_new_cycle and start_epoch > 0:
        # Old thresholds (from another dataset/config) are unreachable
        # → reset so the model can save improvements from the start
        early_stop.best_loss = float('inf')
        early_stop.best_dose_acc = 0.0
        early_stop.counter = 0
        best_loss_so_far = float('inf')
        print(f"  ✅ ES thresholds reset: best_loss=inf, best_dose_acc=0.0")
    else:
        early_stop.best_loss = best_loss_so_far
        early_stop.counter = es_counter_loaded
        # Restore best DoseAcc3% from history (for ES dual criterion)
        early_stop.best_dose_acc = history.get('best_dose_acc', 0.0)

    # ─── Collapse detector: suspend during first new cycle ───
    # When we force a new cycle (dataset changed, etc.), the metrics
    # will start low during warmup. The collapse detector must NOT trigger
    # comparing against old thresholds (which we already reset above).
    # We suspend it for cycle_len epochs to allow time for complete warmup.
    if _starting_new_cycle and start_epoch > 0:
        _collapse_disabled_until = start_epoch + cycle_len
        print(f"  ✅ Collapse detector suspended until epoch {_collapse_disabled_until}")
    else:
        _collapse_disabled_until = 0  # Not suspended (active from the start)

    print("=" * 70)
    print("  SURROGATE TRAINING — 3D TransUNet + FiLM")
    print("=" * 70)
    print(f"  Session:           #{session_number + 1}")
    print(f"  Epochs:            {start_epoch} → {CFG.num_epochs} "
          f"(~{epochs_this_session} estimated, cycle #{cycle_number+1})")
    current_lr = optimizer.param_groups[0]['lr']
    print(f"  OneCycleLR:        cycle #{cycle_number+1} ({cycle_len} ep/cycle), "
          f"max_lr={effective_max_lr:.2e}, div={CFG.onecycle_div_factor}")
    if _starting_new_cycle and start_epoch > 0:
        print(f"  🔄 New cycle:      FORCED ({'manual' if force_new_cycle else 'auto-detected'})")
        print(f"     Cycle end:      epoch {current_cycle_end}")
    print(f"  ES criterio dual:  val_loss < {early_stop.best_loss:.6f} OR "
          f"DoseAcc3% > {early_stop.best_dose_acc:.1f}%+{CFG.es_dose_acc_threshold}pp")
    print(f"  Mixed precision:   {'Yes (FP16)' if use_amp else 'No (FP32)'}")
    print(f"  Device:            {DEVICE}")
    print(f"  Batch size:        {CFG.batch_size}")
    print(f"  Train batches:     {len(train_loader)}")
    print(f"  Val batches:       {len(val_loader)}")
    print(f"  Resuming:          {'Yes (epoch ' + str(start_epoch) + ')' if start_epoch > 0 else 'No (from scratch)'}")
    print(f"  Setup took:        {setup_elapsed:.0f}s ({setup_elapsed/60:.1f} min)")
    print(f"  Tiempo restante:   {time_for_training:.0f}s ({time_for_training/3600:.1f}h)")
    print(f"  Est. per epoch:    {epoch_time_estimate:.0f}s ({epoch_time_estimate/60:.1f} min)")
    print(f"  Timeout (total):   {CFG.kaggle_time_limit}s from notebook start")
    if swa_model is not None:
        swa_status = f"from epoch {CFG.swa_start_epoch}, every {CFG.swa_update_every} ep"
        if swa_n > 0:
            swa_status += f" ({swa_n} previous updates)"
        print(f"  SWA:               {swa_status}")
    print("=" * 70)

    t_global_start = time.time()
    timeout_triggered = False
    _n_collapses = 0  # Collapse counter detected in this session

    for epoch in range(start_epoch, end_epoch):
        t_epoch = time.time()
        criterion.set_epoch(epoch)

        # ─── Check timeout BEFORE starting the epoch ───
        # Safety net: even though we calculated epochs_this_session, we verify
        # in case the initial estimate was optimistic (epoch_time variability).
        total_elapsed = time.time() - T_NOTEBOOK_START
        epoch_estimate = history['epoch_time'][-1] if history['epoch_time'] else epoch_time_estimate
        if total_elapsed + epoch_estimate + 120 > CFG.kaggle_time_limit:
            training_elapsed = time.time() - t_global_start
            print(f"\n  ⏰ TIMEOUT: {total_elapsed:.0f}s total notebook time "
                  f"({training_elapsed:.0f}s de training), "
                  f"would need ~{epoch_estimate:.0f}s more for 1 epoch.")
            print(f"     Guardando checkpoint y saliendo...")
            history['n_restarts'] = session_number
            save_full_checkpoint(
                checkpoint_path, model, optimizer, scaler,
                epoch - 1, early_stop.best_loss, history, early_stop.counter,
                scheduler=scheduler,
                cycle_len=cycle_len, steps_per_epoch=steps_per_epoch,
                swa_model=swa_model, swa_n=swa_n
            )
            timeout_triggered = True
            break

        # ─── Check OneCycleLR cycle transition ───
        if epoch == current_cycle_end:
            cycle_number = epoch // cycle_len
            effective_max_lr = CFG.learning_rate * (CFG.lr_restart_decay ** cycle_number)
            effective_pct = CFG.restart_pct_start
            scheduler = create_onecycle_scheduler(
                optimizer, CFG, steps_per_epoch,
                cycle_epochs=cycle_len, max_lr=effective_max_lr, pct_start=effective_pct
            )
            current_cycle_end = (cycle_number + 1) * cycle_len
            print(f"\n  🔄 Warm restart: cycle #{cycle_number+1}, "
                  f"max_lr={effective_max_lr:.2e}, "
                  f"next restart at epoch {current_cycle_end}")

        # Train (scheduler updates per-batch inside train_one_epoch)
        train_loss, train_comp = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE, scaler,
            scheduler=scheduler
        )

        # Current LR (after the epoch)
        current_lr = optimizer.param_groups[0]['lr']

        # Validate (includes physical metrics — DoseAcc in dose zone)
        val_loss, val_comp, phys_metrics = validate(model, val_loader, criterion, DEVICE)

        # Tiempo
        epoch_time = time.time() - t_epoch

        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(current_lr)
        history['epoch_time'].append(epoch_time)
        history['train_components'].append(train_comp)
        history['val_components'].append(val_comp)
        history['physics_metrics'].append(phys_metrics)
        # Early stopping (dual: val_loss + DoseAcc3%)
        early_stop(val_loss, model, epoch=epoch, physics_metrics=phys_metrics)
        history['best_dose_acc'] = early_stop.best_dose_acc
        if hasattr(criterion, '_transition_start') and criterion._transition_start is not None:
            history['transition_start_epoch'] = criterion._transition_start

        # ─── SWA: accumulate averaged weights ───
        if (swa_model is not None
                and epoch >= CFG.swa_start_epoch
                and (epoch - CFG.swa_start_epoch) % CFG.swa_update_every == 0):
            swa_model.update_parameters(model)
            swa_n += 1

        # ─── Collapse detection ───
        # If DoseAcc3% drops >15pp from the best recorded, the model
        # has diverged (numerical instability, high LR + transition, etc.)
        # → Restore best model weights to recover.
        # Limit: max 3 restorations per session to avoid infinite loop.
        # SUSPENDED during the first new cycle (force_new_cycle) to
        # allow time for warmup without triggering false alarms.
        current_da = phys_metrics.get('dose_acc_3pct', 0.0)
        if (current_da < early_stop.best_dose_acc - 15.0
                and epoch > start_epoch + 5
                and epoch >= _collapse_disabled_until
                and _n_collapses < 3):
            _n_collapses += 1
            print(f"\n  🚨 COLLAPSE DETECTED (#{_n_collapses}): "
                  f"DoseAcc3% {current_da:.1f}% vs best {early_stop.best_dose_acc:.1f}% "
                  f"(drop of {early_stop.best_dose_acc - current_da:.1f}pp)")
            if os.path.exists(best_model_path):
                state = torch.load(best_model_path, map_location=DEVICE, weights_only=True)
                model.load_state_dict(state['model_state_dict'])
                print(f"     → Weights restored from best model (best epoch)")
                print(f"     → Optimizer/scheduler keep their current state")
                print(f"     → Training continues from the next epoch")
            else:
                print(f"     → No best_model to restore, continuing...")

        # ─── Auto-reset if loss scale changed (new loss, new data, etc.) ───
        # If on resumption the current val_loss is >2x the saved best_loss,
        # the checkpoint comes from another configuration → reset for fair comparisons.
        # (Threshold reduced from 5x to 2x to detect subtler changes like
        # duplicated dataset or new augmentation)
        if epoch == start_epoch and start_epoch > 0 and early_stop.best_loss > 0:
            ratio = val_loss / early_stop.best_loss
            if ratio > 2.0:
                old_best = early_stop.best_loss
                old_da = early_stop.best_dose_acc
                early_stop.best_loss = val_loss
                early_stop.best_dose_acc = phys_metrics.get('dose_acc_3pct', 0.0)
                early_stop.counter = 0
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'val_loss': val_loss,
                }, early_stop.save_path)
                print(f"\n  ⚠️  LOSS SCALE CHANGED: previous best_loss={old_best:.6f} "
                      f"vs current val_loss={val_loss:.6f} (ratio {ratio:.1f}x)")
                print(f"      → best_loss reset to {val_loss:.6f}")
                print(f"      → best_DoseAcc reset to {early_stop.best_dose_acc:.1f}% "
                      f"(era {old_da:.1f}%)")
                print(f"      → Current model saved as new baseline\n")
                history['best_dose_acc'] = early_stop.best_dose_acc

        # ─── Log: frecuencia adaptativa ───
        # Always during transition (to detect problems),
        # every 5 epochs, first 5, on improvement, or on stopping.
        improved_now = (early_stop.counter == 0)
        is_transition = (hasattr(criterion, '_transition_start')
                        and criterion._transition_start is not None
                        and criterion._get_transition_factor() < 1.0)
        if (epoch % 5 == 0 or epoch < (start_epoch + 5)
                or improved_now or early_stop.should_stop or is_transition):
            tag = "🏆" if improved_now else "  "
            print(f"{tag}Epoch {epoch:>4d}/{CFG.num_epochs} │ "
                  f"Train: {train_loss:.5f} │ Val: {val_loss:.5f} │ "
                  f"LR: {current_lr:.2e} │ {epoch_time:.1f}s │ "
                  f"ES: {early_stop.counter}/{CFG.patience}")
            # Physical metrics (dose zone)
            mae_dose_str = f"{phys_metrics.get('mae_dose', 0):.5f}"
            print(f"         ↳ DoseAcc3%: {phys_metrics['dose_acc_3pct']:.1f}% │ "
                  f"DoseAcc5%: {phys_metrics['dose_acc_5pct']:.1f}% │ "
                  f"D_max err: {phys_metrics['dmax_err_pct']:.2f}% │ "
                  f"MAE_dose: {mae_dose_str} │ MAE_global: {phys_metrics['mae']:.5f}")
            # Log of transition state
            if hasattr(criterion, '_transition_start') and criterion._transition_start is not None:
                t_factor = criterion._get_transition_factor()
                if t_factor < 1.0:
                    grad_f = 3.0 - 2.0 * t_factor
                    l1_f = t_factor
                    print(f"         ↳ Transition: {t_factor*100:.0f}% | "
                          f"grad_factor: {grad_f:.2f}x | "
                          f"L1_weight: {criterion.w_l1 * l1_f:.3f}/{criterion.w_l1:.3f}")

        # ─── Periodic checkpoint (every 20 epochs) ───
        if (epoch + 1) % 20 == 0:
            history['n_restarts'] = session_number
            save_full_checkpoint(
                checkpoint_path, model, optimizer, scaler,
                epoch, early_stop.best_loss, history, early_stop.counter,
                cycle_len=cycle_len, steps_per_epoch=steps_per_epoch,
                swa_model=swa_model, swa_n=swa_n
            )

            # During transition: force best model update
            # because loss changes and the classic criterion doesn't detect improvements
            if CFG.force_best_during_transition:
                t_now = criterion._get_transition_factor() if hasattr(criterion, '_transition_start') and criterion._transition_start is not None else 1.0
                if t_now < 1.0:
                    torch.save({'model_state_dict': model.state_dict(), 'val_loss': val_loss}, best_model_path)
                    early_stop.best_loss = val_loss
                    early_stop.best_dose_acc = phys_metrics.get('dose_acc_3pct', 0.0)
                    early_stop.counter = 0
                    print(f"  🔄 Transition active ({t_now*100:.0f}%): best_model updated | "
                          f"best_loss→{val_loss:.6f} | best_DoseAcc→{early_stop.best_dose_acc:.1f}%")
        if early_stop.should_stop:
            print(f"\n  🛑 Early stopping at epoch {epoch} (no improvement in {CFG.patience} epochs)")
            break

    total_time = time.time() - t_global_start
    n_executed = len(history['train_loss']) - start_epoch  # new epochs in this session
    last_epoch = (start_epoch + n_executed - 1) if n_executed > 0 else start_epoch - 1

    # ─── Save final checkpoint (if training occurred) ───
    if not timeout_triggered and n_executed > 0:
        history['n_restarts'] = session_number
        save_full_checkpoint(
            checkpoint_path, model, optimizer, scaler,
            last_epoch, early_stop.best_loss, history, early_stop.counter,
            cycle_len=cycle_len, steps_per_epoch=steps_per_epoch,
            swa_model=swa_model, swa_n=swa_n
        )

    # Restore best model
    if os.path.exists(best_model_path):
        best_val = early_stop.load_best(model)
    else:
        best_val = early_stop.best_loss

    # ─── SWA: update BatchNorm and save averaged model ───
    if swa_model is not None and swa_n > 0:
        time_remaining_swa = CFG.kaggle_time_limit - (time.time() - T_NOTEBOOK_START)
        if time_remaining_swa > 120:  # Only if >2min margin
            print(f"\n  🔄 SWA: updating BatchNorm ({swa_n} accumulated updates)...")
            try:
                torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)
                torch.save({
                    'model_state_dict': swa_model.module.state_dict(),
                    'swa_n': swa_n,
                }, swa_model_path)
                print(f"  ✅ SWA model saved at: {swa_model_path}")
            except Exception as e:
                print(f"  ⚠️  Error updating SWA BN: {e}")
                print(f"     Saving SWA without BN update...")
                torch.save({
                    'model_state_dict': swa_model.module.state_dict(),
                    'swa_n': swa_n,
                }, swa_model_path)
        else:
            print(f"\n  ⏰ SWA: no time for BN update. Saving weights without updating BN...")
            torch.save({
                'model_state_dict': swa_model.module.state_dict(),
                'swa_n': swa_n,
            }, swa_model_path)
    elif swa_model is not None:
        print(f"\n  ℹ️  SWA: no updates (epoch < {CFG.swa_start_epoch})")

    print(f"\n{'='*70}")
    print(f"  TRAINING COMPLETED")
    print(f"  Best val loss:      {best_val:.6f}")
    print(f"  Epochs executed:    {n_executed} "
          f"(cumulative total: {len(history['train_loss'])})")
    print(f"  Total time:         {total_time:.0f}s ({total_time/60:.1f} min)")
    print(f"  Tiempo notebook:    {time.time() - T_NOTEBOOK_START:.0f}s")
    if _n_collapses > 0:
        print(f"  ⚠️  Collapses detected and restored: {_n_collapses}")
    if timeout_triggered:
        print(f"  ⏰ Stopped by timeout (Kaggle)")
        print(f"     Upload '{CFG.checkpoint_name}' + '{CFG.split_indices_name}' as dataset")
        print(f"     to continue in another Kaggle session.")
        print(f"     Or set skip_training=True in the next session to go directly to evaluation.")
    print(f"  Model saved at: {best_model_path}")
    print(f"  Checkpoint en:      {checkpoint_path}")
    if swa_model is not None and swa_n > 0:
        print(f"  SWA model at:       {swa_model_path} ({swa_n} updates)")
    print(f"{'='*70}")

    # ─── Final metrics: validate the BEST model (don't use history[-1]) ───
    # Previously it printed history['physics_metrics'][-1] which is the LAST epoch,
    # not necessarily the best. Now we re-validate the loaded best model.
    time_remaining = CFG.kaggle_time_limit - (time.time() - T_NOTEBOOK_START)
    if time_remaining > 300:
        # There is time → real validation of the best model
        print(f"\n  📊 Validating best model (best checkpoint)...")
        _, _, best_phys = validate(model, val_loader, criterion, DEVICE)
        print(f"     Dose Accuracy (3%): {best_phys['dose_acc_3pct']:.1f}%")
        print(f"     Dose Accuracy (5%): {best_phys['dose_acc_5pct']:.1f}%")
        print(f"     D_max error:        {best_phys['dmax_err_pct']:.2f}%")
        print(f"     MAE global:         {best_phys['mae']:.5f}")
        print(f"     MAE zona dosis:     {best_phys.get('mae_dose', 0):.5f}")
    else:
        # No time → find the best epoch in history
        print(f"\n  📊 Best model metrics (from history, no time for re-validation):")
        best_da_idx = max(
            range(len(history['physics_metrics'])),
            key=lambda i: history['physics_metrics'][i].get('dose_acc_3pct', 0)
                          if history['physics_metrics'][i] else 0
        )
        best_phys = history['physics_metrics'][best_da_idx]
        if best_phys:
            print(f"     (Best epoch: {best_da_idx})")
            print(f"     Dose Accuracy (3%): {best_phys['dose_acc_3pct']:.1f}%")
            print(f"     Dose Accuracy (5%): {best_phys['dose_acc_5pct']:.1f}%")
            print(f"     D_max error:        {best_phys['dmax_err_pct']:.2f}%")
            print(f"     MAE global:         {best_phys['mae']:.5f}")
            print(f"     MAE zona dosis:     {best_phys.get('mae_dose', 0):.5f}")


    print(f"{'='*70}")
    print(f"{'='*70}")




In [ ]:
"""
CELL 14: Training curves visualization
=====================================================
We plot the evolution of loss and its components.
This is essential for diagnosing training problems.
"""

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

# --- 1. Total loss ---
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train', alpha=0.8)
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val', alpha=0.8, linewidth=2)
axes[0, 0].set_title('Total Loss', fontweight='bold')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_yscale('log')
# Mark best epoch
best_epoch = np.argmin(history['val_loss'])
axes[0, 0].axvline(best_epoch + 1, color='green', ls='--', alpha=0.5, label=f'Best: ep.{best_epoch+1}')

# --- 2. Learning Rate ---
axes[0, 1].plot(epochs_range, history['lr'], 'g-', linewidth=2)
axes[0, 1].set_title('Learning Rate Schedule (warm restarts)', fontweight='bold')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('LR')
axes[0, 1].grid(True, alpha=0.3)
# Mark session restarts (where LR jumps → warm restart)
lr_array = np.array(history['lr'])
for i in range(1, len(lr_array)):
    if lr_array[i] > lr_array[i-1] * 2.0:  # Detect LR jump (warm restart)
        axes[0, 1].axvline(i + 1, color='orange', ls='--', alpha=0.7, label='Warm restart' if i == 1 else '')
axes[0, 1].legend()

# --- 3. Time per epoch ---
axes[0, 2].plot(epochs_range, history['epoch_time'], 'purple', alpha=0.7)
axes[0, 2].set_title('Time per Epoch', fontweight='bold')
axes[0, 2].set_xlabel('Epoch'); axes[0, 2].set_ylabel('Seconds')
axes[0, 2].grid(True, alpha=0.3)
avg_time = np.mean(history['epoch_time'])
axes[0, 2].axhline(avg_time, color='red', ls='--', label=f'Mean: {avg_time:.1f}s')
axes[0, 2].legend()

# --- 4-6. Loss components ---
component_names = ['mse_w', 'gradient', 'ssim', 'dvh', 'falloff', 'cosine']
component_labels = ['Weighted MSE', 'Gradients', 'SSIM', 'DVH', 'Fall-off', 'Cosine']
colors = ['blue', 'orange', 'green', 'red', 'purple', 'cyan']

for i, (comp_name, comp_label, color) in enumerate(zip(component_names, component_labels, colors)):
    train_vals = [c.get(comp_name, 0) for c in history['train_components']]
    val_vals = [c.get(comp_name, 0) for c in history['val_components']]

    if i < 2:
        ax = axes[1, i]
        ax.plot(epochs_range, train_vals, color=color, alpha=0.5, linestyle='--', label='Train')
        ax.plot(epochs_range, val_vals, color=color, label='Val', linewidth=1.5)
        ax.set_title(f'{comp_label} Loss', fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    else:
        # SSIM + DVH + Fall-off + Cosine comparten el tercer panel
        if i == 2:
            axes[1, 2].clear()
        axes[1, 2].plot(epochs_range, train_vals, color=color, alpha=0.5, linestyle='--')
        axes[1, 2].plot(epochs_range, val_vals, color=color, label=f'{comp_label} (val)', linewidth=1.5)
        axes[1, 2].set_title('SSIM + DVH + Fall-off + Cosine', fontweight='bold')
        axes[1, 2].set_xlabel('Epoch'); axes[1, 2].legend(fontsize=8)
        axes[1, 2].grid(True, alpha=0.3)

fig.suptitle('Training Curves — TransUNet + FiLM Surrogate',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_OUTPUT, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💡 DIAGNOSTICS:")
train_final = history['train_loss'][-1]
val_final = history['val_loss'][-1]
ratio = val_final / (train_final + 1e-8)
if ratio > 2.0:
    print(f"   ⚠️ Possible OVERFITTING: val/train ratio = {ratio:.1f}")
    print(f"   Consider: more data, more dropout, higher weight decay")
elif ratio < 1.1:
    print(f"   ✅ Good train/val balance: ratio = {ratio:.2f}")
    print(f"   The model generalizes well")
else:
    print(f"   ℹ️ Val/train ratio = {ratio:.2f} — acceptable")

if history['val_loss'][-1] > history['val_loss'][len(history['val_loss'])//2]:
    print(f"   ⚠️ Val loss rising → possible late overfitting")
else:
    print(f"   ✅ Val loss descending → healthy training")

---
# PART 6: SURROGATE EVALUATION
---

## 6.1 — Dosimetric Evaluation Metrics

### How Do We Know the Surrogate Is "Good Enough"?

In clinical dosimetry, the standard of comparison is the **Gamma Index** (Low et al., 1998):

#### Gamma Index ($\gamma$)

It combines two criteria simultaneously:
1. **Dose difference** (DD): does the predicted dose differ by more than X% of the maximum?
2. **Distance-to-agreement** (DTA): how far is the nearest point with the same dose?

$$\gamma(\mathbf{r}_r) = \min_{\mathbf{r}_e} \sqrt{\frac{|\mathbf{r}_r - \mathbf{r}_e|^2}{\Delta d^2} + \frac{|D_r(\mathbf{r}_r) - D_e(\mathbf{r}_e)|^2}{\Delta D^2}}$$

Where:
- $\mathbf{r}_r$ is the reference position (MC), $\mathbf{r}_e$ the evaluation position (network)
- $\Delta d$ = distance criterion (typical: 3 mm)
- $\Delta D$ = dose criterion (typical: 3% of $D_{max}$)

**Interpretation**: $\gamma \leq 1$ → the point PASSES. The reported metric is the **passing rate**: percentage of voxels with $\gamma \leq 1$.

**Clinical criterion**: passing rate ≥ 95% with 3%/3mm is acceptable for treatment planning.

### Dose-Zone Masking (>10% of $D_{max}$)

Following clinical practice (AAPM TG-218), the gamma index is computed only in the **clinical dose zone** — voxels where the dose exceeds 10% of $D_{max}$. Low-dose regions (bremsstrahlung scatter, leakage) are clinically irrelevant and would artificially inflate the pass rate. This same masking is applied to:
- **DoseAcc3%/5%** (training validation metric)
- **Relative error** (computed only in the dose zone)
- **DVH analysis** (the DVH is inherently computed over the PTV/OAR volume)

### Other Metrics

| Metric | Formula | Clinical Significance |
|---|---|---|
| MAE | $\frac{1}{N}\sum\|D_{pred} - D_{true}\|$ | Mean absolute error |
| RMSE | $\sqrt{\frac{1}{N}\sum(D_{pred} - D_{true})^2}$ | Root mean squared error |
| Rel. Error | $\frac{\|D_{pred} - D_{true}\|}{D_{true}}$ | Relative error (in dose zone only) |
| $\Delta D_{95}$ | $D_{95,pred} - D_{95,true}$ | Difference in tumor coverage |
| $\Delta D_{max}$ | $D_{max,pred} - D_{max,true}$ | Difference in maximum dose |
| PDD error | Error in depth dose profile | Accuracy in the characteristic curve |
| DoseAcc X% | % voxels within X% agreement (dose zone) | Fast gamma-like metric for training |

### Evaluation Outputs

The evaluation code generates, for each test sample:
1. **Summary table**: MAE, RMSE, relative error (mean, std, min, max), gamma pass rate
2. **Axial visualization**: density, predicted dose, MC dose, difference map, gamma index map, pass/fail map
3. **Sagittal visualization**: same metrics on a perpendicular plane
4. **PDD comparison**: predicted vs MC depth dose profiles
5. **Lateral profile comparison**: predicted vs MC at mid-depth
6. **DVH comparison**: cumulative dose-volume histograms

In [ ]:
"""
CELL 15: Gamma Index 3D and evaluation metrics implementation
======================================================================
The gamma index is the clinical standard for comparing dose distributions.
We implement an efficient version with scipy.
"""
from scipy.ndimage import distance_transform_edt

def gamma_index_3d(reference, evaluation, dose_threshold_pct=3.0,
                   distance_threshold_mm=3.0, voxel_size_mm=(2.0, 2.0, 2.0),
                   dose_cutoff_pct=10.0):
    """
    Computes the 3D Gamma Index between two dose distributions.

    Algorithm:
      For each voxel of the reference (MC):
      1. Compute the dose difference with all nearby voxels of the evaluation (network)
      2. Compute the physical distance to all nearby voxels
      3. γ = min(sqrt((Δd/Δd_threshold)² + (ΔD/ΔD_threshold)²))
      4. Si γ ≤ 1 → the voxel PASSES

    Optimization:
      Instead of comparing all against all (O(N²)), we use a
      local search with radius = 2 × distance_threshold. This reduces complexity
      dramatically.

    Args:
        reference: 3D array (Z, Y, X) — reference distribution (MC)
        evaluation: 3D array — distribution to evaluate (prediction)
        dose_threshold_pct: dose criterion (% of D_max)
        distance_threshold_mm: distance criterion (mm)
        voxel_size_mm: voxel size in mm (dz, dy, dx)
        dose_cutoff_pct: ignore voxels with dose < X% of max

    Returns:
        gamma_map: 3D array with γ values
        passing_rate: % of voxels that pass (γ ≤ 1)
        stats: dictionary with statistics
    """
    ref = reference.astype(np.float64)
    eva = evaluation.astype(np.float64)

    # Normalize to reference maximum
    ref_max = ref.max()
    if ref_max <= 0:
        return np.zeros_like(ref), 100.0, {'mean_gamma': 0.0}

    ref_norm = ref / ref_max
    eva_norm = eva / ref_max  # Same normalization

    # Thresholds
    dose_threshold = dose_threshold_pct / 100.0  # Fraction of D_max
    dose_cutoff = dose_cutoff_pct / 100.0

    # Mask: only evaluate where dose is significant
    mask = ref_norm > dose_cutoff

    if mask.sum() == 0:
        return np.zeros_like(ref), 100.0, {'mean_gamma': 0.0}

    # Simplified but robust gamma index
    # For each voxel, we compute γ based on the local dose difference
    # and the distance to the nearest dose point

    gamma_map = np.full_like(ref, np.inf)

    # Search radius in voxels
    search_radius = [int(np.ceil(2 * distance_threshold_mm / vs)) for vs in voxel_size_mm]

    # Direct dose difference (DD term)
    dose_diff = np.abs(ref_norm - eva_norm)
    dd_term = (dose_diff / dose_threshold) ** 2

    # For the DTA term, we search in a local neighborhood
    nz, ny, nx = ref.shape

    # Efficient method: precompute neighborhood offsets
    offsets = []
    for dz in range(-search_radius[0], search_radius[0] + 1):
        for dy in range(-search_radius[1], search_radius[1] + 1):
            for dx in range(-search_radius[2], search_radius[2] + 1):
                dist_mm = np.sqrt((dz * voxel_size_mm[0])**2 +
                                  (dy * voxel_size_mm[1])**2 +
                                  (dx * voxel_size_mm[2])**2)
                if dist_mm <= 2 * distance_threshold_mm:
                    offsets.append((dz, dy, dx, dist_mm))

    # For each offset, compute gamma
    for dz, dy, dx, dist_mm in offsets:
        # Shifted evaluation array
        z_start = max(0, dz)
        z_end = min(nz, nz + dz)
        y_start = max(0, dy)
        y_end = min(ny, ny + dy)
        x_start = max(0, dx)
        x_end = min(nx, nx + dx)

        z_start_e = max(0, -dz)
        z_end_e = min(nz, nz - dz)
        y_start_e = max(0, -dy)
        y_end_e = min(ny, ny - dy)
        x_start_e = max(0, -dx)
        x_end_e = min(nx, nx - dx)

        if z_end <= z_start or y_end <= y_start or x_end <= x_start:
            continue

        ref_patch = ref_norm[z_start:z_end, y_start:y_end, x_start:x_end]
        eva_patch = eva_norm[z_start_e:z_end_e, y_start_e:y_end_e, x_start_e:x_end_e]

        local_dose_diff = np.abs(ref_patch - eva_patch) / dose_threshold
        local_dist = dist_mm / distance_threshold_mm
        local_gamma = np.sqrt(local_dose_diff**2 + local_dist**2)

        gamma_map[z_start:z_end, y_start:y_end, x_start:x_end] = np.minimum(
            gamma_map[z_start:z_end, y_start:y_end, x_start:x_end],
            local_gamma
        )

    # Apply mask
    gamma_masked = gamma_map[mask]
    passing_rate = 100.0 * np.sum(gamma_masked <= 1.0) / len(gamma_masked)

    stats = {
        'passing_rate': passing_rate,
        'mean_gamma': float(np.mean(gamma_masked)),
        'max_gamma': float(np.max(gamma_masked)),
        'median_gamma': float(np.median(gamma_masked)),
        'n_evaluated': int(mask.sum()),
    }

    return gamma_map, passing_rate, stats


def compute_dose_metrics(pred, target, mask=None):
    """
    Computes standard dosimetric metrics.

    Args:
        pred, target: 3D normalized arrays [0, 1]
        mask: optional mask (evaluate only region of interest)

    Returns:
        dict with MAE, RMSE, D95, D_max, D_mean
    """
    if mask is not None:
        p = pred[mask]
        t = target[mask]
    else:
        # Only where there is significant dose
        sig_mask = target > 0.1 * target.max()
        p = pred[sig_mask]
        t = target[sig_mask]

    if len(t) == 0:
        return {}

    mae = np.mean(np.abs(p - t))
    rmse = np.sqrt(np.mean((p - t) ** 2))
    rel_error = np.mean(np.abs(p - t) / (t + 1e-8)) * 100  # %

    # DVH metrics
    d95_pred = np.percentile(p, 5) if len(p) > 0 else 0  # D95 = dose received by 95% of vol
    d95_true = np.percentile(t, 5) if len(t) > 0 else 0
    d_max_pred = p.max() if len(p) > 0 else 0
    d_max_true = t.max() if len(t) > 0 else 0
    d_mean_pred = p.mean() if len(p) > 0 else 0
    d_mean_true = t.mean() if len(t) > 0 else 0

    return {
        'MAE': mae,
        'RMSE': rmse,
        'Rel_Error_%': rel_error,
        'D95_pred': d95_pred,
        'D95_true': d95_true,
        'D95_diff': abs(d95_pred - d95_true),
        'Dmax_pred': d_max_pred,
        'Dmax_true': d_max_true,
        'Dmax_diff': abs(d_max_pred - d_max_true),
        'Dmean_pred': d_mean_pred,
        'Dmean_true': d_mean_true,
        'Dmean_diff': abs(d_mean_pred - d_mean_true),
    }


print("✅ Evaluation metrics defined:")
print("   • gamma_index_3d(): 3D Gamma Index with configurable criterion (default 3%/3mm)")
print("   • compute_dose_metrics(): MAE, RMSE, D95, D_max, D_mean")

In [ ]:
"""
CELL 16: Complete evaluation — visualization and quantitative metrics
=======================================================================
We run the evaluation on the test set:
1. Model predictions
2. Gamma Index 3D
3. Visual comparison (axial/sagittal slices)
4. Comparative DVH
5. Metrics summary table
"""

def evaluate_model_full(model, test_loader, config, device, n_visualize=3):
    """
    Exhaustive evaluation of the surrogate model.

    Genera:
    - Gamma index for each test set sample
    - Visual comparison plots
    - DVH comparativos
    - Metrics table

    Args:
        model: TransUNetFiLM3D entrenado
        test_loader: DataLoader de test
        config: Config instance
        device: torch device
        n_visualize: number of samples to visualize in detail
    """
    model.eval()
    all_metrics = []
    all_gamma_rates = []

    print("=" * 70)
    print("COMPLETE SURROGATE MODEL EVALUATION")
    print("=" * 70)

    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            density = batch['density'].to(device)    # (B, 1, D, H, W)
            params = batch['params'].to(device)       # (B, 10)
            dose_true = batch['dose'].to(device)      # (B, 1, D, H, W)

            # Prediction
            dose_pred, _ = model(density, params)

            # Convert to numpy for metrics
            for i in range(density.shape[0]):
                pred_np = dose_pred[i, 0].cpu().numpy()
                true_np = dose_true[i, 0].cpu().numpy()
                dens_np = density[i, 0].cpu().numpy()
                params_np = params[i].cpu().numpy()

                # Gamma Index
                gamma_map, pass_rate, gamma_stats = gamma_index_3d(
                    true_np, pred_np,
                    dose_threshold_pct=config.gamma_dose_threshold if hasattr(config, 'gamma_dose_threshold') else 3.0,
                    distance_threshold_mm=config.gamma_dist_threshold if hasattr(config, 'gamma_dist_threshold') else 3.0,
                )
                gamma_stats['sample_idx'] = batch_idx * test_loader.batch_size + i
                all_gamma_rates.append(pass_rate)

                # Dosimetric metrics
                metrics = compute_dose_metrics(pred_np, true_np)
                metrics['gamma_pass_rate'] = pass_rate
                metrics['params'] = params_np
                all_metrics.append(metrics)

                # Detailed visualization of the first samples
                sample_idx = batch_idx * test_loader.batch_size + i
                if sample_idx < n_visualize:
                    _visualize_comparison(
                        dens_np, true_np, pred_np, gamma_map,
                        params_np, metrics, sample_idx
                    )

    # ================================================================
    # SUMMARY TABLE
    # ================================================================
    print("\n" + "=" * 70)
    print("METRICS SUMMARY (Test Set)")
    print("=" * 70)

    maes = [m['MAE'] for m in all_metrics]
    rmses = [m['RMSE'] for m in all_metrics]
    rel_errors = [m['Rel_Error_%'] for m in all_metrics]
    d95_diffs = [m['D95_diff'] for m in all_metrics]

    print(f"\n{'Metric':<25} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
    print("-" * 65)
    print(f"{'MAE (norm.)':<25} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
    print(f"{'RMSE (norm.)':<25} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
    print(f"{'Relative error (%)':<25} {np.mean(rel_errors):>10.2f} {np.std(rel_errors):>10.2f} {np.min(rel_errors):>10.2f} {np.max(rel_errors):>10.2f}")
    print(f"{'ΔD95':<25} {np.mean(d95_diffs):>10.4f} {np.std(d95_diffs):>10.4f} {np.min(d95_diffs):>10.4f} {np.max(d95_diffs):>10.4f}")
    print(f"{'Gamma Pass Rate (%)':<25} {np.mean(all_gamma_rates):>10.2f} {np.std(all_gamma_rates):>10.2f} {np.min(all_gamma_rates):>10.2f} {np.max(all_gamma_rates):>10.2f}")

    # Clinical criterion
    mean_gamma = np.mean(all_gamma_rates)
    print(f"\n{'—' * 50}")
    if mean_gamma >= 95:
        print(f"✅ RESULT: Mean Gamma Pass Rate = {mean_gamma:.1f}% ≥ 95%")
        print("   The model meets the clinical standard for treatment planning")
    elif mean_gamma >= 90:
        print(f"⚠️  RESULT: Mean Gamma Pass Rate = {mean_gamma:.1f}%")
        print("   Acceptable for screening, but requires refinement for clinical use")
    else:
        print(f"❌ RESULT: Mean Gamma Pass Rate = {mean_gamma:.1f}% < 90%")
        print("   The model needs more training or data")

    return all_metrics, all_gamma_rates


def _visualize_comparison(density, dose_true, dose_pred, gamma_map,
                          params, metrics, sample_idx):
    """Comparative visualization for a sample."""

    nz, ny, nx = dose_true.shape
    mid_z, mid_y, mid_x = nz // 2, ny // 2, nx // 2

    fig, axes = plt.subplots(4, 3, figsize=(18, 20))
    # Decode energy and patient type for the title
    en_map = {0.0: '6MV', 0.5: '10MV', 1.0: '15MV'}
    en_val = min(en_map.keys(), key=lambda k: abs(k - params[0]))
    en_str = en_map[en_val] + (' FFF' if params[1] > 0.5 else '')
    pat_map = {0.0: 'pelvis', 0.5: 'head-neck', 1.0: 'lung'}
    pat_val = min(pat_map.keys(), key=lambda k: abs(k - params[7]))
    pat_str = pat_map[pat_val]

    fig.suptitle(
        f"Sample #{sample_idx} — {en_str}, "
        f"Fx={params[2]:.1f}cm, Fy={params[3]:.1f}cm, "
        f"SSD={params[4]:.1f}cm, G={params[5]:.1f}°, "
        f"Coll={params[6]:.1f}°, {pat_str}, "
        f"d={params[8]:.1f}cm, s={params[9]:.1f}cm\n"
        f"MAE={metrics['MAE']:.4f} | RMSE={metrics['RMSE']:.4f} | "
        f"γ Pass Rate={metrics['gamma_pass_rate']:.1f}%",
        fontsize=12, fontweight='bold'
    )

    # Common normalization
    vmax = max(dose_true.max(), dose_pred.max())

    # --- Row 1: Density + MC Dose (reference) ---
    ax = axes[0, 0]
    ax.imshow(density[mid_z], cmap='bone', aspect='auto')
    ax.set_title('Density (axial)', fontweight='bold')
    ax.set_ylabel('Axial Slice\n(z=mid)')

    ax = axes[0, 1]
    im = ax.imshow(dose_true[mid_z], cmap='jet', aspect='auto', vmin=0, vmax=vmax)
    ax.set_title('MC Dose (reference)', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[0, 2]
    im = ax.imshow(dose_pred[mid_z], cmap='jet', aspect='auto', vmin=0, vmax=vmax)
    ax.set_title('NETWORK Dose (prediction)', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

    # --- Row 2: Difference + Gamma (axial) ---
    ax = axes[1, 0]
    diff = dose_pred[mid_z] - dose_true[mid_z]
    vdiff = max(abs(diff.min()), abs(diff.max())) + 1e-8
    im = ax.imshow(diff, cmap='bwr', aspect='auto', vmin=-vdiff, vmax=vdiff)
    ax.set_title('Difference (Pred - MC)', fontweight='bold')
    ax.set_ylabel('Difference\nand Gamma')
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[1, 1]
    gamma_slice = gamma_map[mid_z].copy()
    gamma_slice[gamma_slice == np.inf] = np.nan
    im = ax.imshow(gamma_slice, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=2)
    ax.set_title('Gamma Index (green = pass)', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[1, 2]
    # Binary map: pass / fail
    pass_map = np.zeros_like(gamma_map[mid_z])
    valid = ~np.isinf(gamma_map[mid_z])
    pass_map[valid & (gamma_map[mid_z] <= 1)] = 1  # Pass
    pass_map[valid & (gamma_map[mid_z] > 1)] = -1   # Fail
    colors = ['red', 'white', 'green']
    cmap_pass = plt.cm.colors.ListedColormap(colors)
    im = ax.imshow(pass_map, cmap=cmap_pass, aspect='auto', vmin=-1, vmax=1)
    ax.set_title('Pass/Fail (γ ≤ 1)', fontweight='bold')

    # --- Row 3: Sagittal slices ---
    ax = axes[2, 0]
    ax.imshow(density[:, mid_y, :], cmap='bone', aspect='auto')
    ax.set_title('Density (sagittal)', fontweight='bold')
    ax.set_ylabel('Sagittal Slice\n(y=mid)')

    ax = axes[2, 1]
    im = ax.imshow(dose_true[:, mid_y, :], cmap='jet', aspect='auto', vmin=0, vmax=vmax)
    ax.set_title('MC Dose (sagittal)', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[2, 2]
    im = ax.imshow(dose_pred[:, mid_y, :], cmap='jet', aspect='auto', vmin=0, vmax=vmax)
    ax.set_title('NETWORK Dose (sagittal)', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

    # --- Row 4: PDD + Lateral profile + DVH ---
    ax = axes[3, 0]
    pdd_true = dose_true[:, mid_y, mid_x]
    pdd_pred = dose_pred[:, mid_y, mid_x]
    depth = np.arange(len(pdd_true))
    ax.plot(depth, pdd_true, 'b-', linewidth=2, label='MC')
    ax.plot(depth, pdd_pred, 'r--', linewidth=2, label='RED')
    ax.fill_between(depth, pdd_true, pdd_pred, alpha=0.3, color='gray')
    ax.set_xlabel('Depth (voxels)')
    ax.set_ylabel('Dose (normalized)')
    ax.set_title('PDD — Percentage Depth Dose', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[3, 1]
    profile_true = dose_true[mid_z, mid_y, :]
    profile_pred = dose_pred[mid_z, mid_y, :]
    x_pos = np.arange(len(profile_true))
    ax.plot(x_pos, profile_true, 'b-', linewidth=2, label='MC')
    ax.plot(x_pos, profile_pred, 'r--', linewidth=2, label='RED')
    ax.set_xlabel('Lateral position (voxels)')
    ax.set_ylabel('Dose')
    ax.set_title('Lateral Profile', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[3, 2]
    # DVH simplificado
    bins = np.linspace(0, vmax, 100)
    dvh_true = np.array([np.sum(dose_true >= b) / dose_true.size * 100 for b in bins])
    dvh_pred = np.array([np.sum(dose_pred >= b) / dose_pred.size * 100 for b in bins])
    ax.plot(bins / vmax * 100, dvh_true, 'b-', linewidth=2, label='MC')
    ax.plot(bins / vmax * 100, dvh_pred, 'r--', linewidth=2, label='RED')
    ax.set_xlabel('Dose (% of maximum)')
    ax.set_ylabel('Volume (%)')
    ax.set_title('DVH — Dose Volume Histogram', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'evaluation_sample_{sample_idx}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"   → Saved: evaluation_sample_{sample_idx}.png\n")


# ================================================================
# RUN EVALUATION
# ================================================================
print("Running complete evaluation on the test set...\n")

all_metrics, all_gamma_rates = evaluate_model_full(
    model, test_loader, CFG, DEVICE, n_visualize=3
)



---

# 🎯 PART 7: Treatment Optimizer v2 (Stage 2)

## The Inverse Problem of Radiation Therapy

So far we have solved the **forward problem** (direct):

$$\text{Surrogate}: \quad (\rho, \mathbf{p}) \xrightarrow{\text{TransUNet}} D(\mathbf{r})$$

Now we address the **inverse problem** (what is really useful in the clinic):

$$\text{Optimizador}: \quad D_{\text{objetivo}} \xrightarrow{?} \mathbf{p}^*$$

## Mathematical Formulation (extended ICRU-83)

Objective function with 6 clinical terms:

$$\mathcal{L}_{\text{opt}}(\mathbf{p}) = -\alpha \cdot D_{98} - \delta \cdot D_{98} - \zeta \cdot V_{95} + \beta \cdot D_{\max}^{\text{OAR}} + \varepsilon \cdot D_{\text{mean}}^{\text{OAR}} + \gamma \cdot \text{HI}$$

where:
- $D_{95}$, $D_{98}$: dose to 95%/98% of the tumor volume → **maximize**
- $V_{95}$: fraction of the tumor with dose ≥ 95% of $D_{50}$ → **maximize**
- $\text{HI} = (D_2 - D_{98}) / D_{50}$: homogeneity index → **minimize** (ideal = 0)
- $D_{\max}^{\text{OAR}}$: soft-top-1% of the OAR → **minimize**
- $D_{\text{mean}}^{\text{OAR}}$: mean dose to the OAR → **minimize**

## 6 Improvements in Optimizer v2

### 1. L-BFGS (quasi-Newton) Instead of Adam

For $n = 5$ continuous parameters, L-BFGS converges in **~25 steps** vs 200 for Adam. It uses the last $m$ gradient pairs $(s_k, y_k)$ to approximate $H^{-1}$ (the inverse Hessian):

$$\mathbf{p}_{k+1} = \mathbf{p}_k - H_k^{-1} \nabla \mathcal{L}(\mathbf{p}_k)$$

**This is "the gradient thing"**: leveraging curvature information from the Jacobian $\partial D / \partial \mathbf{p}$ of the surrogate to not only know *which direction to move* (Adam) but also *how far to move in each direction* (L-BFGS).

### 2. Sigmoid Reparameterization (gradients always flow)

Instead of hard clipping (`torch.clamp`, gradient = 0 at boundaries):

$$p_{\text{physical}} = p_{\min} + (p_{\max} - p_{\min}) \cdot \sigma(p_{\text{raw}})$$

$$\frac{\partial p}{\partial p_{\text{raw}}} = (p_{\max} - p_{\min}) \cdot \sigma(1 - \sigma) > 0 \quad \forall \; p_{\text{raw}}$$

Inspired by: PyDoseRT (Simkó et al., arXiv:2512.18863, 2025) — BeamValidationLayer.

### 3. Branch-and-Evaluate for Discrete Parameters

Energy ∈ {6MV, 10MV, 15MV}, FFF ∈ {yes, no} → **6 combinations** evaluated explicitly. Gradient optimization does not make sense for them (they are categorical). The best is selected.

### 4. Multi-start

$n$ random initializations per branch (with fast L-BFGS, the extra cost is ~$n \times 1$ second).

### 5. Extended ICRU-83 Clinical Objective

D95, D98, V95, HI + Dmax_OAR, Dmean_OAR (6 terms vs 3 previously).

### 6. MC Dropout Post-Optimization Evaluation

Quantify confidence of the optimal plan (Pim & Pryer, arXiv:2509.18155, 2025).

## Competitive Advantage

| Method | Time | Forward passes |
|--------|--------|----------------|
| Eclipse inverse planning | 5 min – 1 hour | ~10³ AcurosXB |
| Surrogate + Adam v1 | ~30 s | ~200 |
| **Surrogate + L-BFGS v2** | **~15 s** | **~2000 (6 branches × 3 starts × 25 steps × ~5 line search)** |

In [ ]:
"""
CELDA 17: Optimizador de Tratamiento EBRT v2 — TrueBeam SVC 3.0
=================================================================
State-of-the-art inverse planning optimizer.

Improvements over v1 (Adam + hard clipping):
  1. L-BFGS (quasi-Newton): converges in ~25 iters vs 200
     → Leverages curvature of the surrogate Jacobian ∂D/∂p
  2. Sigmoid reparameterization: gradients ALWAYS flow
     → p_physical = p_min + (p_max - p_min) · σ(raw)
  3. Branch-and-evaluate: enumerates {energy × FFF} (6 combos)
     → Discrete parameters are not gradient-optimized
  4. Multi-start: n_restarts inicializaciones aleatorias
     → Escapes local minima
  5. Extended ICRU-83 clinical objective:
     → D95, D98, V95, HI, Dmax_OAR, Dmean_OAR
  6. MC Dropout post-optimization evaluation (optional)
     → Quantifies optimal plan uncertainty

Basado en:
  - PyDoseRT (Simkó et al., arXiv:2512.18863, Dec 2025)
  - Pim & Pryer (arXiv:2509.18155, Sep 2025)
"""

import time


class EBRTOptimizer:
    """
    EBRT treatment optimizer v2 — State of the art.

    Uses the trained surrogate as a differentiable engine to
    find the optimal beam parameters of the TrueBeam SVC 3.0.

    Optimization architecture:
      1. Enumerate 6 discrete configurations {energy × FFF}
      2. For each one, launch n_restarts with L-BFGS
      3. Select the best of the 6 × n_restarts solutions
      4. Optionally, evaluate uncertainty with MC Dropout

    Beam parameters (10 total):
      DISCRETE (enumerated, not gradient-optimized):
        - beam_energy:    Beam energy (0=6MV, 0.5=10MV, 1.0=15MV)
        - is_fff:         Flattening Filter Free (0=FF, 1=FFF)
      CONTINUOUS (optimized with L-BFGS + sigmoid):
        - field_x_cm:     X field size/jaws (1-40 cm)
        - field_y_cm:     Y field size/HD MLC (1-22 cm)
        - ssd_cm:         Source-to-surface distance (80-110 cm)
        - gantry_deg:     Gantry angle (0-360°)
        - collimator_deg: Collimator angle (0-360°)
      FIXED PER PATIENT (not optimized):
        - patient_type:   Anatomical type (0=pelvis, 0.5=head-neck, 1=lung)
        - depth_cm:       Tumor depth (2-18 cm)
        - size_cm:        Tumor size (1-8 cm)
    """

    # ─── Physical constraints of TrueBeam SVC 3.0 + HD MLC ───
    PARAM_BOUNDS = {
        'beam_energy':    (0.0, 1.0),      # Ordinal: 6MV/10MV/15MV
        'is_fff':         (0.0, 1.0),      # Binario: FF o FFF
        'field_x_cm':     (1.0, 40.0),     # Jaws X (max 40 cm)
        'field_y_cm':     (1.0, 22.0),     # HD MLC Y (max 22 cm!)
        'ssd_cm':         (80.0, 110.0),   # Standard SSD
        'gantry_deg':     (0.0, 360.0),    # Full gantry rotation
        'collimator_deg': (0.0, 360.0),    # Collimator rotation
        'patient_type':   (0.0, 1.0),      # 0=pelvis, 0.5=head-neck, 1=lung
        'depth_cm':       (2.0, 18.0),     # Tumor depth
        'size_cm':        (1.0, 8.0),      # Tumor size
    }

    # Discrete configurations to enumerate: 3 energies × 2 filters = 6 branches
    ENERGY_FFF_COMBOS = [
        (0.0, 0.0, '6MV FF'),
        (0.0, 1.0, '6MV FFF'),
        (0.5, 0.0, '10MV FF'),
        (0.5, 1.0, '10MV FFF'),
        (1.0, 0.0, '15MV FF'),
        (1.0, 1.0, '15MV FFF'),
    ]

    # Indices of each parameter type in the vector of 10
    CONTINUOUS_INDICES = [2, 3, 4, 5, 6]   # field_x, field_y, SSD, gantry, collimator
    FIXED_INDICES = [7, 8, 9]              # patient_type, depth, size
    N_CONTINUOUS = 5

    def __init__(self, surrogate_model, device, config,
                 alpha_coverage=1.0, beta_oar=0.5, gamma_homogeneity=0.3,
                 delta_d98=0.5, epsilon_dmean_oar=0.3, zeta_v95=0.5):
        """
        Args:
            surrogate_model: TransUNetFiLM3D entrenado
            device: torch device
            config: Config object
            alpha_coverage:    D95 tumor weight (maximize)
            beta_oar:          Dmax OAR weight (minimize)
            gamma_homogeneity: HI weight (minimize)
            delta_d98:         D98 tumor weight (maximize)
            epsilon_dmean_oar: Dmean OAR weight (minimize)
            zeta_v95:          V95 weight (maximize)
        """
        self.model = surrogate_model
        self.model.eval()
        # Freeze ALL surrogate weights
        for p in self.model.parameters():
            p.requires_grad = False

        self.device = device
        self.config = config

        # Clinical objective weights
        self.alpha   = alpha_coverage
        self.beta    = beta_oar
        self.gamma   = gamma_homogeneity
        self.delta   = delta_d98
        self.epsilon = epsilon_dmean_oar
        self.zeta    = zeta_v95

        # Bounds as tensors
        bounds = list(self.PARAM_BOUNDS.values())
        self.param_min = torch.tensor(
            [b[0] for b in bounds], device=device, dtype=torch.float32
        )
        self.param_max = torch.tensor(
            [b[1] for b in bounds], device=device, dtype=torch.float32
        )

        # Bounds only for the 5 continuous parameters
        self.cont_min = self.param_min[self.CONTINUOUS_INDICES]
        self.cont_max = self.param_max[self.CONTINUOUS_INDICES]

        # MC Dropout stats (se rellena si mc_eval_samples > 0)
        self.last_mc_stats = None

    # ═══════════════════════════════════════
    # PARAMETER TRANSFORMATIONS
    # ═══════════════════════════════════════

    def normalize_params(self, params_physical):
        """Physical → [0,1] for the model."""
        return (params_physical - self.param_min) / (self.param_max - self.param_min + 1e-8)

    def denormalize_params(self, params_norm):
        """[0,1] → physical."""
        return params_norm * (self.param_max - self.param_min) + self.param_min

    def _raw_to_physical(self, raw_continuous):
        """
        Sigmoid reparameterization: (-∞, +∞) → [min, max].
        Gradients ALWAYS flow (never zero).

            p = p_min + (p_max - p_min) · σ(raw)
            dp/draw = (p_max - p_min) · σ · (1 - σ)  > 0  ∀ raw
        """
        return self.cont_min + (self.cont_max - self.cont_min) * torch.sigmoid(raw_continuous)

    def _physical_to_raw(self, physical_continuous):
        """Inverse: physical → raw (logit). For initialization."""
        normed = (physical_continuous - self.cont_min) / (self.cont_max - self.cont_min + 1e-8)
        normed = normed.clamp(0.001, 0.999)
        return torch.logit(normed)

    def _build_full_params(self, raw_continuous, energy_val, fff_val, patient_fixed):
        """
        Assembles vector of 10 parameters in physical space.
        Only raw_continuous has requires_grad=True → gradients flow
        through sigmoid(raw_continuous) via torch.cat.
        """
        continuous_physical = self._raw_to_physical(raw_continuous)
        discrete = torch.tensor(
            [energy_val, fff_val], device=self.device, dtype=torch.float32
        )
        return torch.cat([discrete, continuous_physical, patient_fixed])

    # ═══════════════════════════════════════
    # CLINICAL OBJECTIVE (ICRU-83)
    # ═══════════════════════════════════════

    def clinical_objective(self, dose_3d, tumor_mask, oar_mask):
        """
        Extended clinical objective function (ICRU-83).

        Tumor metrics (all differentiable via soft-percentiles):
          - D95: dose to 95% of the volume → MAXIMIZE
          - D98: dose to 98% of the volume (near-minimum) → MAXIMIZE
          - V95: fraction of tumor with dose ≥ 95% of D50 → MAXIMIZE
          - HI:  (D5 - D95) / D50 → MINIMIZAR (ideal = 0)

        OAR metrics:
          - Dmax: soft-top-1% of OAR → MINIMIZE
          - Dmean: OAR mean → MINIMIZE

        Loss = -α·D95 - δ·D98 - ζ·V95 + β·Dmax_OAR + ε·Dmean_OAR + γ·HI
        """
        dose = dose_3d.squeeze()  # (D, H, W)

        # ── Tumor dose ──
        tumor_dose = dose[tumor_mask]
        if tumor_dose.numel() == 0:
            return torch.tensor(0.0, device=self.device, requires_grad=True), {}

        sorted_tumor, _ = torch.sort(tumor_dose)
        n = sorted_tumor.numel()

        # D95: 5th percentile (95% of the volume receives at least this)
        n5 = max(1, int(0.05 * n))
        d95 = sorted_tumor[:n5].mean()

        # D98: percentil 2 (near-minimum dose, ICRU-83)
        n2 = max(1, int(0.02 * n))
        d98 = sorted_tumor[:n2].mean()

        # D2 and D50 for HI (ICRU Report 83, §3.7.3)
        n98_hi = max(1, int(0.98 * n))
        d2  = sorted_tumor[n98_hi:].mean()  # Top 2% = near-maximum
        d50 = sorted_tumor[n // 2]

        # HI (ICRU-83 §3.7.3): (D2 - D98) / D50 — ideal = 0
        hi = (d2 - d98) / (d50 + 1e-8)

        # V95: fraction with dose ≥ 95% of D50 (soft sigmoid, differentiable)
        # detach d50 to avoid double gradient through the threshold
        threshold_95 = 0.95 * d50.detach()
        v95 = torch.sigmoid(50.0 * (tumor_dose - threshold_95)).mean()

        # ── OAR dose ──
        oar_dose = dose[oar_mask]
        if oar_dose.numel() > 0:
            sorted_oar, _ = torch.sort(oar_dose, descending=True)
            n1pct = max(1, int(0.01 * len(sorted_oar)))
            dmax_oar  = sorted_oar[:n1pct].mean()
            dmean_oar = oar_dose.mean()
        else:
            dmax_oar  = torch.tensor(0.0, device=self.device)
            dmean_oar = torch.tensor(0.0, device=self.device)

        # ── Total loss: minimize ──
        loss = (
            -self.alpha   * d95
            -self.delta   * d98
            -self.zeta    * v95
            +self.beta    * dmax_oar
            +self.epsilon * dmean_oar
            +self.gamma   * hi
        )

        info = {
            'D95_tumor': d95.item(),
            'D98_tumor': d98.item(),
            'D2_tumor':  d2.item(),
            'D50_tumor': d50.item(),
            'Dmax_OAR':  dmax_oar.item(),
            'Dmean_OAR': dmean_oar.item() if isinstance(dmean_oar, torch.Tensor) else dmean_oar,
            'V95':       v95.item(),
            'HI':        hi.item(),
            'loss':      loss.item(),
        }

        return loss, info

    # ═══════════════════════════════════════
    # ANATOMICAL MASKS
    # ═══════════════════════════════════════

    def create_anatomical_masks(self, density):
        """
        Creates tumor and OAR masks from the density map.

        ⚠️ HEURISTIC: in production, use RT-Structure contours (DICOM).
        Here we approximate with density thresholds for demonstration.

        Args:
            density: tensor (D, H, W) o (C, D, H, W) — se usa canal 0 si 4D
        """
        # If multi-channel, use channel 0 = physical density
        if density.dim() == 4:
            density = density[0]

        tumor_mask = (density > 1.02) & (density < 1.07)
        bone_mask  = density > 1.2
        oar_mask   = (density > 0.3) & (density < 1.02) & (~tumor_mask) & (~bone_mask)

        # Heuristic fallback: central sphere
        if tumor_mask.sum() < 10:
            nz, ny, nx = density.shape
            cz, cy, cx = nz // 2, ny // 2, nx // 2
            r = min(nz, ny, nx) // 8
            zz, yy, xx = torch.meshgrid(
                torch.arange(nz, device=density.device),
                torch.arange(ny, device=density.device),
                torch.arange(nx, device=density.device),
                indexing='ij'
            )
            tumor_mask = ((zz - cz)**2 + (yy - cy)**2 + (xx - cx)**2) < r**2

        if oar_mask.sum() < 10:
            from scipy.ndimage import binary_dilation
            oar_np = binary_dilation(
                tumor_mask.cpu().numpy(), iterations=5
            ) & ~tumor_mask.cpu().numpy()
            oar_mask = torch.tensor(oar_np, device=density.device)

        return tumor_mask, oar_mask

    # ═══════════════════════════════════════
    # SINGLE BRANCH OPTIMIZATION (L-BFGS)
    # ═══════════════════════════════════════

    def _optimize_branch(self, density_input, tumor_mask, oar_mask,
                         energy_val, fff_val, patient_fixed,
                         n_restarts, n_lbfgs_steps):
        """
        Optimizes continuous parameters for ONE discrete configuration.
        Multi-start L-BFGS with sigmoid reparameterization.

        Args:
            density_input: (1, C, D, H, W) prepared for the model
            tumor_mask, oar_mask: boolean masks (D, H, W)
            energy_val, fff_val: fixed discrete values for this branch
            patient_fixed: tensor (3,) with fixed patient parameters
            n_restarts: inicializaciones aleatorias
            n_lbfgs_steps: L-BFGS optimization steps

        Returns:
            best_loss, best_params_physical (10,), best_history
        """
        best_loss = float('inf')
        best_params = None
        best_history = None

        for restart in range(n_restarts):
            # Initialization: restart 0 = range center, rest = random
            if restart == 0:
                init_phys = (self.cont_min + self.cont_max) / 2.0
            else:
                init_phys = self.cont_min + (
                    self.cont_max - self.cont_min
                ) * torch.rand(self.N_CONTINUOUS, device=self.device)

            # Convert to raw space (logit = inverse of sigmoid)
            raw = self._physical_to_raw(init_phys).clone().detach().requires_grad_(True)

            # L-BFGS: quasi-Newton with Wolfe line search
            optimizer = torch.optim.LBFGS(
                [raw],
                lr=1.0,
                max_iter=15,           # Line search iterations per step
                history_size=10,       # (s,y) pairs to approximate H⁻¹
                line_search_fn='strong_wolfe',
                tolerance_grad=1e-7,
                tolerance_change=1e-9,
            )

            history = []
            prev_loss = float('inf')
            stagnant_count = 0

            for step in range(n_lbfgs_steps):
                # ── Closure for L-BFGS ──
                # (L-BFGS calls the closure multiple times per step
                #  during the line search)
                def closure():
                    optimizer.zero_grad()
                    full_params = self._build_full_params(
                        raw, energy_val, fff_val, patient_fixed
                    )
                    params_norm = self.normalize_params(full_params).unsqueeze(0)
                    dose_pred, _ = self.model(density_input, params_norm)
                    loss, _ = self.clinical_objective(dose_pred, tumor_mask, oar_mask)
                    loss.backward()
                    return loss

                optimizer.step(closure)

                # Evaluate at the accepted point (without computational graph)
                with torch.no_grad():
                    full_params = self._build_full_params(
                        raw, energy_val, fff_val, patient_fixed
                    )
                    params_norm = self.normalize_params(full_params).unsqueeze(0)
                    dose_pred, _ = self.model(density_input, params_norm)
                    _, info = self.clinical_objective(dose_pred, tumor_mask, oar_mask)

                info['params'] = full_params.cpu().numpy()
                info['iteration'] = step
                history.append(info)

                # Early stopping by convergence
                current_loss = info.get('loss', float('inf'))
                if abs(prev_loss - current_loss) < 1e-8:
                    stagnant_count += 1
                    if stagnant_count >= 3:
                        break
                else:
                    stagnant_count = 0
                prev_loss = current_loss

            # Better than previous restarts?
            final_loss = history[-1].get('loss', float('inf'))
            if final_loss < best_loss:
                best_loss = final_loss
                with torch.no_grad():
                    best_params = self._build_full_params(
                        raw, energy_val, fff_val, patient_fixed
                    ).clone()
                best_history = history

        return best_loss, best_params, best_history

    # ═══════════════════════════════════════
    # MC DROPOUT EVALUATION
    # ═══════════════════════════════════════

    def _mc_dropout_eval(self, density_input, params_physical,
                         tumor_mask, oar_mask, n_samples):
        """
        Evaluates plan uncertainty with MC Dropout.
        Activa dropout, hace n_samples forward passes,
        computes statistics of clinical metrics.
        """
        self.model.train()  # Activa dropout

        params_norm = self.normalize_params(params_physical).unsqueeze(0)
        metrics_list = []

        with torch.no_grad():
            for _ in range(n_samples):
                dose, _ = self.model(density_input, params_norm)
                _, info = self.clinical_objective(dose, tumor_mask, oar_mask)
                metrics_list.append(info)

        self.model.eval()  # Restaurar

        # Statistics
        keys = ['D95_tumor', 'D98_tumor', 'Dmax_OAR', 'Dmean_OAR', 'V95', 'HI', 'loss']
        stats = {}
        for k in keys:
            vals = [m[k] for m in metrics_list if k in m]
            if vals:
                arr = np.array(vals)
                stats[f'{k}_mean']    = float(np.mean(arr))
                stats[f'{k}_std']     = float(np.std(arr))
                stats[f'{k}_ci95_lo'] = float(np.percentile(arr, 2.5))
                stats[f'{k}_ci95_hi'] = float(np.percentile(arr, 97.5))

        return stats

    # ═══════════════════════════════════════
    # MAIN METHOD: optimize()
    # ═══════════════════════════════════════

    def optimize(self, density_volume, patient_params_fixed=None,
                 n_restarts=3, n_lbfgs_steps=25, mc_eval_samples=0,
                 verbose=True):
        """
        Optimizes EBRT beam parameters for a patient.

        Estrategia:
          1. For each {energy × FFF} combination (6 branches):
             - Launch n_restarts initializations with L-BFGS
          2. Select the best of the 6 × n_restarts solutions
          3. Optionally, evaluate uncertainty with MC Dropout

        Total forward passes ≈ 6 × n_restarts × n_lbfgs_steps × ~6
        (with defaults: ~2700 passes → ~15-30 seconds on GPU)

        Args:
            density_volume: (C, D, H, W) or (D, H, W) — patient anatomy
            patient_params_fixed: dict with {patient_type, depth_cm, size_cm}
                                   or None to use central values
            n_restarts: random initializations per discrete branch
            n_lbfgs_steps: L-BFGS steps per restart
            mc_eval_samples: if > 0, MC Dropout post-optimization evaluation
            verbose: print progress

        Returns:
            best_params: dict with optimal parameters (physical units)
            history: list of dicts with winning branch convergence
        """
        t0 = time.time()

        # ── Preparar input ──
        if density_volume.dim() == 3:
            density_volume = density_volume.unsqueeze(0)  # (D,H,W) → (1,D,H,W)
        density_input = density_volume.unsqueeze(0).to(self.device)  # → (1,C,D,H,W)

        # ── Anatomical masks ──
        tumor_mask, oar_mask = self.create_anatomical_masks(density_volume)

        # ── Fixed patient parameters ──
        if patient_params_fixed is not None:
            pf = patient_params_fixed
            patient_fixed = torch.tensor(
                [pf['patient_type'], pf['depth_cm'], pf['size_cm']],
                device=self.device, dtype=torch.float32
            )
        else:
            # Default central values
            patient_fixed = torch.tensor(
                [0.5, 10.0, 4.5], device=self.device, dtype=torch.float32
            )

        if verbose:
            total_est = len(self.ENERGY_FFF_COMBOS) * n_restarts * n_lbfgs_steps * 6
            print(f"\n{'='*70}")
            print(f"  TREATMENT OPTIMIZATION v2 — TrueBeam SVC 3.0 + HD MLC")
            print(f"{'='*70}")
            print(f"  Method: L-BFGS (quasi-Newton) + sigmoid reparameterization")
            print(f"  Discrete branches: {len(self.ENERGY_FFF_COMBOS)} (energy × FFF)")
            print(f"  Restarts per branch: {n_restarts}")
            print(f"  L-BFGS steps per restart: {n_lbfgs_steps}")
            print(f"  Forward passes estimados: ~{total_est}")
            print(f"  Tumor voxels: {tumor_mask.sum().item():,}")
            print(f"  OAR voxels:   {oar_mask.sum().item():,}")
            print(f"  Pesos: α_D95={self.alpha}, β_Dmax={self.beta}, γ_HI={self.gamma}")
            print(f"         δ_D98={self.delta}, ε_Dmean={self.epsilon}, ζ_V95={self.zeta}")
            if mc_eval_samples > 0:
                print(f"  MC Dropout evaluation: {mc_eval_samples} samples")
            print(f"{'='*70}\n")

        # ── Branch-and-evaluate ──
        branch_results = []

        for energy_val, fff_val, label in self.ENERGY_FFF_COMBOS:
            if verbose:
                print(f"  🔍 Rama {label:<10s}:", end=" ", flush=True)

            branch_t0 = time.time()
            loss, params, history = self._optimize_branch(
                density_input, tumor_mask, oar_mask,
                energy_val, fff_val, patient_fixed,
                n_restarts, n_lbfgs_steps
            )
            branch_dt = time.time() - branch_t0

            branch_results.append({
                'label': label,
                'loss': loss,
                'params': params,
                'history': history,
            })

            if verbose:
                h = history[-1]
                n_steps = len(history)
                print(f"Loss={loss:+.4f} | D95={h['D95_tumor']:.4f} | "
                      f"D98={h['D98_tumor']:.4f} | Dmax_OAR={h['Dmax_OAR']:.4f} | "
                      f"V95={h['V95']:.3f} | {n_steps} steps | {branch_dt:.1f}s")

        # ── Select best branch ──
        best_branch = min(branch_results, key=lambda r: r['loss'])
        best_params_physical = best_branch['params']
        best_history = best_branch['history']

        # Convert to dict
        param_names = list(self.PARAM_BOUNDS.keys())
        best_params_dict = {
            name: float(val)
            for name, val in zip(param_names, best_params_physical.cpu().numpy())
        }

        elapsed = time.time() - t0

        if verbose:
            print(f"\n{'='*70}")
            print(f"  RESULT — Winning branch: {best_branch['label']}")
            print(f"{'='*70}")
            for name, val in best_params_dict.items():
                bounds = self.PARAM_BOUNDS[name]
                rng = bounds[1] - bounds[0]
                pct = (val - bounds[0]) / rng * 100 if rng > 0 else 50
                bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
                print(f"  {name:<16s}: {val:8.3f}  [{bounds[0]:.1f}|{bar}|{bounds[1]:.1f}]")

            # Final metrics
            h_final = best_history[-1]
            print(f"\n  D95  = {h_final['D95_tumor']:.4f}")
            print(f"  D98  = {h_final['D98_tumor']:.4f}")
            print(f"  V95  = {h_final['V95']:.3f}")
            print(f"  HI   = {h_final['HI']:.4f}")
            print(f"  Dmax_OAR  = {h_final['Dmax_OAR']:.4f}")
            print(f"  Dmean_OAR = {h_final['Dmean_OAR']:.4f}")

            # Comparative table of branches
            print(f"\n  {'Rama':<12} {'Loss':>8} {'D95':>8} {'D98':>8} "
                  f"{'Dmax_OAR':>10} {'V95':>6}")
            print(f"  {'-'*56}")
            for r in sorted(branch_results, key=lambda x: x['loss']):
                h = r['history'][-1]
                marker = ' ◀' if r['label'] == best_branch['label'] else ''
                print(f"  {r['label']:<12} {r['loss']:>+8.4f} "
                      f"{h['D95_tumor']:>8.4f} {h['D98_tumor']:>8.4f} "
                      f"{h['Dmax_OAR']:>10.4f} {h['V95']:>6.3f}{marker}")

            print(f"\n  Total time: {elapsed:.1f}s")
            print(f"{'='*70}")

        # ── MC Dropout evaluation (optional) ──
        self.last_mc_stats = None
        if mc_eval_samples > 0:
            if verbose:
                print(f"\n  🎲 MC Dropout evaluation ({mc_eval_samples} samples)...")

            mc_stats = self._mc_dropout_eval(
                density_input, best_params_physical,
                tumor_mask, oar_mask, mc_eval_samples
            )
            self.last_mc_stats = mc_stats

            if verbose:
                print(f"  {'Metric':<16} {'Mean':>10} {'Std':>10} {'95% CI':>22}")
                print(f"  {'-'*60}")
                for k in ['D95_tumor', 'D98_tumor', 'Dmax_OAR', 'V95', 'HI']:
                    if f'{k}_mean' in mc_stats:
                        print(f"  {k:<16} {mc_stats[f'{k}_mean']:>10.4f} "
                              f"{mc_stats[f'{k}_std']:>10.4f} "
                              f"[{mc_stats[f'{k}_ci95_lo']:.4f}, "
                              f"{mc_stats[f'{k}_ci95_hi']:.4f}]")
                print(f"{'='*70}")

        return best_params_dict, best_history


print("✅ EBRTOptimizer v2 defined — TrueBeam SVC 3.0 + HD MLC")
print("   Method: L-BFGS (quasi-Newton) + sigmoid reparameterization")
print("   Branch-and-evaluate: 6 energy × FFF combinations")
print("   Multi-start: escapes local minima")
print("   Objetivo ICRU-83: D95, D98, V95, HI, Dmax_OAR, Dmean_OAR")
print("   MC Dropout: post-optimization uncertainty evaluation")

In [ ]:
"""
CELL 18: Optimization v2 execution + visualization
============================================================
We run the v2 optimizer on an anatomy from the test set.

Improvements over v1:
  - L-BFGS converges in ~25 steps vs 200 of Adam
  - Branch-and-evaluate: compares 6 energy × FFF configurations
  - Multi-start: 3 initializations per branch
  - MC Dropout: uncertainty evaluation of the optimal plan
"""

# ================================================================
# RUN OPTIMIZATION v2
# ================================================================

# Take a sample from the test set as the "patient" anatomy
test_sample = next(iter(test_loader))
patient_density = test_sample['density'][0, 0]  # (D, H, W) — 1 canal (density)
patient_params_norm = test_sample['params'][0].cpu().numpy()  # (10,) en [0,1]

# Create v2 optimizer
optimizer_ebrt = EBRTOptimizer(
    surrogate_model=model,
    device=DEVICE,
    config=CFG,
    alpha_coverage=1.0,     # D95: coverage priority
    beta_oar=0.5,           # Dmax OAR
    gamma_homogeneity=0.3,  # HI
    delta_d98=0.5,          # D98: near-minimum dose
    epsilon_dmean_oar=0.3,  # Dmean OAR
    zeta_v95=0.5,           # V95: volumetric coverage
)

# De-normalize to show in physical units
patient_params_physical = optimizer_ebrt.denormalize_params(
    torch.tensor(patient_params_norm, device=DEVICE, dtype=torch.float32)
).cpu().numpy()

print(f"Patient anatomy: shape={patient_density.shape}")
print(f"Original beam parameters:")
param_names_display = [
    'Beam energy', 'FFF', 'Field X (cm)', 'Field Y (cm)',
    'SSD (cm)', 'Gantry (°)', 'Collimator (°)',
    'Patient type', 'Tumor depth (cm)', 'Tumor size (cm)',
]
for pname, pval in zip(param_names_display, patient_params_physical):
    print(f"   {pname}: {pval:.2f}")

# Fixed patient parameters (extracted from the sample)
patient_fixed = {
    'patient_type': float(patient_params_physical[7]),
    'depth_cm':     float(patient_params_physical[8]),
    'size_cm':      float(patient_params_physical[9]),
}
print(f"\nFixed parameters: type={patient_fixed['patient_type']:.2f}, "
      f"prof={patient_fixed['depth_cm']:.1f}cm, "
      f"tam={patient_fixed['size_cm']:.1f}cm\n")

# Optimize: 6 branches × 3 restarts × 25 steps L-BFGS + MC Dropout eval
best_params, opt_history = optimizer_ebrt.optimize(
    density_volume=patient_density.to(DEVICE),
    patient_params_fixed=patient_fixed,
    n_restarts=3,
    n_lbfgs_steps=25,
    mc_eval_samples=20,  # Evaluate uncertainty with MC Dropout
    verbose=True,
)

# ================================================================
# CONVERGENCE VISUALIZATION
# ================================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('L-BFGS Convergence — Optimization v2 — TrueBeam SVC 3.0',
             fontsize=16, fontweight='bold')

iterations = [h['iteration'] for h in opt_history]

# 1. Total loss
ax = axes[0, 0]
losses = [h['loss'] for h in opt_history]
ax.plot(iterations, losses, 'b-', linewidth=2, marker='o', markersize=4)
ax.set_xlabel('L-BFGS Step')
ax.set_ylabel('Loss (clinical objective)')
ax.set_title('Objective Function (L-BFGS)', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=min(losses), color='r', linestyle='--', alpha=0.5,
           label=f'Best: {min(losses):.4f}')
ax.legend()

# 2. Tumor D95 + D98
ax = axes[0, 1]
d95s = [h['D95_tumor'] for h in opt_history]
d98s = [h.get('D98_tumor', h['D95_tumor']) for h in opt_history]
ax.plot(iterations, d95s, 'g-', linewidth=2, marker='o', markersize=4, label='D95')
ax.plot(iterations, d98s, 'lime', linewidth=2, marker='s', markersize=4, label='D98')
ax.set_xlabel('L-BFGS Step')
ax.set_ylabel('Tumor dose')
ax.set_title('Tumor Coverage (D95, D98) ↑', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Dmax OAR + Dmean OAR
ax = axes[0, 2]
dmax_oars = [h['Dmax_OAR'] for h in opt_history]
dmean_oars = [h.get('Dmean_OAR', 0) for h in opt_history]
ax.plot(iterations, dmax_oars, 'r-', linewidth=2, marker='o', markersize=4, label='Dmax')
ax.plot(iterations, dmean_oars, 'salmon', linewidth=2, marker='s', markersize=4, label='Dmean')
ax.set_xlabel('L-BFGS Step')
ax.set_ylabel('OAR dose')
ax.set_title('OAR Dose (Dmax, Dmean) ↓', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. HI + V95 (dual axis)
ax = axes[1, 0]
his = [h['HI'] for h in opt_history]
v95s = [h.get('V95', 0) for h in opt_history]
ax2 = ax.twinx()
line1, = ax.plot(iterations, his, 'm-', linewidth=2, marker='o', markersize=4, label='HI ↓')
line2, = ax2.plot(iterations, v95s, 'c-', linewidth=2, marker='s', markersize=4, label='V95 ↑')
ax.set_xlabel('L-BFGS Step')
ax.set_ylabel('HI', color='m')
ax2.set_ylabel('V95', color='c')
ax.set_title('Homogeneity & Volumetric Coverage', fontweight='bold')
ax.legend([line1, line2], ['HI ↓', 'V95 ↑'], loc='center right')
ax.grid(True, alpha=0.3)

# 5. Continuous parameter evolution (only the 5 optimized)
ax = axes[1, 1]
param_names = list(EBRTOptimizer.PARAM_BOUNDS.keys())
param_labels = [
    'Energy', 'FFF', 'Field X', 'Field Y', 'SSD',
    'Gantry', 'Collimator', 'Patient', 'Depth', 'Size',
]
params_history = np.array([h['params'] for h in opt_history])
for i in EBRTOptimizer.CONTINUOUS_INDICES:
    name = param_names[i]
    bounds = EBRTOptimizer.PARAM_BOUNDS[name]
    normalized = (params_history[:, i] - bounds[0]) / (bounds[1] - bounds[0])
    ax.plot(iterations, normalized, linewidth=2, marker='o', markersize=4,
            label=param_labels[i])
ax.set_xlabel('L-BFGS Step')
ax.set_ylabel('Normalized value [0, 1]')
ax.set_title('Continuous Parameter Evolution', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 6. Initial vs final comparison (all 10 parameters)
ax = axes[1, 2]
initial_params = params_history[0]
final_params = params_history[-1]
x = np.arange(len(param_names))
width = 0.35
ax.bar(x - width/2, initial_params, width, label='Initial',
       color='lightcoral', edgecolor='black')
ax.bar(x + width/2, final_params, width, label='Optimized',
       color='lightgreen', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(['E', 'FFF', 'Fx', 'Fy', 'SSD', 'G°', 'C°', 'Pat', 'd', 's'],
                   fontsize=7, rotation=45)
ax.set_ylabel('Value')
ax.set_title('Parameters: Initial vs Optimized', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('optimization_convergence_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("→ Saved: optimization_convergence_v2.png")

In [ ]:
"""
CELL 19: Visual comparison — Initial Dose vs Optimized Dose
=================================================================
We compare the dose distribution before and after optimization
to demonstrate the clinical impact of the optimizer.

DataLoader params are already normalized to [0,1].
Optimized params come in physical units and are normalized with
optimizer_ebrt.normalize_params() before passing them to the model.
"""

with torch.no_grad():
    density_input = patient_density.unsqueeze(0).unsqueeze(0).to(DEVICE)

    # Dose with original parameters
    # patient_params_norm is already in [0,1] → pass directly to model
    params_orig_input = torch.tensor(
        patient_params_norm, device=DEVICE, dtype=torch.float32
    ).unsqueeze(0)  # (1, 10)
    dose_original, _ = model(density_input, params_orig_input)
    dose_original = dose_original[0, 0].cpu().numpy()

    # Dose with optimized parameters
    # best_params is in physical units → normalize to [0,1]
    best_vals = [best_params[k] for k in EBRTOptimizer.PARAM_BOUNDS.keys()]
    params_opt_norm = optimizer_ebrt.normalize_params(
        torch.tensor(best_vals, device=DEVICE, dtype=torch.float32)
    ).unsqueeze(0)
    dose_optimized, _ = model(density_input, params_opt_norm)
    dose_optimized = dose_optimized[0, 0].cpu().numpy()

# Visualizar
dens_np = patient_density.cpu().numpy()
nz = dens_np.shape[0]
mid_z = nz // 2

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    'Dose Comparison: Original vs Optimized — TrueBeam SVC 3.0',
    fontsize=16, fontweight='bold'
)

vmax = max(dose_original.max(), dose_optimized.max())

# Row 1: Axial slice
ax = axes[0, 0]
ax.imshow(dens_np[mid_z], cmap='bone', aspect='auto')
ax.set_title('Patient Anatomy', fontweight='bold')
ax.set_ylabel('Axial Slice', fontweight='bold')

ax = axes[0, 1]
im = ax.imshow(dose_original[mid_z], cmap='jet', aspect='auto', vmin=0, vmax=vmax)
pp = patient_params_physical  # De-normalizado en Cell 18
# Decode original energy
en_labels_map = {0.0: '6MV', 0.5: '10MV', 1.0: '15MV'}
en_orig = min(en_labels_map.keys(), key=lambda k: abs(k - pp[0]))
en_orig_str = en_labels_map[en_orig] + (' FFF' if pp[1] > 0.5 else '')
ax.set_title(f'Dose — Original Params\n'
             f'{en_orig_str}, Fx={pp[2]:.1f}cm, '
             f'Fy={pp[3]:.1f}cm', fontweight='bold', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

ax = axes[0, 2]
im = ax.imshow(dose_optimized[mid_z], cmap='jet', aspect='auto', vmin=0, vmax=vmax)
# Decode optimized energy
en_opt = min(en_labels_map.keys(), key=lambda k: abs(k - best_params['beam_energy']))
en_opt_str = en_labels_map[en_opt] + (' FFF' if best_params['is_fff'] > 0.5 else '')
ax.set_title(f'Dose — Optimized Params\n'
             f'{en_opt_str}, Fx={best_params["field_x_cm"]:.1f}cm, '
             f'Fy={best_params["field_y_cm"]:.1f}cm', fontweight='bold', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

# Row 2: Difference + Comparative DVH
ax = axes[1, 0]
diff = dose_optimized[mid_z] - dose_original[mid_z]
vd = max(abs(diff.min()), abs(diff.max())) + 1e-8
im = ax.imshow(diff, cmap='bwr', aspect='auto', vmin=-vd, vmax=vd)
ax.set_title('Difference (Optimized - Original)', fontweight='bold')
ax.set_ylabel('Analysis', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# Comparative PDD
ax = axes[1, 1]
mid_y, mid_x = dens_np.shape[1] // 2, dens_np.shape[2] // 2
pdd_orig = dose_original[:, mid_y, mid_x]
pdd_opt = dose_optimized[:, mid_y, mid_x]
depth = np.arange(len(pdd_orig))
ax.plot(depth, pdd_orig, 'b-', linewidth=2, label='Original')
ax.plot(depth, pdd_opt, 'r-', linewidth=2, label='Optimized')
ax.fill_between(depth, pdd_orig, pdd_opt, alpha=0.3,
                color='green', where=pdd_opt > pdd_orig, label='Improvement')
ax.fill_between(depth, pdd_orig, pdd_opt, alpha=0.3,
                color='red', where=pdd_opt < pdd_orig)
ax.set_xlabel('Depth (voxels)')
ax.set_ylabel('Dose')
ax.set_title('Comparative PDD', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Comparative DVH
ax = axes[1, 2]
bins = np.linspace(0, vmax, 100)
dvh_orig = [np.sum(dose_original >= b) / dose_original.size * 100 for b in bins]
dvh_opt = [np.sum(dose_optimized >= b) / dose_optimized.size * 100 for b in bins]
ax.plot(bins / vmax * 100, dvh_orig, 'b-', linewidth=2, label='Original')
ax.plot(bins / vmax * 100, dvh_opt, 'r-', linewidth=2, label='Optimized')
ax.set_xlabel('Dose (% of maximum)')
ax.set_ylabel('Volume (%)')
ax.set_title('Comparative DVH', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dose_comparison_original_vs_optimized.png', dpi=150, bbox_inches='tight')
plt.show()
print("→ Saved: dose_comparison_original_vs_optimized.png")

# Quantitative summary (both in physical units)
print(f"\n{'='*60}")
print(f"QUANTITATIVE SUMMARY — TrueBeam SVC 3.0")
print(f"{'='*60}")
print(f"{'Parameter':<20} {'Original':>12} {'Optimized':>12} {'Δ':>10}")
print(f"{'-'*54}")
for i, name in enumerate(EBRTOptimizer.PARAM_BOUNDS.keys()):
    orig_val = patient_params_physical[i]  # Ya de-normalizado
    opt_val = best_params[name]            # Already in physical units
    delta = opt_val - orig_val
    print(f"{name:<20} {orig_val:>12.3f} {opt_val:>12.3f} {delta:>+10.3f}")

---

# 📊 PART 8: Conclusions and Future Work

## Clinical Equipment

| Component | Specification |
|------------|---------------|
| **Accelerator** | Varian TrueBeam™ SVC 3.0 |
| **MLC** | HD Millennium 120 leaves (32×2.5mm + 28×5.0mm per bank) |
| **Energies** | 6 MV, 10 MV, 15 MV (FF) + 6 MV FFF, 10 MV FFF |
| **Max MLC field** | 40 × 22 cm² (asymmetric) |
| **TPS** | Eclipse 17.0.0 with AcurosXB (Dose-to-medium, 2 mm) |
| **Techniques** | 3D-CRT, IMRT, IGRT, RapidArc, SBRT, SRS |

## Architecture Summary

| Component | Implementation | Justification |
|------------|---------------|---------------|
| **Encoder** | 3D CNN (4 levels, InstanceNorm, separable convolutions) | Captures local density features with reduced parameter count |
| **Bottleneck** | 3D Transformer (4 layers, 8 heads, learnable 3D positional enc.) | Long-range dependencies (bone shadowing, distal attenuation) |
| **Decoder** | FiLM-conditioned (γ, β per level from MLP) | Modulates dose according to 10 beam parameters (energy, FFF, field, SSD, gantry, collimator, patient type, tumor depth/size) |
| **Positional Encoding** | Learnable 3D | Anatomy has no periodicity — sinusoidal encoding inappropriate |
| **Loss** | **7** physics-informed components | MSE$^w$ + gradients + SSIM + DVH + dose fall-off + cosine similarity + L1 (with smooth transition) |
| **Deep Supervision** | Auxiliary losses at 3 decoder levels ($\lambda_{aux} = 0.5^{level}$) | Stabilizes gradients, improves convergence |
| **Uncertainty** | MC Dropout (Gal & Ghahramani, 2016) | Epistemic uncertainty maps without extra storage or ensemble cost |
| **Stage 2** | L-BFGS gradient-based optimization + branch-and-evaluate | O(seconds) vs O(minutes-hours) of Eclipse inverse planning |

## Target Metrics vs State of the Art

| Metric | State of the art (literature) | Our target |
|---------|------------------------------|------------------|
| MAE (normalized dose) | 0.01 - 0.03 | < 0.02 |
| Gamma Pass Rate (3%/3mm) | 90 - 98% | > 95% |
| Inference time | 10-100 ms | < 50 ms |
| Speedup vs AcurosXB | — | > 1000× vs 3-4 min |

## Innovations of This Work

1. **FiLM conditioning for EBRT**: Feature-wise Linear Modulation conditioning dose prediction on 10 beam parameters of the TrueBeam SVC 3.0 (including energy, FFF, collimator, anatomical type)
2. **Physics-informed loss with 7 components**: weighted MSE, spatial gradient consistency, 3D SSIM, DVH loss, dose fall-off, cosine similarity (global shape alignment), and L1 with smooth transition mechanism
3. **11-augmentation pipeline**: elastic deformation, 3D rotation, translation, random scaling, random flip, Gaussian blur, intensity scaling, Gaussian noise, gamma correction, brightness shift, low-resolution simulation, coarse dropout — systematically covering geometric and intensity invariances
4. **3 patient types**: Pelvis (low heterogeneity), head-and-neck (steep gradients, bone + air cavities), lung (ρ=0.26, the most challenging case for AcurosXB)
5. **Dual early stopping**: composite loss + DoseAcc3% clinical metric — prevents premature stopping when loss stagnates but clinical accuracy improves
6. **Collapse detection and auto-recovery**: monitors DoseAcc3% drop >15pp → restores best checkpoint, with automatic suspension during warm restarts
7. **Stochastic Weight Averaging (SWA)**: Izmailov et al. (2018) — averaged weights over late-training epochs for flatter minima and better generalization
8. **Smooth loss transition**: new loss components are ramped in linearly over configurable epochs, preventing destabilization of pre-trained models
9. **End-to-end differentiable optimizer (Stage 2)**: L-BFGS (quasi-Newton) through the surrogate emulating Eclipse inverse planning, with sigmoid reparameterization for gradient-friendly parameter bounds
10. **MC Dropout uncertainty quantification**: epistemic uncertainty maps during both inference and post-optimization evaluation (Pim & Pryer, arXiv:2509.18155, 2025)
11. **Complete pipeline**: Synthetic data → training → evaluation → optimization → [future: real DICOM]

## Limitations

1. **Synthetic training data**: current training uses analytical dose models, not clinical Monte Carlo or AcurosXB data — the surrogate will require fine-tuning once real DICOM data is available
2. **Single-beam only**: the model predicts dose for a single beam; multi-beam configurations and RapidArc (continuous gantry rotation with dynamic MLC) are future work
3. **No MLC fluence map**: the 120 HD MLC leaf positions are not yet encoded as input — this limits the model to open-field or simplified configurations
4. **Fixed volume resolution**: input is resampled to a fixed grid (e.g., 48×64×64); very large tumors or fine anatomy may require higher resolution
5. **No clinical validation yet**: performance has not been compared against AcurosXB gold standard with real patient data

## Data We Will Receive from the Hospital

- **CT**: DICOM with standard resolution
- **RT-Structure**: Tumor + OAR contours
- **RT-Plan**: All positions of the 120 HD MLC leaves + beam parameters
- **RT-Dose**: Dose calculated by AcurosXB (Dose-to-medium, 2 mm resolution)
- **3 types**: Pelvis, head-and-neck, lung (several patients of each)

## Future Work

1. **DICOM → HDF5 Pipeline**: Convert CT, RT-Structure, and RT-Dose to aligned 3D volumes; extract electron density from CT using HU-density curve
2. **MLC fluence map**: Encode the 120 HD MLC positions as an additional 2D input channel (fluence map) — this captures the intensity modulation of IMRT/RapidArc
3. **Transfer learning**: Pre-train with synthetic data (this notebook), fine-tune with real DICOM data (few-shot learning)
4. **Multi-beam / arc**: Extend to multi-beam configurations and RapidArc (continuous gantry rotation with dynamic MLC)
5. **Uncertainty**: ✅ MC Dropout implemented (Gal & Ghahramani, ICML 2016) — generates epistemic uncertainty maps without extra storage cost. Future: compare with Deep Ensembles, especially critical in lung
6. **Clinical validation**: Compare surrogate vs AcurosXB (gold standard) with Gamma Index 3%/3mm; target > 95% pass rate
7. **Inverse planning acceleration**: Replace AcurosXB calls within the Eclipse optimizer loop → reduce planning time from minutes-hours to seconds

## References

1. Chen, J. et al. "TransUNet: Transformers Make Strong Encoders for Medical Image Segmentation" (2021)
2. Perez, E. et al. "FiLM: Visual Reasoning with a General Conditioning Layer" — *AAAI* (2018)
3. Kontaxis, C. et al. "DeepDose: Deep learning for 3D dose prediction" — *Med. Phys.* (2020)
4. Low, D.A. et al. "A technique for the quantitative evaluation of dose distributions" — *Med. Phys.* (1998) — Gamma Index
5. ICRU Report 91: "Prescribing, Recording, and Reporting of Stereotactic Treatments" (2014)
6. Saravanakumar, A. et al. "Installation and Commissioning of TrueBeam SVC 3.0" — *IJAMSCR* 13(3), 2025 — Measured PDD data
7. Gal, Y. & Ghahramani, Z. "Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning" — *ICML*, 2016 — MC Dropout
8. Izmailov, P. et al. "Averaging Weights Leads to Wider Optima and Better Generalization" — *UAI* (2018) — Stochastic Weight Averaging
9. Smith, L. N. & Topin, N. "Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates" — *AMLC* (2018) — OneCycleLR
10. Loshchilov, I. & Hutter, F. "Decoupled Weight Decay Regularization" — *ICLR* (2019) — AdamW
11. Loshchilov, I. & Hutter, F. "SGDR: Stochastic Gradient Descent with Warm Restarts" — *ICLR* (2017) — Cosine annealing with restarts
12. Ronneberger, O. et al. "U-Net: Convolutional Networks for Biomedical Image Segmentation" — *MICCAI* (2015)
13. Çiçek, Ö. et al. "3D U-Net: Learning Dense Volumetric Segmentation from Sparse Annotation" — *MICCAI* (2016)
14. Simkó, A. et al. "PyDoseRT: Patient-Specific Differentiable Dose Calculation for Radiation Therapy" — *arXiv:2512.18863* (2025) — BeamValidationLayer / sigmoid reparameterization
15. Pim, J. & Pryer, T. "Uncertainty-aware dose prediction using a deep learning surrogate model" — *arXiv:2509.18155* (2025) — MC Dropout for optimization evaluation
16. Zhao, H. et al. "Loss Functions for Image Restoration with Neural Networks" — *IEEE TCI* (2017) — L1 vs L2 comparison (MD Anderson)
17. Xing, L. et al. "Dosimetric Deep Learning for Real-Time Adaptive Radiation Therapy" — *Med. Phys.* (2020)

---

*Notebook developed for the Santander X competition — Medical Physics Team*  
*Architecture: 3D TransUNet + FiLM | Framework: PyTorch ≥ 2.0 | Accelerator: TrueBeam SVC 3.0*

In [ ]:
"""
CELL 20: Model export + speed benchmark
==========================================================
We save the trained model and measure the speedup vs AcurosXB.

AcurosXB (Eclipse): ~3-4 minutes per dose calculation (according to medical physicist).
Surrogate (TransUNet+FiLM): ~ms per inference.
"""

# ================================================================
# 1. SAVE THE MODEL
# ================================================================
save_path = 'ebrt_surrogate_transunet_film3d.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': CFG.__dict__,
    'training_history': history if 'history' in dir() else None,
    'architecture': 'TransUNetFiLM3D',
    'accelerator': 'TrueBeam SVC 3.0',
    'mlc': 'HD Millennium 120 leaves',
    'energies': ['6MV', '10MV', '15MV', '6FFF', '10FFF'],
    'n_beam_params': 10,
    'patient_types': ['pelvis', 'head-neck', 'lung'],
    'description': 'Surrogate model for EBRT dose prediction — TrueBeam SVC 3.0 — Santander X',
}, save_path)

model_size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"✅ Model saved: {save_path} ({model_size_mb:.1f} MB)")

# ================================================================
# 2. BENCHMARK DE VELOCIDAD
# ================================================================
print(f"\n{'='*60}")
print(f"BENCHMARK: Surrogate vs AcurosXB (Eclipse)")
print(f"{'='*60}")

model.eval()
dummy_density = torch.randn(1, 1, *CFG.volume_size, device=DEVICE)
dummy_params = torch.randn(1, CFG.n_beam_params, device=DEVICE)

# Warmup
with torch.no_grad():
    for _ in range(10):
        _ = model(dummy_density, dummy_params)

# Benchmark
n_runs = 100
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()

t_start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        out, _ = model(dummy_density, dummy_params)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
t_total = time.time() - t_start

t_inference_ms = (t_total / n_runs) * 1000
t_acuros_s = 210  # ~3.5 min (average 3-4 min according to medical physicist)
speedup = t_acuros_s / (t_inference_ms / 1000)

print(f"\n  Device:          {DEVICE}")
print(f"  Volume:              {CFG.volume_size}")
print(f"  N° beam params:       {CFG.n_beam_params}")
print(f"  Runs:                 {n_runs}")
print(f"  Inference time:       {t_inference_ms:.2f} ms/sample")
print(f"  AcurosXB time:        {t_acuros_s:.0f} s/sample (~3-4 min)")
print(f"  ⚡ Speedup:           {speedup:,.0f}×")
print(f"\n  → With {speedup:,.0f}× speedup, we can evaluate {speedup:,.0f} sets of")
print(f"    parameters in the time AcurosXB takes to calculate ONE.")
print(f"  → Inverse planning: from minutes-hours to seconds.")

# Model parameters
n_params_total = sum(p.numel() for p in model.parameters())
n_params_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  Total parameters:       {n_params_total:,}")
print(f"  Trainable parameters:   {n_params_train:,}")
print(f"  Model size:             {model_size_mb:.1f} MB")

print(f"\n{'='*60}")
print(f"🎉 NOTEBOOK COMPLETED — Pipeline EBRT end-to-end")
print(f"{'='*60}")
print(f"  Accelerator:     TrueBeam SVC 3.0 + HD MLC Millennium")
print(f"  Energies:       6/10/15 MV (FF) + 6/10 MV (FFF)")
print(f"  Beam parameters: 10 (energy, FFF, field X/Y, SSD, gantry, coll, patient, depth, size)")
print(f"  Patients:       pelvis | head-neck | lung")
print(f"  Stage 1: Surrogate (TransUNet+FiLM)    → {'✅ Entrenado'}")
print(f"  Stage 2: Optimizador (gradient-based)   → {'✅ Implemented'}")
print(f"  Evaluation (Gamma Index 3%/3mm)         → {'✅ Complete'}")
print(f"  Export                             → {'✅ Saved'}")
print(f"  Next step: Receive DICOM data from hospital → fine-tune")
print(f"{'='*60}")